# FS5A_VGN_stage_expression

Existing analysis and drawing code is collected in this one panel notebook. Original shared methods may produce the complete original group of figures. The final drawing section below retains this panel's existing layout.

Configure the existing ROOT placeholders for the inputs and outputs of each original section. The primary notebook cells retain their original order and numeric positions. Standalone programs and source helpers are complete independent source sections; their original command-line, file, namespace, and kernel assumptions remain in force. No new Run All workflow or runtime worker generator is added.

---

# Figure 4 upstream analysis methods

Final VGN/Tbx3 input preparation, expression and activity summaries, and B/C/D assembly. Run the supplied VGN input-preparation, accepted 1,010-cell trajectory, SCENIC network, and modules workers to produce the required upstream containers. The input-preparation source contains its exact producer functions; the superseded diffusion-pseudotime fitting workflow is omitted.

Every actual storage address is replaced by `ROOT` or a string placeholder `ROOT`. Configure each input, output, runtime, program, and worker path independently before execution. HDF5 keys, public URLs, archived-source labels, numerical settings, seeds, statistical tests, and selection gates retain their original meaning.

This source-only release contains no expression matrices, fitted models, execution outputs, or hidden notebook attachments. The files were checked statically; the analysis was not rerun.

---

# VGN developmental expression and Tbx3 B-D

Seven genes: **Meis2, Npas3, Sall3, Tlx3, Tbx3, Onecut1, Pcdh19**.

- Lines reproduce native-library pseudobulk log10(1+CPM), averaging libraries within source and then sources equally.
- **B** preserves the original 7,355-cell E9.5-E18.5 UMAP. Tbx3 activity remains unavailable because shared-axis target coverage is 27/37, below 80%; gray is not evidence of absent biological activity.
- **C** preserves the actual CVG/VGN cohort (1,010 cells), primary path (478), and one validated 12-gene Tbx3 module. Raw RNA is minmax-scaled per gene, averaged, then GAM-fitted (gamma=1.5).
- **D** follows scFates 0.4.0 `features`: fitted RNA is minmax-scaled per target across the 478 fitted cells and then averaged. This differs from the C module GAM. Exact cell-ID matching colors only **168 CVG cells** on the original UMAP; the other 7,187 cells remain gray. No scores are extrapolated.
- The one-sided network uses the published vector-ribbon geometry, with Tbx3 at the center and its 11 non-self module targets. The module score includes all 12 members. Links are inferred SCENIC candidates, not demonstrated binding or causality. No second module or unsupported functional title is added.

Reference: [Petitpre et al., 2022, Fig.3](https://www.nature.com/articles/s41467-022-31580-1), [author figure code](https://github.com/LouisFaure/sgnfates_paper/blob/main/make_fig3.ipynb), [scFates 0.4.0 D formula](https://github.com/LouisFaure/scFates/blob/v0.4.0/scFates/plot/features.py).

This notebook consolidates preparation and plotting for the current delivery. Previously validated SCENIC and trajectory inference are reused, with their parameters, statistics and source hashes retained in the single result bundle. It does not silently retrain networks or change thresholds.


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#1. =========== Shared paths, reproducibility and approved outputs
from pathlib import Path
import os
import sys
import json
import hashlib
sys.dont_write_bytecode = True
# Set this storage path independently before running.
ROOT = ROOT
# Set this storage path independently before running.
if ROOT.name == 'Code_def': ROOT = ROOT
# Set this storage path independently before running.
if not (ROOT).is_dir():
    # Set this storage path independently before running.
    ROOT = ROOT
# Set this storage path independently before running.
FIGURE_DIR = ROOT
# Set this storage path independently before running.
RESULT_DIR = ROOT
# Set this storage path independently before running. VGN-results.h5.
RESULT_FILE = ROOT
# Set this storage path independently before running.
WORK_DIR = ROOT
# Set this storage path independently before running. 05_fig4B-D_VGN.ipynb. Use the supplied fig4_VGN_program_analysis.ipynb source.
NOTEBOOK_FILE = ROOT
REBUILD_RESULTS = False
SEED, DPI = 42, 600
GENES = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
for directory in (FIGURE_DIR, RESULT_DIR, WORK_DIR): directory.mkdir(parents=True, exist_ok=True)
# Set this storage path independently before running.
os.environ['MPLCONFIGDIR'] = str(ROOT)
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image

In [ ]:
"""Read-only data preparation for the requested VGN line and Tbx3 B/C/D figures."""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#2. =========== Imports and compact HDF5 readers
from pathlib import Path
import hashlib
import json
import h5py
import numpy as np
import pandas as pd
from scipy import sparse


def _vgn_digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def _vgn_frame(group):
    columns = json.loads(group.attrs['columns']) if 'columns' in group.attrs else list(group)
    columns = [key for key in columns if key != '_index']
    table = pd.DataFrame({key: group[key].asstr()[:] if group[key].dtype.kind in 'OUS'
                          else group[key][:] for key in columns})
    if '_index' in group:
        table.index = group['_index'].asstr()[:]
    return table


def _vgn_read_array(node):
    if isinstance(node, h5py.Dataset):
        return node.asstr()[:] if node.dtype.kind in 'OUS' else node[:]
    categories = _vgn_read_array(node['categories'])
    codes = node['codes'][:]
    result = np.full(len(codes), '', dtype=object)
    valid = codes >= 0
    result[valid] = categories[codes[valid]]
    return result


def _vgn_json(handle, key):
    return json.loads(handle[key].asstr()[()])


#2a. =========== Reproduce the existing library/source-balanced native RNA curves
def _vgn_line_data(root, genes, track):
    # Set this storage path independently before running. RNA-screen.h5.
    rna = track(ROOT)
    # Set this storage path independently before running. activity-screen.h5.
    activity = track(ROOT)
    with h5py.File(rna, 'r') as handle:
        # These tables use a RangeIndex, as in the original line-plot code.
        obs = pd.DataFrame({key: _vgn_read_array(handle['cells'][key])
                            for key in handle['cells']})
        tfs = handle['TFs'].asstr()[:]
        positions = pd.Index(tfs).get_indexer(genes)
        measured = np.flatnonzero(positions >= 0)
        order = measured[np.argsort(positions[measured])]
        counts = np.full((len(obs), len(genes)), -1, dtype=np.int64)
        counts[:, order] = handle['counts'][:, positions[order]]
        vgn = pd.DataFrame({key: _vgn_read_array(handle['VGN_native_library_pseudobulk'][key])
                            for key in handle['VGN_native_library_pseudobulk']})
    with h5py.File(activity, 'r') as handle:
        aobs = pd.DataFrame({key: _vgn_read_array(handle['obs'][key])
                             for key in handle['obs']})
        np.testing.assert_array_equal(obs.cell_id, handle['cells'].asstr()[:])
        np.testing.assert_array_equal(obs.cell_id, aobs['_index'])
    extras = [genes[j] for j in np.flatnonzero(positions < 0)]
    assert extras == ['Pcdh19']
    for study in obs.study_id.unique():
        # Set this storage path independently before running.
        native_path = ROOT
        with np.load(track(native_path / f'{study}_axes.npz'), allow_pickle=False) as axis:
            rows = pd.Index(obs.cell_id).get_indexer(axis['cells'])
            native_pos = pd.Index(axis['genes']).get_indexer(extras)
        assert (rows >= 0).all() and obs.study_id.iloc[rows].eq(study).all()
        present = native_pos >= 0
        if present.any():
            native = sparse.load_npz(track(native_path / f'{study}_counts.npz')).tocsr()
            columns = pd.Index(genes).get_indexer(np.asarray(extras)[present])
            counts[np.ix_(rows, columns)] = native[:, native_pos[present]].toarray()
    good = obs.source_state_evaluation_allowed & ~obs.source_rna_mixture_review_flag
    obs['identity'] = np.where(obs.study_id.eq('GSE178931'),
        np.where(aobs.cell_type_plot.eq('Differentiating SGN'), 'SGN', 'CVG'),
        np.where(obs.study_id.eq('GSE231513'), 'CVG-like', 'SGN'))
    records = []
    for (study, stage, identity, library), part in obs.loc[good].groupby(
            ['study_id', 'stage_standardized', 'identity', 'library_id']):
        ii = part.index.to_numpy()
        total = float(part.library_total.sum())
        values = counts[ii]
        measured = (values >= 0).all(axis=0)
        np.testing.assert_array_equal(measured, (values >= 0).any(axis=0))
        for j, gene in enumerate(genes):
            if not measured[j]:
                continue
            cpm = float(values[:, j].sum() / total * 1e6)
            records.append({'gene': gene, 'study': study, 'stage': stage,
                'identity': identity, 'library': library, 'cells': len(ii),
                'expr': float(np.log10(1 + cpm))})
    for _, row in vgn.loc[vgn.TF.isin(genes)].iterrows():
        records.append({'gene': row.TF, 'study': row.study, 'stage': row.stage,
            'identity': 'VGN', 'library': row.library_id, 'cells': int(row.cells),
            'expr': float(row.log10_1p_full_endogenous_pseudobulk_CPM)})
    # Set this storage path independently before running. VGN520-common-counts.npz.
    reference_path = track(ROOT)
    with np.load(reference_path, allow_pickle=False) as handle:
        reference = pd.DataFrame({'cell': handle['cells'], 'stage': handle['stage'],
            'library': handle['library_id'], 'total': handle['full_endogenous_total']})
    assert len(reference) == 520 and not set(reference.cell).intersection(obs.cell_id)
    for (stage, library), part in reference.groupby(['stage', 'library']):
        # Set this storage path independently before running.
        qc = track(ROOT /
                   f'{stage.replace(".5", "")}_GSE178931__{library}.h5ad')
        with h5py.File(qc, 'r') as handle:
            source_cells = _vgn_read_array(handle['obs'][handle['obs'].attrs['_index']])
            source_genes = _vgn_read_array(handle['var/gene_id_analysis'])
            rows = pd.Index(source_cells).get_indexer(part.cell)
            columns = pd.Index(source_genes).get_indexer(extras + ['Meis2'])
            assert (rows >= 0).all() and (columns >= 0).all()
            np.testing.assert_array_equal(
                _vgn_read_array(handle['obs/total_counts_endogenous'])[rows], part.total)
            node = handle['X']
            assert node.attrs['encoding-type'] == 'csr_matrix'
            native = sparse.csr_matrix((node['data'][:], node['indices'][:], node['indptr'][:]),
                                       shape=tuple(node.attrs['shape']))
            selected = native[rows][:, columns].toarray()
        assert (selected >= 0).all() and np.array_equal(selected, np.round(selected))
        expression = np.log10(1 + selected.sum(axis=0) / part.total.sum() * 1e6)
        check = vgn.loc[vgn.TF.eq('Meis2') & vgn.library_id.eq(library)]
        assert len(check) == 1 and int(check.cells.iloc[0]) == len(part)
        np.testing.assert_allclose(expression[-1],
            check.log10_1p_full_endogenous_pseudobulk_CPM.iloc[0], atol=1e-12, rtol=0)
        for j, gene in enumerate(extras):
            records.append({'gene': gene, 'study': 'GSE178931', 'stage': stage,
                'identity': 'VGN', 'library': library, 'cells': len(part),
                'expr': float(expression[j])})
    units = pd.DataFrame(records)
    assert not units.duplicated(['gene', 'study', 'stage', 'identity', 'library']).any()
    assert np.isfinite(units.expr).all()
    units['age'] = units.stage.str[1:].astype(float)
    units['display_type'] = units.identity.replace({'CVG-like': 'CVG'})
    source = units.groupby(['gene', 'study', 'stage', 'age', 'display_type'],
                           as_index=False).expr.mean()
    points = source.groupby(['gene', 'stage', 'age', 'display_type'],
                            as_index=False).expr.mean()
    expected_ages = {'CVG': [9.5, 10.5, 11.5, 13.5], 'VGN': [11.5, 13.5],
                    'SGN': [11.5, 13.5, 14.5, 15.5, 16.5, 17.5, 18.5]}
    for gene in genes:
        for identity, ages in expected_ages.items():
            selected = points.loc[points.gene.eq(gene) &
                                  points.display_type.eq(identity)].sort_values('age')
            assert selected.age.tolist() == ages
    expected = {
        'Meis2': (2.5066719700015003, 1.8940479387399884, 2.2062386322181484, 2.5540468745907523),
        'Npas3': (.9282563596426009, .7336424725792932, .9714389934156881, 1.4054498393780541),
        'Sall3': (0., .09229789590891971, .879733112605054, 2.208247911791241),
        'Pcdh19': (1.3426267707, .8103542867, .8197308614, 1.7452178938),
        'Tlx3': (.6600239333272321, .15234775622538638, 2.512395569425404, 1.8605084171933841),
        'Tbx3': (.5467942348815206, .44483049832420346, 2.2831485537423153, 1.564270157853805),
        'Onecut1': (.7878465510724536, 0., 1.8492034642098982, 1.5900562876971758)}
    for gene, values in expected.items():
        z = points.loc[points.gene.eq(gene)].set_index(['display_type', 'age']).expr
        actual = [z.loc[(identity, age)] for identity in ['SGN', 'VGN'] for age in [11.5, 13.5]]
        np.testing.assert_allclose(actual, values, atol=6e-7, rtol=0)
    cvg_expected = {'Meis2': (1.7939295425, 1.3306404845),
        'Npas3': (.7807507629, 1.1000281097), 'Sall3': (.2086257215, .4308168693),
        'Pcdh19': (1.2250303359, .6190038183), 'Tlx3': (1.8330174019, 1.2298949827),
        'Tbx3': (1.9523876660, .9038402172), 'Onecut1': (1.7164276103, 0.)}
    for gene, values in cvg_expected.items():
        z = points.loc[points.gene.eq(gene) & points.display_type.eq('CVG')].set_index('age').expr
        np.testing.assert_allclose(z.loc[[11.5, 13.5]], values, atol=6e-7, rtol=0)
    np.testing.assert_allclose(points.loc[points.gene.eq('Pcdh19') &
        points.display_type.eq('SGN')].sort_values('age').expr,
        [1.342627, .810354, .759606, 1.321273, 1.564139, 1.537127, 1.196972], atol=6e-7, rtol=0)
    assert points.expr.between(0, 4.1).all()
    return units, source, points, {gene: int((counts[:, j] < 0).sum())
                                  for j, gene in enumerate(genes)}


#2b. =========== Frozen B/C and exact paper D formula without extrapolation
def prepare_vgn_data(ROOT):
    """Return compact figure inputs; never write, refit, rescore AUCell or change any input."""
    root = Path(ROOT).resolve()
    used = {}

    def track(path):
        path = Path(path).resolve()
        key = str(path.relative_to(root))
        if key not in used:
            used[key] = _vgn_digest(path)
        return path

    genes = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
    units, source, points, missing = _vgn_line_data(root, genes, track)
    # Set this storage path independently before running. Modules.h5. Produced by fig4_VGN_modules_analysis.py.
    modules_path = track(ROOT)
    # Set this storage path independently before running. Network.h5. Produced by fig4_VGN_network_analysis.py.
    network_path = track(ROOT)
    # Set this storage path independently before running. Gene expression plots.h5.
    xy_path = track(ROOT)
    with h5py.File(modules_path, 'r') as handle:
        np.testing.assert_array_equal(handle['anchors'].asstr()[:], genes)
        old_obs = _vgn_frame(handle['original_obs'])
        old_ids = handle['original_cell_ids'].asstr()[:]
        old_xy = handle['X_umap_original'][:]
        actual = _vgn_frame(handle['obs'])
        actual_ids = handle['cell_ids'].asstr()[:]
        np.testing.assert_array_equal(old_obs.index.astype(str), old_ids)
        np.testing.assert_array_equal(actual.index.astype(str), actual_ids)
        primary = handle['primary_path'][:].astype(bool)
        module_defs = _vgn_json(handle, 'module_definitions_json')
        selected = _vgn_json(handle, 'selected_modules_json')
        assert len(module_defs) == 1 and module_defs[0]['anchor'] == 'Tbx3'
        assert selected['Tbx3'] == [0] and all(not selected[gene] for gene in genes if gene != 'Tbx3')
        definition = module_defs[0]
        members = np.asarray(definition['genes'], dtype=object)
        assert len(members) == 12 and len(set(members)) == 12
        membership_hash = hashlib.sha256('\n'.join(sorted(members)).encode()).hexdigest()
        assert membership_hash == definition['gene_sha256']
        target_genes = handle['target_genes'].asstr()[:]
        columns = pd.Index(target_genes).get_indexer(members)
        assert (columns >= 0).all()
        # Select after loading the small 1010 x 59 matrix to preserve module order.
        target_rna = handle['target_RNA'][:][:, columns]
        target_min = handle['target_min'][:][columns]
        target_span = handle['target_span'][:][columns]
        assert (target_span > 0).all()
        scores = handle['module_scores'][:]
        np.testing.assert_array_equal(((target_rna - target_min) / target_span).mean(axis=1), scores[:, 0])
        c_fitted = handle['module_fitted_primary'][:]
        fitted_indices = handle['target_fitted_gene_indices'][:].astype(int)
        fitted_genes = target_genes[fitted_indices]
        fitted_columns = pd.Index(fitted_genes).get_indexer(members)
        assert (fitted_columns >= 0).all()
        target_fitted = handle['target_fitted_primary'][:][:, fitted_columns]
        assert target_fitted.shape == (478, 12) and np.isfinite(target_fitted).all()
        # scFates v0.4.0 features(): minmax each fitted gene first, then average genes.
        d_min = target_fitted.min(axis=0)
        d_span = target_fitted.max(axis=0) - d_min
        assert (d_span > 0).all()
        d_primary = ((target_fitted - d_min) / d_span).mean(axis=1)
        assert d_primary.shape == (478,) and np.isfinite(d_primary).all()
        assert ((d_primary >= 0) & (d_primary <= 1)).all()
        primary_rows = np.flatnonzero(primary)
        assert len(primary_rows) == 478 and c_fitted.shape == (478, 1)
        position = pd.Index(old_ids).get_indexer(actual_ids[primary_rows])
        matched = position >= 0
        old_rows = position[matched]
        actual_rows = primary_rows[matched]
        fit_rows = np.flatnonzero(matched)
        assert matched.sum() == 168 and actual.type.iloc[actual_rows].eq('CVG').all()
        assert (~matched).sum() == 310 and actual.type.iloc[primary_rows[~matched]].eq('VGN').all()
        assert len(np.unique(old_rows)) == 168
        d_atlas = np.full(7355, np.nan)
        d_atlas[old_rows] = d_primary[fit_rows]
        assert np.isfinite(d_atlas).sum() == 168
        definitions = _vgn_json(handle, 'definitions_json')
        association = _vgn_frame(handle['target_association'])
        association = association.loc[association.anchor.eq('Tbx3')].reset_index(drop=True)
        communities = _vgn_frame(handle['official_communities'])
        communities = communities.loc[communities.anchor.eq('Tbx3')].reset_index(drop=True)
        statuses = handle['B_status'].asstr()[:]
        assert np.isnan(handle['B_fitted_original'][:]).all()
        assert not handle['B_display_allowed'][:].any()
        assert not np.any(statuses == 'available')
        b = {'cell_ids': old_ids, 'obs': old_obs, 'xy': old_xy, 'anchors': np.asarray(genes),
             'status': statuses, 'definitions': definitions,
             'available': np.zeros(len(genes), dtype=bool),
             'display_vmax': np.full(len(genes), np.nan),
             'stage_counts': old_obs.stage_standardized.value_counts().to_dict()}
        c = {'cell_ids': actual_ids, 'obs': actual, 'type': actual.type.to_numpy(),
             'native_t': handle['t_native'][:], 'primary': primary,
             'fork_t': float(handle['fork_t'][()]), 'grid': handle['fit_grid'][:],
             'module_scores': scores, 'module_grid_fitted': handle['module_grid_fitted'][:],
             'module_fitted_primary': c_fitted, 'members': members,
             'target_min': target_min, 'target_span': target_span, 'target_RNA': target_rna,
             'target_fitted_primary': target_fitted,
             'target_fitted_source_indices': fitted_indices[fitted_columns],
             'module_definitions': module_defs, 'selected_modules': selected,
             'methods': _vgn_json(handle, 'methods_json')}
        d = {'fitted': d_atlas, 'primary_fitted': d_primary, 'primary_actual_rows': primary_rows,
             'matched_original_rows': old_rows, 'matched_actual_rows': actual_rows,
             'matched_primary_fit_rows': fit_rows, 'primary_to_original_positions': position,
             'matched_cell_ids': actual_ids[actual_rows], 'fit_gene_min': d_min,
             'fit_gene_span': d_span,
             'formula': 'mean over 12 genes of (per-gene fitted expression - its primary478 minimum) / its primary478 span',
             'scope': 'Original7355 UMAP; only168 exact-ID-matched CVG primary-path cells colored; all others unavailable',
             'C_module_GAM_used_as_D': False, 'extrapolation': False}
        upstream = {'Modules.h5': _vgn_json(handle, 'input_sha256_json')}
    with h5py.File(xy_path, 'r') as handle:
        np.testing.assert_array_equal(handle['cell_ids'].asstr()[:], old_ids)
        np.testing.assert_array_equal(handle['X_umap'][:], old_xy)
    assert old_xy.shape == (7355, 2) and len(actual) == 1010
    assert actual.type.value_counts().to_dict() == {'VGN': 520, 'CVG': 490}
    assert sorted(actual.age.unique()) == [9.5, 11.5, 13.5]

    #2c. =========== Full Tbx3 signature and its eleven non-self inferred GRN links
    with h5py.File(network_path, 'r') as handle:
        assert handle.attrs['stage'] == 'complete'
        np.testing.assert_array_equal(handle['cell_ids'].asstr()[:], actual_ids)
        regulon = _vgn_json(handle, 'regulons_json')['Tbx3']
        signature = regulon['gene2weight']
        assert len(signature) == 37 and set(members).issubset(signature)
        assert regulon['own_TF_in_regulon'] and signature['Tbx3'] == 1.0
        tfs = handle['TFs'].asstr()[:]
        inference_genes = handle['genes'].asstr()[:]
        assert len(inference_genes) == 15214
        tf_index = int(np.flatnonzero(tfs == 'Tbx3')[0])
        grn = handle['grn']
        all_links = []
        targets = set(members) - {'Tbx3'}
        chunk = 200000
        self_links = 0
        for start in range(0, len(grn['TF_index']), chunk):
            stop = min(start + chunk, len(grn['TF_index']))
            tf_values = grn['TF_index'][start:stop]
            chosen = tf_values == tf_index
            target_values = grn['target_index'][start:stop][chosen]
            target_names = inference_genes[target_values]
            importances = grn['importance'][start:stop][chosen]
            self_links += int((target_names == 'Tbx3').sum())
            for target, importance in zip(target_names, importances):
                if target in targets:
                    all_links.append({'source': 'Tbx3', 'target': target, 'importance': float(importance)})
        edges = pd.DataFrame(all_links)
        edges['member_order'] = pd.Index(members).get_indexer(edges.target)
        edges = edges.sort_values('member_order').drop(columns='member_order').reset_index(drop=True)
        assert len(edges) == 11 and edges.target.is_unique and set(edges.target) == targets
        assert self_links == 0 and (edges.importance > 0).all()
        for row in edges.itertuples():
            assert row.importance == signature[row.target]
        network = {'TF': 'Tbx3', 'signature_genes': np.asarray(list(signature), dtype=object),
            'signature_weights': np.asarray(list(signature.values()), dtype=float),
            'signature_definition': regulon, 'module_members': members, 'edges': edges,
            'motif_support': [row for row in _vgn_json(handle, 'motif_support_json') if row['TF'] == 'Tbx3'],
            'pruning_methods': _vgn_json(handle, 'pruning_methods_json'),
            'inference_methods': _vgn_json(handle, 'grn_methods_json'),
            'training_methods': _vgn_json(handle, 'methods_json'),
            'resources_sha256': _vgn_json(handle, 'resource_sha256_json'),
            'self_edge_drawn': False, 'direct_binding_or_causality_claim': False,
            'edge_semantics': 'Motif-pruned, GRNBoost2-inferred TF-target candidates; importance is not a binding probability'}
        upstream['Network.h5'] = _vgn_json(handle, 'source_sha256_json')
    assert {key: _vgn_digest(root / key) for key in used} == used
    diagnostic = {'line_units': len(units), 'line_source_means': len(source),
        'line_points': len(points), 'line_early_means_verified': True,
        'B_cells': len(old_ids), 'B_all_activity_blocked': True,
        'C_cells': len(actual_ids), 'C_primary_cells': int(primary.sum()),
        'C_modules': scores.shape[1], 'D_colored_cells': int(np.isfinite(d_atlas).sum()),
        'D_gray_cells': int(np.isnan(d_atlas).sum()), 'D_not_C_module_GAM': True,
        'network_module_edges': len(edges), 'protected_inputs_unchanged': True}
    # A flat contract avoids saving duplicate array copies in the final result bundle.
    signature_metadata = {key: value for key, value in regulon.items() if key != 'gene2weight'}
    paper_methods = {
        'line': {'unit': 'log10(1+sum raw counts/sum full endogenous library totals*1e6)',
            'averaging': 'Equal library weights within source/stage/type, then equal source weights',
            'E10.5': 'Source identity retained; rendered using the ordinary CVG marker',
            'scope': 'Historical native-source identities and quality flags; no new marker selection'},
        'B': {'scope': 'Original7355 E9.5-E18.5 UMAP; actual VGN520 absent',
            'ranking_axis': 'Frozen shared11467; never combined with native15214 AUC',
            'activity': 'All seven panels blocked by their saved original SCENIC status; no RNA replacement'},
        'C': c['methods'],
        'D': {'source_url': 'https://raw.githubusercontent.com/LouisFaure/scFates/v0.4.0/scFates/plot/features.py',
            'source_sha256': '4d36b6a3faac270201db1cebf7bb26dfab41586719c2bcbfa476fae5f242753d',
            'source_lines': '216-218 fitted RNA,231 per-gene minmax,499-502 mean_trajectory',
            'formula': d['formula'], 'scope': d['scope'],
            'normalization_scope': 'The478 fitted primary-path cells; no source atlas range substituted',
            'C_module_GAM_used_as_D': False, 'extrapolation': False},
        'network': {'regulon': signature_metadata,
            'pruning': network['pruning_methods'], 'inference': network['inference_methods'],
            'training': network['training_methods'], 'resource_sha256': network['resources_sha256'],
            'edge_semantics': network['edge_semantics'], 'self_edge_drawn': False,
            'direct_binding_or_causality_claim': False},
        'upstream_source_sha256': upstream}
    return {'genes': np.asarray(genes), 'line_units': units, 'line_source': source,
        'line_points': points, 'line_missing_source_features': missing,
        'B_obs': old_obs, 'B_cell_ids': old_ids, 'B_xy': old_xy,
        'B_status': statuses, 'B_definitions': definitions, 'B_display_vmax': b['display_vmax'],
        'B_stage_counts': b['stage_counts'],
        'C_obs': actual, 'C_cell_ids': actual_ids, 'C_type': c['type'],
        'C_primary': primary, 'C_t': c['native_t'], 'C_fork_t': c['fork_t'],
        'C_scores': scores, 'C_curve': c['module_grid_fitted'], 'C_grid': c['grid'],
        'C_fit_at_cells': c_fitted, 'C_target_RNA': target_rna,
        'C_gene_min': target_min, 'C_gene_span': target_span,
        'C_target_association': association, 'C_dynamic_communities': communities,
        'D_primary': d_primary, 'D_atlas': d_atlas,
        'D_fit_gene_matrix': target_fitted, 'D_fit_gene_min': d_min, 'D_fit_gene_span': d_span,
        'D_fit_gene_source_indices': c['target_fitted_source_indices'],
        'D_primary_actual_rows': primary_rows, 'D_primary_to_old_positions': position,
        'D_match_old_rows': old_rows, 'D_match_actual_rows': actual_rows,
        'D_match_fit_rows': fit_rows, 'D_match_cell_ids': d['matched_cell_ids'],
        'module_definition': definition, 'module_genes': members,
        'network_edges': edges, 'network_signature_genes': network['signature_genes'],
        'network_signature_weights': network['signature_weights'],
        'network_motif_support': network['motif_support'],
        'source_hashes': used, 'paper_methods': paper_methods, 'diagnostics': diagnostic}


In [ ]:
#3. =========== Embedded one-module geometry from the published Fig.3d
PAPER_NETWORK_TEMPLATE = {'source': 'Petitpre et al. 2022, Nature Communications, Fig.3d', 'source_sha256': '8dda1b31fe727e1ee88d73932b6b0dd485ab070855ec39bd4d5587e1b2826bd4', 'source_page': 6, 'bbox': [296.0, 204.0, 400.0, 290.0], 'center': [384.737473256552, 248.26325348336002], 'slots': [{'point': [342.18483272487, 217.1121474525599], 'bbox': [337.93301391, 215.0451475425599, 346.43665153973996, 219.1791473625599], 'path': [['m', [384.70498657, 247.80102588]], ['c', [383.90998656, 247.92102588], [383.08198655, 247.97902588], [382.22698665, 247.97902588]], ['c', [374.84298611, 247.98302588], [365.42998696, 243.62802603], [357.62598610000003, 237.53002598]], ['c', [349.81798554, 231.43702556000005], [343.62198639, 223.59702541000001], [342.59298706000004, 216.73302507999995]], ['l', [342.20198822000003, 216.79202511000005]], ['c', [343.27098847, 223.83602572999996], [349.53098679000004, 231.70402576000004], [357.38298607, 237.84102585000005]], ['c', [365.23998642000004, 243.97302581999998], [374.69198704, 248.37002587999996], [382.22698665, 248.37402588999998]], ['c', [383.09998656, 248.37402588999998], [383.94798655, 248.31502588], [384.76398657000004, 248.19202587999996]], ['l', [384.70498657, 247.80102588]]]}, {'point': [337.58536149108, 279.76703766336], 'bbox': [332.75500488, 277.59413770336, 342.41571810216, 281.93993762336], 'path': [['m', [384.5409851, 247.95898487]], ['c', [383.1479851, 255.21098471999994], [376.32698536, 263.25498439], [367.8289852, 269.45598461]], ['c', [359.33598518, 275.66198398], [349.16998672, 280.03198672999997], [341.15998459, 280.02798511]], ['c', [339.92298507, 280.02798511], [338.73898697, 279.92398502], [337.61998367, 279.70898487]], ['l', [337.54498672, 280.09698535]], ['c', [338.69198608, 280.31798602], [339.90098572, 280.42298557], [341.15998459, 280.42298557]], ['c', [349.30898666, 280.4199834], [359.51598549, 276.01398517], [368.061985, 269.77498485]], ['c', [376.6039853, 263.53198482000005], [383.4899851, 255.46098471999994], [384.92898511, 248.03398487000004]], ['l', [384.5409851, 247.95898487]]]}, {'point': [313.35505509218797, 245.62421539336003], 'bbox': [309.1281128, 243.45131543336004, 317.581997384376, 247.79711535336003], 'path': [['m', [384.6000061, 247.85199023999996]], ['c', [377.81400633, 254.16299011], [364.97900582, 257.2509904], [351.80200577, 257.24699068999996]], ['c', [336.5300064, 257.25198985], [320.80100631999994, 253.1039900899999], [313.49700928, 245.22599031999994]], ['l', [313.20700836, 245.49499013000002]], ['c', [320.66500472999996, 253.51199007999992], [336.46000671999997, 257.6369900799999], [351.80200577, 257.6419902]], ['c', [365.03900527999997, 257.63799049], [377.94200609999996, 254.56699039], [384.8690061, 248.14099023999995]], ['l', [384.6000061, 247.85199023999996]]]}, {'point': [356.088734364884, 244.5250502825599], 'bbox': [350.6889038, 242.4580503725599, 361.48856492976796, 246.5920501925599], 'path': [['m', [384.61499023, 247.83898975]], ['c', [382.29399013, 249.61298979000003], [378.49899006, 250.44698978999998], [374.39698982, 250.44498968999994]], ['c', [367.40699005, 250.45298982999998], [359.5329895, 248.02498975000003], [356.52198982, 244.05298974000004]], ['l', [356.20698928999997, 244.29298974999995]], ['c', [359.38298987999997, 248.43698973000005], [367.31798935, 250.83198977999996], [374.39698982, 250.83998967000002]], ['c', [378.55099011, 250.83798981999996], [382.41299033, 250.01098967999997], [384.85399022999997, 248.15398975000005]], ['l', [384.61499023, 247.83898975]]]}, {'point': [318.41408894212, 267.16614654336], 'bbox': [310.92410278, 264.99324658336, 325.90407510424, 269.33904650336], 'path': [['m', [384.56201171, 247.90002489999995]], ['c', [381.04601168, 254.22502469999995], [373.26701163999996, 259.80202532], [364.01501083, 263.7800250099999]], ['c', [354.76401138, 267.76002551], [344.04501340999997, 270.1470246399999], [334.66701126, 270.1470246399999]], ['c', [328.40001297, 270.1470246399999], [322.73401259999997, 269.0790257499999], [318.50901031, 266.7280240099999]], ['l', [318.31700896999996, 267.07302523]], ['c', [322.63001251, 269.47102404], [328.35601042999997, 270.54102563999993], [334.66701126, 270.54202509999993]], ['c', [344.11101149999996, 270.54102563999993], [354.87101172999996, 268.1430249299999], [364.17101097, 264.1430249299999]], ['c', [373.47001171, 260.14002467], [381.31401180999995, 254.54302501999996], [384.90701171, 248.09302489999993]], ['l', [384.56201171, 247.90002489999995]]]}, {'point': [346.20866888772, 262.10511139336006], 'bbox': [341.73901367, 259.93221143336007, 350.67832410544, 264.27801135336006], 'path': [['m', [384.55599975, 247.91198780000002]], ['c', [382.62599981, 252.00998784], [377.81299972, 255.82098771000005], [372.05499935, 258.58798744]], ['c', [366.29899979000004, 261.35598804000006], [359.60300064, 263.08798744], [353.93199921, 263.086988]], ['c', [351.02399825000003, 263.086988], [348.38800049, 262.63198806], [346.2879982, 261.63998748000006]], ['l', [346.11899948, 261.99698782999997]], ['c', [348.29299927, 263.02298786000006], [350.98299789000004, 263.48098804000006], [353.93199921, 263.48198750000006]], ['c', [359.68199921, 263.48098804000006], [366.42000008, 261.7349877500001], [372.22699928000003, 258.94398738000007]], ['c', [378.03099965, 256.14898826], [382.9089998, 252.31698800000004], [384.91299975000004, 248.0809878]], ['l', [384.55599975, 247.91198780000002]]]}, {'point': [322.756078766624, 226.05335357336003], 'bbox': [318.77011108, 223.88045361336003, 326.742046453248, 228.22625353336002], 'path': [['m', [384.6499939, 247.81799365999996]], ['c', [381.24699379000003, 249.42599369999994], [376.98399402, 250.16099358999998], [372.28799344000004, 250.16099358999998]], ['c', [363.14399339000005, 250.16199375999997], [352.3639946, 247.37599365999995], [343.0929947, 242.91899349999994]], ['c', [333.81999207, 238.46499397999992], [326.06099320000004, 232.33299401], [322.9339943, 225.7029938899999]], ['l', [322.57699204000005, 225.87199451999993]], ['c', [325.77799226, 232.63999320999994], [333.60199357000005, 238.79299404999995], [342.92199326, 243.27499344]], ['c', [352.24299240000005, 247.75399366], [363.06499482000004, 250.5549936399999], [372.28799344000004, 250.55599356999994]], ['c', [377.0249939, 250.55599356999994], [381.34199382, 249.81699360999994], [384.8189939, 248.17499365999993]], ['l', [384.6499939, 247.81799365999996]]]}, {'point': [335.78853088996004, 244.15320709336004], 'bbox': [332.52700805, 241.98030713336004, 339.05005372992, 246.32610705336003], 'path': [['m', [384.60598754, 247.84698535999996]], ['c', [380.19398737, 251.64298536], [372.34598732, 253.4589853399999], [364.09498787, 253.45598555999993]], ['c', [352.93998718, 253.46198513000002], [341.05598831, 250.13998533999995], [335.93298721, 244.16198541999995]], ['l', [335.63398742, 244.4199853199999]], ['c', [340.91498564, 250.54998541999998], [352.86398697, 253.8449855], [364.09498787, 253.85098554]], ['c', [372.40198707, 253.84798528], [380.31898737, 252.04398537999998], [384.86298754, 248.14598535999994]], ['l', [384.60598754, 247.84698535999996]]]}, {'point': [341.931003113304, 228.58631256336002], 'bbox': [336.13500976, 226.41341260336003, 347.726996466608, 230.75921252336002], 'path': [['m', [384.6659851, 247.8109746]], ['c', [382.79998505, 248.50097460000006], [380.62498522, 248.82197463], [378.2609849, 248.82197463]], ['c', [371.64598465, 248.82397461000005], [363.56498527, 246.30497456], [356.6639843, 242.46397448000005]], ['c', [349.76098633, 238.62597418999997], [344.042984, 233.4569745], [342.11598587000003, 228.22997523000004]], ['l', [341.74598694, 228.36697435999997]], ['c', [343.73898697, 233.74597502000006], [349.5219841, 238.93797445999996], [356.47198486, 242.80897475000006]], ['c', [363.42398453, 246.67697466000004], [371.54798507, 249.21597456999996], [378.2609849, 249.21697460999997]], ['c', [380.65998506, 249.21697460999997], [382.87998509, 248.89197457], [384.8029851, 248.1819746]], ['l', [384.6659851, 247.8109746]]]}, {'point': [355.255089154796, 233.17708885255996], 'bbox': [347.3269043, 231.11008894255997, 363.183274009592, 235.24408876255995], 'path': [['m', [384.6749878, 247.80798388999995]], ['c', [383.58998777, 248.14998388999993], [382.36598779, 248.3109839099999], [381.04898787999997, 248.3109839099999]], ['c', [376.45298768, 248.31298388999994], [370.73598768, 246.34498383999994], [365.88198854, 243.41198398999995]], ['c', [361.02498819, 240.48198367999998], [357.03598787, 236.5799842099999], [355.84898759, 232.79198408999991]], ['l', [355.47198869, 232.91098358]], ['c', [356.71998788, 236.85798408999995], [360.77398874, 240.78498413], [365.67798806, 243.7499838099999]], ['c', [370.5839882, 246.71098388999997], [376.34298803, 248.70398390999992], [381.04898787999997, 248.70598388999997]], ['c', [382.39798785, 248.70598388999997], [383.66198779, 248.54198391], [384.7939878, 248.1849838899999]], ['l', [384.6749878, 247.80798388999995]]]}, {'point': [324.71179432656004, 252.19410064336], 'bbox': [315.2901001, 250.02120068336, 334.13348855312, 254.36700060336], 'path': [['m', [384.58499145, 247.86700488999998]], ['c', [378.29899168000003, 255.14800502000003], [363.83999062000004, 259.16400481999995], [350.22599030000003, 259.15800525]], ['c', [340.01898955, 259.1610045599999], [330.28799057000003, 256.90500498999995], [324.83999252, 252.18400480000003]], ['l', [324.58098984000003, 252.48300505999998]], ['c', [330.16299057000003, 257.3060050099999], [339.96199035, 259.5500045], [350.22599030000003, 259.55300476]], ['c', [363.91599083, 259.54700518999994], [378.43899155, 255.55800487], [384.88399145, 248.12600489]], ['l', [384.58499145, 247.86700488999998]]]}]}

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#4. =========== One compact result bundle with lossless arrays and tables
from urllib.parse import quote, unquote
def _plain(value):
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, Path): return str(value)
    raise TypeError(type(value).__name__)

def write_value(parent, key, value):
    if isinstance(value, pd.DataFrame):
        group = parent.create_group(key); group.attrs['kind'] = 'dataframe'
        group.attrs['columns'] = json.dumps(value.columns.tolist())
        write_value(group, '_index', value.index.to_numpy())
        for column in value.columns: write_value(group, column, value[column].to_numpy())
    elif isinstance(value, dict):
        group = parent.create_group(key); group.attrs['kind'] = 'dict'
        for name, item in value.items(): write_value(group, quote(str(name), safe=''), item)
    elif isinstance(value, np.ndarray):
        if value.dtype.kind in 'OUS':
            node = parent.create_dataset(key, data=value.astype(str).astype(object), dtype=h5py.string_dtype())
        else:
            node = parent.create_dataset(key, data=value, compression='gzip' if value.ndim else None)
        node.attrs['kind'] = 'ndarray'
    else:
        node = parent.create_dataset(key, data=json.dumps(value, default=_plain), dtype=h5py.string_dtype())
        node.attrs['kind'] = 'json'

def read_value(node):
    kind = node.attrs['kind']
    if kind == 'dataframe':
        columns = json.loads(node.attrs['columns'])
        out = pd.DataFrame({key: read_value(node[key]) for key in columns})
        out.index = read_value(node['_index']); return out
    if kind == 'dict': return {unquote(key): read_value(node[key]) for key in node}
    if kind == 'json': return json.loads(node.asstr()[()])
    return node.asstr()[:] if node.dtype.kind in 'OUS' else node[:]

if REBUILD_RESULTS or not RESULT_FILE.exists():
    data = prepare_vgn_data(ROOT)
    data['paper_network_geometry'] = PAPER_NETWORK_TEMPLATE
    with h5py.File(RESULT_FILE, 'w') as handle:
        handle.attrs['schema'] = 'VGN_fig4BD_v1'
        write_value(handle, 'data', data)
else:
    with h5py.File(RESULT_FILE, 'r') as handle: data = read_value(handle['data'])
np.testing.assert_array_equal(data['genes'], GENES)
assert len(data['line_units']) == 302 and len(data['line_points']) == 91
assert data['C_primary'].sum() == 478 and np.isfinite(data['D_atlas']).sum() == 168
assert data['network_edges'].shape[0] == 11 and len(data['module_genes']) == 12
print('Ready: 7 RNA curves, 1 Tbx3 module; B=7355 cells, C path=478, D matched=168.')

In [ ]:
#5. =========== Editable vector PDF exporter
"""Strict vector PDF export for the Fig4 B/C/D Matplotlib canvas.

No analysis is performed. Each native Matplotlib label becomes one live Arial
text object, including the TF italic run in the two supported mathtext titles.
All geometry is written directly, without XObjects, glyph outlines or images.
The caller declares at most four data axes; their clip rectangles are collected
into one sibling viewport group per axis. Other artists must fit their axes.
"""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
from pathlib import Path
import re
import sys
import math
import numpy as np
from matplotlib.backend_bases import RendererBase
from matplotlib.path import Path as MplPath

try:
    from reportlab.pdfgen import canvas
except ImportError:
    # Set this storage path independently before running.
    sys.path.append(str(ROOT))
    from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

_FONTS = {'normal':'FigArial','italic':'FigArialItalic',
          'bold':'FigArialBold','bolditalic':'FigArialBoldItalic'}

def _register():
    for key, file in [('normal','arial.ttf'),('italic','ariali.ttf'),
                      ('bold','arialbd.ttf'),('bolditalic','arialbi.ttf')]:
        if _FONTS[key] not in pdfmetrics.getRegisteredFontNames():
            # Set this storage path independently before running.
            pdfmetrics.registerFont(TTFont(_FONTS[key],str(ROOT/file)))

def _font(prop):
    bold=prop.get_weight() in ('bold','heavy','black','semibold','demibold') or isinstance(prop.get_weight(),(int,float)) and prop.get_weight()>=600
    italic=prop.get_style() in ('italic','oblique')
    return _FONTS['bolditalic' if bold and italic else 'bold' if bold else 'italic' if italic else 'normal']

def _runs(text,prop,ismath):
    if not ismath:
        if '$' in text: raise ValueError('Unexpected dollar in plain label: '+text)
        return [(text,_font(prop))]
    # Current Fig4 titles use only this explicitly audited mathtext form.
    found=re.fullmatch(r'\$\\it\{([^{}]+)\}([^$]*)\$(.*)',text)
    if found is None: raise ValueError('Unsupported math label: '+text)
    tf,inside,after=found.groups()
    return [(tf,_FONTS['italic']),(inside+after,_FONTS['normal'])]

class _Recorder(RendererBase):
    def __init__(self,fig,agg,viewports,halo_labels):
        super().__init__()
        self.dpi=fig.dpi;self.unit=72/fig.dpi;self.agg=agg
        self.width,self.height=fig.bbox.size;self.ops=[];self.viewports=viewports;self.halo_labels=halo_labels
    def points_to_pixels(self,p): return p/self.unit
    def flipy(self):return False
    def get_canvas_width_height(self):return self.width,self.height
    def get_text_width_height_descent(self,s,prop,ismath):
        runs=_runs(s,prop,ismath)
        width=sum(pdfmetrics.stringWidth(t,f,prop.get_size_in_points()) for t,f in runs)/self.unit
        # Match the source Arial baseline/ascent geometry; width uses the same
        # embedded font as the PDF, including the semantic mixed italic title.
        _,h,d=self.agg.get_text_width_height_descent(re.sub(r'\$|\\it\{|\}', '',s),prop,False)
        return width,h,d
    def _clip(self,gc):
        if gc.get_clip_path()[0] is not None:
            raise ValueError('Nonrectangular artist clip is unsupported')
        b=gc.get_clip_rectangle()
        if b is None:return None
        for i,box in enumerate(self.viewports):
            if np.allclose(b.bounds,box,rtol=0,atol=1e-5):return i
        return None
    def draw_path(self,gc,path,transform,rgbFace=None):
        if gc.get_hatch():raise ValueError('Hatching/Pattern is forbidden')
        p=transform.transform_path(path)
        self.ops.append(('path',self._clip(gc),p,gc.get_rgb(),rgbFace,
                         gc.get_linewidth(),gc.get_alpha(),gc.get_forced_alpha(),
                         gc.get_dashes(),gc.get_capstyle(),gc.get_joinstyle()))
    def draw_text(self,gc,x,y,s,prop,angle,ismath=False,mtext=None):
        self.ops.append(('text',None,float(x),float(y),str(s),prop.copy(),float(angle),
                         ismath,gc.get_rgb(),gc.get_alpha(),id(mtext) in self.halo_labels))
    def draw_image(self,*args,**kwargs):raise ValueError('Raster image is forbidden')
    def start_rasterizing(self):pass
    def stop_rasterizing(self):pass
    def start_filter(self):raise ValueError('Raster filter is forbidden')
    def stop_filter(self,*args):raise ValueError('Raster filter is forbidden')

def _pdf_path(pdf,path,unit):
    out=pdf.beginPath();current=None;start=None
    for vertices,code in path.iter_segments(remove_nans=True,simplify=False,curves=True):
        v=np.asarray(vertices).reshape(-1,2)*unit
        if code==MplPath.MOVETO:
            current=v[0];start=current;out.moveTo(*current)
        elif code==MplPath.LINETO:
            current=v[0];out.lineTo(*current)
        elif code==MplPath.CURVE3:
            control,end=v
            a=current+(control-current)*2/3;b=end+(control-end)*2/3
            out.curveTo(*a,*b,*end);current=end
        elif code==MplPath.CURVE4:
            out.curveTo(*v.ravel());current=v[-1]
        elif code==MplPath.CLOSEPOLY:
            out.close();current=start
        elif code==MplPath.STOP:break
        else:raise ValueError('Unknown path opcode '+str(code))
    return out

def strict_editable_pdf(fig,destination,*,data_axes=(),live_halo_width=.20):
    """Write a strict vector PDF. Return declared clip boxes in PDF points.

    The artist canvas remains unchanged after return. Rasterization flags and
    white text path effects are replaced by a narrow live text fill+stroke
    (render mode 2) in the same single label object; no outlined font paths are
    emitted. The width is narrower than Agg's halo to preserve the black stems.
    """
    _register()
    if len(data_axes)>4:raise ValueError('At most four sibling data viewports')
    fig.canvas.draw();agg=fig.canvas.get_renderer()
    viewports=[tuple(ax.bbox.bounds) for ax in data_axes]
    effects=[]
    for artist in fig.findobj():
        if hasattr(artist,'get_path_effects'):
            old=artist.get_path_effects()
            if old:effects.append((artist,old));artist.set_path_effects([])
    rec=_Recorder(fig,agg,viewports,{id(a) for a,e in effects})
    try:fig.draw(rec)
    finally:
        for a,e in effects:a.set_path_effects(e)
    dst=Path(destination);dst.parent.mkdir(parents=True,exist_ok=True)
    pdf=canvas.Canvas(str(dst),pagesize=tuple(fig.get_size_inches()*72),
                      pageCompression=1,invariant=1,initialFontName=_FONTS['normal'])
    unit=rec.unit
    def emit(op):
        if op[0]=='text':
            _,_,x,y,s,prop,angle,ismath,color,alpha,halo=op
            if not s:return
            pdf.saveState();pdf.setFillColorRGB(*color[:3]);pdf.setFillAlpha(alpha)
            text=pdf.beginText();rad=math.radians(angle)
            if halo and live_halo_width>0:
                pdf.setStrokeColorRGB(1,1,1);pdf.setStrokeAlpha(alpha)
                pdf.setLineWidth(float(live_halo_width));text.setTextRenderMode(2)
            text.setTextTransform(math.cos(rad),math.sin(rad),-math.sin(rad),math.cos(rad),x*unit,y*unit)
            for value,font in _runs(s,prop,ismath):
                if value:text.setFont(font,prop.get_size_in_points());text.textOut(value)
            pdf.drawText(text);pdf.restoreState();return
        _,_,path,stroke,face,lw,alpha,forced,dashes,cap,join=op
        pdf.saveState();pdf.setLineWidth(lw)
        pdf.setLineCap({'butt':0,'round':1,'projecting':2}[cap])
        pdf.setLineJoin({'miter':0,'round':1,'bevel':2}[join])
        offset,pattern=dashes
        pdf.setDash([] if pattern is None else [float(v) for v in pattern],0.0 if offset is None else float(offset))
        if stroke is not None:
            pdf.setStrokeColorRGB(*stroke[:3]);pdf.setStrokeAlpha(alpha if forced else (stroke[3] if len(stroke)>3 else alpha))
        if face is not None:
            pdf.setFillColorRGB(*face[:3]);pdf.setFillAlpha(alpha if forced else (face[3] if len(face)>3 else alpha))
        pdf.drawPath(_pdf_path(pdf,path,unit),stroke=int(stroke is not None and lw>0),fill=int(face is not None),fillMode=1)
        pdf.restoreState()
    done=set()
    for op in rec.ops:
        key=op[1]
        if key is None:emit(op);continue
        if key in done:continue
        done.add(key);pdf.saveState()
        p=pdf.beginPath();p.rect(*(np.asarray(viewports[key])*unit))
        pdf.clipPath(p,stroke=0,fill=0,fillMode=1)
        for candidate in rec.ops:
            if candidate[1]==key:emit(candidate)
        pdf.restoreState()
    pdf.showPage();pdf.save()
    if done!=set(range(len(data_axes))):raise AssertionError('Declared clip has no artist')
    return {'expected_clips':len(done),'clip_boxes_pt':(np.asarray(viewports)*unit).tolist(),
            'editable_label_objects':sum(op[0]=='text' for op in rec.ops),
            'label_strings':[op[4] for op in rec.ops if op[0]=='text'],
            'vector_paths':sum(op[0]=='path' for op in rec.ops),
            'text_effects':'White live text fill+stroke, single object; stroke width '+str(live_halo_width)+' pt.',
            'halo_labels':len(effects)}

In [ ]:
#6. =========== Shared vector drawing and figure validation
import matplotlib.patheffects as pe
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath
from matplotlib.colors import LinearSegmentedColormap, Normalize


def check_figure(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    clipped = []
    for text in fig.findobj(matplotlib.text.Text):
        if text.get_visible() and text.get_text():
            box = text.get_window_extent(renderer)
            if not fig.bbox.contains(*box.p0) or not fig.bbox.contains(*box.p1):
                clipped.append(text.get_text())
    assert not clipped, clipped
    return {'clipped_labels': clipped}


def save_figure(fig, stem):
    qa = check_figure(fig)
    png, pdf = FIGURE_DIR / (stem + '.png'), FIGURE_DIR / (stem + '.pdf')
    fig.savefig(png, dpi=DPI, facecolor='white')
    qa.update(strict_editable_pdf(fig, pdf, data_axes=()))
    from PIL import Image
    with Image.open(png) as im:
        assert im.size == tuple(round(v * DPI) for v in fig.get_size_inches())
        assert all(abs(v - DPI) < .1 for v in im.info['dpi'])
        qa.update({'pixels': list(im.size), 'dpi': list(im.info['dpi'])})
    qa.update({'png': str(png), 'pdf': str(pdf)})
    plt.close(fig)
    return qa


#7. =========== Seven source-balanced developmental expression curves
def render_lines(data):
    FIGSIZE = (15.2, 6.4)
    X_LIMITS, Y_LIMITS = (9.1, 18.9), (-.10, 4.10)
    X_TICKS, Y_TICKS = (9.5, 11.5, 13.5, 16.5, 18.5), (0, 1, 2, 3, 4)
    MEAN_LINE_WIDTH, MEAN_MARKER_SIZE = 1.65, 3.8
    LIBRARY_AREA, LIBRARY_ALPHA = 10.0, .25
    ORDER = ('CVG', 'VGN', 'SGN')
    COLORS = {'CVG': '#8D77A1', 'VGN': '#5B85A7', 'SGN': '#69A18B'}
    units, points = data['line_units'], data['line_points']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9,
        'text.color': '#30343A', 'axes.labelcolor': '#30343A',
        'xtick.color': '#50555B', 'ytick.color': '#50555B', 'savefig.facecolor': 'white'})
    fig = plt.figure(figsize=FIGSIZE)
    grid = fig.add_gridspec(2, 4, left=.058, right=.985, bottom=.13, top=.94,
                           hspace=.53, wspace=.25)
    axes = []
    for j, gene in enumerate(GENES):
        row, col = (0, j) if j < 3 else (1, j - 3)
        ax = fig.add_subplot(grid[row, col])
        axes.append(ax)
        for identity in ORDER:
            u = units.loc[units.gene.eq(gene) & units.display_type.eq(identity)]
            p = points.loc[points.gene.eq(gene) & points.display_type.eq(identity)].sort_values('age')
            ax.scatter(u.age, u.expr, s=LIBRARY_AREA, color=COLORS[identity],
                       alpha=LIBRARY_ALPHA, linewidths=0, zorder=2)
            line, = ax.plot(p.age, p.expr, color=COLORS[identity], lw=MEAN_LINE_WIDTH,
                           marker='o', ms=MEAN_MARKER_SIZE, markeredgewidth=0, zorder=3)
            np.testing.assert_array_equal(line.get_ydata(), p.expr)
        ax.set_xlim(X_LIMITS); ax.set_ylim(Y_LIMITS)
        ax.set_xticks(X_TICKS, [f'E{x:g}' for x in X_TICKS])
        ax.xaxis.set_minor_locator(FixedLocator(np.arange(9.5, 19)))
        ax.set_yticks(Y_TICKS)
        if col: ax.tick_params(labelleft=False)
        ax.tick_params(which='major', length=2.6, width=.6, labelsize=8.5, pad=3)
        ax.tick_params(axis='x', which='minor', length=1.5, width=.45, color='#A5AAB0')
        ax.grid(axis='y', color='#EBECEE', linewidth=.55, zorder=0)
        for side in ('top', 'right'): ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#A3A7AD'); ax.spines[side].set_linewidth(.65)
        ax.set_title(gene, fontstyle='italic', fontsize=12, pad=8)
    legend_ax = fig.add_subplot(grid[0, 3]); legend_ax.axis('off')
    handles = [Line2D([0], [0], color=COLORS[t], lw=MEAN_LINE_WIDTH, marker='o',
                      ms=4, markeredgewidth=0, label=t) for t in ORDER]
    handles.append(Line2D([0], [0], marker='o', color='#8C9299', alpha=LIBRARY_ALPHA,
                          linestyle='None', ms=4, label='Individual libraries'))
    legend_ax.legend(handles=handles, loc='center', frameon=False, fontsize=9.5,
                     handlelength=2.2, labelspacing=1.0)
    fig.supylabel('RNA: mean log10(1 + pseudobulk CPM)', x=.012, y=.55, fontsize=10)
    fig.supxlabel('Embryonic age', x=.53, y=.035, fontsize=10)
    fig.canvas.draw(); renderer = fig.canvas.get_renderer()
    for ax in axes:
        boxes = [t.get_window_extent(renderer) for t in ax.get_xticklabels() if t.get_visible()]
        assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:])
    return save_figure(fig, 'VGN-lines')


#8. =========== One supported module in the published ribbon geometry
def draw_tbx3_network(ax, genes):
    FONT_SIZE, COLOR = 8.8, '#2D7184'
    LABEL_OFFSETS_PT = {'Cadm1': (-4.0, 3.0), 'Gm7271': (2.0, -3.0)}
    ax.set_axis_off(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    fig = ax.figure; fig.canvas.draw()
    area = ax.get_window_extent(fig.canvas.get_renderer())
    width, height = np.asarray(area.size) * 72 / fig.dpi
    bbox = np.asarray(PAPER_NETWORK_TEMPLATE['bbox'], float)
    scale = min(.94 * width / (bbox[2] - bbox[0]), .94 * height / (bbox[3] - bbox[1]))
    offset = (np.array([width, height]) - (bbox[2:] - bbox[:2]) * scale) / 2
    def xy(values):
        p = (np.asarray(values, float) - bbox[:2]) * scale + offset
        p[..., 1] = height - p[..., 1]
        return p / np.array([width, height])
    halo = [pe.withStroke(linewidth=.75, foreground='white')]
    center = ax.text(*xy(PAPER_NETWORK_TEMPLATE['center']), 'Tbx3', transform=ax.transAxes,
        ha='center', va='center', fontsize=FONT_SIZE, fontstyle='italic', zorder=6, path_effects=halo)
    slots = PAPER_NETWORK_TEMPLATE['slots']
    assert len(slots) == len(genes) == 11 and 'Tbx3' not in genes
    prop = FontProperties(family='Arial', style='italic', size=FONT_SIZE)
    gene_order = sorted(genes, key=lambda g: (TextPath((0, 0), g, prop=prop).get_extents().width, g))
    slot_order = sorted(slots, key=lambda s: (s['bbox'][2] - s['bbox'][0], s['point'][1]))
    labels, mapping = [center], []
    for gene, slot in zip(gene_order, slot_order):
        vertices, codes = [], []
        for step in slot['path']:
            vertices.extend(step[1:])
            codes.extend([{'m': MplPath.MOVETO, 'l': MplPath.LINETO, 'c': MplPath.CURVE4}[step[0]]] * len(step[1:]))
        vertices.append(vertices[0]); codes.append(MplPath.CLOSEPOLY)
        ax.add_patch(PathPatch(MplPath(xy(vertices), codes), transform=ax.transAxes,
            facecolor=COLOR, edgecolor='none', linewidth=0, zorder=1))
        point = xy(slot['point']) + np.asarray(LABEL_OFFSETS_PT.get(gene, (0, 0))) / [width, height]
        text = ax.text(*point, gene, transform=ax.transAxes, ha='center', va='center',
            fontstyle='italic', fontsize=FONT_SIZE, color='black', zorder=5, path_effects=halo)
        labels.append(text); mapping.append({'gene': gene, 'template_point': slot['point'],
            'label_offset_points': list(LABEL_OFFSETS_PT.get(gene, (0, 0)))})
    fig.canvas.draw(); renderer = fig.canvas.get_renderer()
    boxes = [(t.get_text(), t.get_window_extent(renderer)) for t in labels]
    overlaps, clipped = [], []
    for i, (name, box) in enumerate(boxes):
        if box.x0 < area.x0 or box.x1 > area.x1 or box.y0 < area.y0 or box.y1 > area.y1:
            clipped.append(name)
        overlaps += [(name, other) for other, b in boxes[i + 1:] if box.overlaps(b)]
    assert not overlaps and not clipped, (overlaps, clipped)
    return {'edges': 11, 'members_including_center': 12, 'self_edge_drawn': False,
        'label_overlaps': overlaps, 'clipped_labels': clipped, 'slot_mapping': mapping,
        'edge_semantics': 'Inferred motif-supported SCENIC target links; geometry carries no edge strength',
        'highlighted_genes': []}


#8a. =========== Paper-style Tbx3 B/C/D with separate RNA summaries for C and D
def render_tbx3(data):
    FIGSIZE = (13.2, 4.15)
    POINT_AREA, C_POINT_AREA = 1.35, 3.1
    D_VMAX, C_YMAX = 1.0, .90
    PALETTE = {'CVG': '#1A6D79', 'VGN': '#00008B'}
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'text.color': 'black',
        'axes.labelcolor': 'black', 'xtick.color': 'black', 'ytick.color': 'black', 'axes.linewidth': .65})
    xy = np.asarray(data['B_xy'])
    primary = np.asarray(data['C_primary'], bool)
    t = np.asarray(data['C_t']); factor = 6.0 / t.max()
    score = np.asarray(data['C_scores'])[:, 0]
    curve = np.asarray(data['C_curve'])[:, 0]
    dscore = np.asarray(data['D_atlas'])
    assert xy.shape == (7355, 2) and int(np.isfinite(dscore).sum()) == 168
    assert primary.sum() == 478 and score.min() >= 0 and score.max() <= C_YMAX
    assert not np.any(np.asarray(data['B_status']) == 'available')
    fig = plt.figure(figsize=FIGSIZE)
    b = fig.add_axes([.025, .28, .225, .57])
    c = fig.add_axes([.305, .39, .215, .36])
    d = fig.add_axes([.565, .28, .20, .57])
    network = fig.add_axes([.785, .26, .20, .60])
    for ax in (b, d):
        ax.scatter(xy[:, 0], xy[:, 1], s=POINT_AREA, color='#D4D4D4', alpha=.65, linewidths=0)
        ax.set_aspect('equal')
        ax.set_xlim(xy[:, 0].min() - 1, xy[:, 0].max() + 1)
        ax.set_ylim(xy[:, 1].min() - 1, xy[:, 1].max() + 1); ax.axis('off')
    b.set_title(r'$\it{Tbx3}(+)$ activity', fontsize=11, pad=7)
    fig.text(.1375, .245, 'Shared targets: 27/37 (<80%)', ha='center', fontsize=7)
    fig.text(.1375, .205, 'Regulon activity unavailable', ha='center', fontsize=7)
    rng = np.random.default_rng(SEED)
    bg = rng.permutation(len(t)); fg = rng.permutation(np.flatnonzero(primary))
    colors = np.asarray([PALETTE[v] for v in data['C_obs'].type])
    c.scatter(t[bg] * factor, score[bg], s=C_POINT_AREA, c=colors[bg], alpha=.20, linewidths=0)
    c.scatter(t[fg] * factor, score[fg], s=C_POINT_AREA, c=colors[fg], alpha=.90, linewidths=0)
    c.plot(np.asarray(data['C_grid']) * factor, curve, color='black', lw=1.05, zorder=5)
    c.axvline(float(data['C_fork_t']) * factor, color='black', ls=(0, (4, 3)), lw=.65)
    c.set_xlim(0, 6.07); c.set_ylim(0, C_YMAX)
    c.set_xticks([0, 2, 4, 6]); c.set_yticks([0, .45, .90])
    c.tick_params(labelsize=7, length=2.5, width=.6, pad=2)
    c.set_xlabel('pseudotime', fontsize=8, labelpad=3)
    c.set_ylabel('Scaled gene-set RNA', fontsize=8, labelpad=4)
    c.set_title('Transient targets (n=12)', fontsize=9, pad=6)
    handles = [Line2D([0], [0], marker='o', ls='None', color=PALETTE[k], label=k, markersize=3)
               for k in ('CVG', 'VGN')]
    c.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, -.29), ncol=2,
        frameon=False, fontsize=7, handletextpad=.25, columnspacing=.8)
    valid = np.flatnonzero(np.isfinite(dscore))
    valid = valid[np.argsort(dscore[valid], kind='stable')]
    cmap = LinearSegmentedColormap.from_list('module_fitted_RNA', ['#FFFFFF', '#2D7184'])
    sc = d.scatter(xy[valid, 0], xy[valid, 1], c=dscore[valid], s=POINT_AREA * 1.2,
        cmap=cmap, norm=Normalize(0, D_VMAX), alpha=1, linewidths=0)
    d.set_title('Transient targets', color='#2D7184', fontsize=10, pad=7)
    barax = fig.add_axes([.612, .235, .105, .022])
    bar = fig.colorbar(sc, cax=barax, orientation='horizontal', ticks=[0, .5, 1])
    bar.outline.set_linewidth(.45); bar.ax.tick_params(labelsize=6.5, length=2, pad=2)
    fig.text(.665, .15, 'Mean scaled fitted target RNA', ha='center', fontsize=6.8)
    netqa = draw_tbx3_network(network, [g for g in data['module_genes'] if g != 'Tbx3'])
    fig.text(.885, .205, 'Inferred SCENIC links', ha='center', fontsize=7)
    fig.text(.885, .16, '12 members: Tbx3 + 11 targets', ha='center', fontsize=6.5)
    for label, x in [('b', .01), ('c', .28), ('d', .545)]:
        fig.text(x, .91, label, fontsize=13, fontweight='bold')
    fig.text(.5, .065,
        'B/D: original E9.5-E18.5 UMAP; D colors 168 matched CVG cells. C: actual E9.5-E13.5 CVG/VGN.',
        fontsize=6.4, ha='center')
    fig.text(.5, .026,
        'Gray D cells have no matched path fit. C uses scaled raw RNA; D uses scaled fitted target RNA.',
        fontsize=6.4, ha='center')
    result = save_figure(fig, 'Tbx3-BCD')
    result.update({'network': netqa, 'B_cells': 7355, 'C_cells': 1010,
        'C_fit_cells': 478, 'D_colored_cells': 168, 'D_other_gray_cells': 7187,
        'C_time_display_factor': float(factor), 'D_formula': 'Mean of gene-wise minmax fitted target RNA'})
    return result

In [ ]:
#7. =========== Render, retain notebook previews and record delivery checks
figure_qa = {'lines': render_lines(data), 'Tbx3': render_tbx3(data)}
with h5py.File(RESULT_FILE, 'r+') as handle:
    if 'figure_qa' in handle: del handle['figure_qa']
    write_value(handle, 'figure_qa', figure_qa)
    if 'notebook_code' in handle: del handle['notebook_code']
    import nbformat
    saved = nbformat.read(NOTEBOOK_FILE, as_version=4)
    code = '\n\n'.join(cell.source for cell in saved.cells if cell.cell_type == 'code')
    handle.create_dataset('notebook_code', data=code, dtype=h5py.string_dtype())
    handle.attrs['notebook_code_sha256'] = hashlib.sha256(code.encode()).hexdigest()
assert len([p for p in RESULT_DIR.iterdir() if p.is_file()]) <= 2
print('Saved 2 PNG/PDF figure pairs and one result HDF5 file.')

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
# Set this storage path independently before running. VGN-lines.png.
display(Image(filename=str(ROOT), width=1100))

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
# Set this storage path independently before running. Tbx3-BCD.png.
display(Image(filename=str(ROOT), width=1200))

The two formulas for C and D are retained separately. Gray B/D cells are unavailable scores, not zero activity. The result directory contains one HDF5 file; all preparation and plotting code is in this notebook.

## Standalone early Tbx3 RNA panel

This panel displays RNA expression on the unchanged original UMAP, with limits enlarged to E9.5-E13.5 cells. It is distinct from the regulon-activity B panel. Gray points indicate zero measured Tbx3 RNA. The original atlas contains no actual VGN cells.

In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
#10. =========== Tbx3 RNA expression in the early region of the original UMAP
def prepare_tbx3_early_rna(data):
    # Set this storage path independently before running. expression.h5.
    source = ROOT
    source_sha = _vgn_digest(source)
    early = data['B_obs'].age.between(9.5, 13.5).to_numpy()
    rows = np.flatnonzero(early)
    with h5py.File(source, 'r') as handle:
        np.testing.assert_array_equal(handle['cell_ids'].asstr()[:], data['B_cell_ids'])
        position = np.flatnonzero(handle['gene_symbols'].asstr()[:] == 'Tbx3')
        assert len(position) == 1
        expression = handle['expression'][:, int(position[0])][early]
        formula = handle.attrs['calculation']
    assert _vgn_digest(source) == source_sha
    assert len(rows) == 1234 and np.isfinite(expression).all() and (expression >= 0).all()
    assert int((expression > 0).sum()) == 166
    return {'gene': 'Tbx3', 'metric': 'RNA expression, not regulon AUCell activity',
        'unit': 'log10(1+CPM)', 'source_calculation': formula,
        'source_path': str(source.relative_to(ROOT)), 'source_sha256': source_sha,
        'original_atlas_rows': rows, 'cell_ids': data['B_cell_ids'][early],
        'xy': data['B_xy'][early], 'RNA': expression,
        'stages': data['B_obs'].stage_standardized.to_numpy()[early],
        'types': data['B_obs'].cell_type_plot.to_numpy()[early],
        'zero_meaning': 'Measured Tbx3 RNA count equals zero',
        'coordinates': 'Unchanged original UMAP; early cells subset and axes limits enlarged',
        'actual_VGN_present': False}


def render_tbx3_early_rna(rna):
    FIGSIZE = (6.0, 3.6)
    POINT_AREA = 3.2
    xy, expression = np.asarray(rna['xy']), np.asarray(rna['RNA'])
    maximum = float(expression.max())
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9, 'text.color': 'black'})
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_axes([.045, .27, .91, .56])
    ax.scatter(xy[:, 0], xy[:, 1], s=POINT_AREA, c='#D4D4D4', alpha=.65, linewidths=0)
    order = np.flatnonzero(expression > 0)
    order = order[np.argsort(expression[order], kind='stable')]
    cmap = LinearSegmentedColormap.from_list('Tbx3_RNA', ['#D4D4D4', '#FAD5D0', '#D71920'])
    norm = Normalize(vmin=0, vmax=maximum)
    sc = ax.scatter(xy[order, 0], xy[order, 1], s=POINT_AREA,
        c=expression[order], cmap=cmap, norm=norm, alpha=1, linewidths=0)
    span = np.ptp(xy, axis=0)
    ax.set_xlim(xy[:, 0].min() - .04 * span[0], xy[:, 0].max() + .04 * span[0])
    ax.set_ylim(xy[:, 1].min() - .07 * span[1], xy[:, 1].max() + .07 * span[1])
    ax.set_aspect('equal'); ax.axis('off')
    fig.text(.035, .92, 'b', fontsize=13, fontweight='bold')
    fig.text(.5, .925, r'$\it{Tbx3}$ RNA expression', ha='center', va='center', fontsize=12)
    fig.text(.5, .855, 'E9.5-E13.5', ha='center', va='center', fontsize=9)
    barax = fig.add_axes([.34, .17, .32, .025])
    bar = fig.colorbar(sc, cax=barax, orientation='horizontal')
    bar.set_ticks([0, 1, 2, maximum]); bar.set_ticklabels(['0', '1', '2', f'{maximum:.2f}'])
    bar.ax.tick_params(length=2, labelsize=7, pad=2); bar.outline.set_linewidth(.5)
    fig.text(.5, .065, 'RNA: log10(1 + CPM)', ha='center', fontsize=8)
    result = save_figure(fig, 'Tbx3-B-RNA')
    result.update({'cells': len(expression), 'positive_cells': len(order),
        'metric': rna['metric'], 'color_limits': [0, maximum],
        'original_coordinates_preserved': True})
    return result


if 'Tbx3_B_RNA' not in data or REBUILD_RESULTS:
    data['Tbx3_B_RNA'] = prepare_tbx3_early_rna(data)
    with h5py.File(RESULT_FILE, 'r+') as handle:
        if 'Tbx3_B_RNA' in handle['data']: del handle['data/Tbx3_B_RNA']
        write_value(handle['data'], 'Tbx3_B_RNA', data['Tbx3_B_RNA'])
rna_qa = render_tbx3_early_rna(data['Tbx3_B_RNA'])
with h5py.File(RESULT_FILE, 'r+') as handle:
    if 'Tbx3_B_RNA' in handle['figure_qa']: del handle['figure_qa/Tbx3_B_RNA']
    write_value(handle['figure_qa'], 'Tbx3_B_RNA', rna_qa)
    import nbformat
    current = nbformat.read(NOTEBOOK_FILE, as_version=4)
    code = '\n\n'.join(c.source for c in current.cells if c.cell_type == 'code')
    if 'notebook_code' in handle: del handle['notebook_code']
    handle.create_dataset('notebook_code', data=code, dtype=h5py.string_dtype())
    handle.attrs['notebook_code_sha256'] = hashlib.sha256(code.encode()).hexdigest()
print('Saved Tbx3-B-RNA.png and .pdf: 1234 early cells, 166 with detected Tbx3 RNA.')


In [ ]:

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")
# Set this storage path independently before running. Tbx3-B-RNA.png.
display(Image(filename=str(ROOT), width=900))

## Original standalone source: fig4_VGN_input_analysis.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Exact VGN input-producer functions producing Early-data.h5; the superseded DPT fit/export workflow is omitted.
# Original source label: Trajectory.py (fig4_VGN_input_analysis)
# Original source SHA256: 3a2d6d9b7aa5d28f67b548dbbda8e2b76c985c07a1750f7ce2f63a78208f6d06
# Exact input-producer definitions extracted; only path placeholders and a prepare_data entry point were added; no analysis was executed.

"""Build a same-source early otic-neuron matrix and audit CVG-to-VGN trajectories."""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Shared configuration
from pathlib import Path
import os
import sys
import json
import hashlib
sys.dont_write_bytecode = True
# Set this storage path independently before running.
ROOT = ROOT
OUT = Path(__file__).resolve().parent
# Set this storage path independently before running.
WORK = ROOT
WORK.mkdir(exist_ok=True,parents=True)
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMBA_NUM_THREADS']:
    os.environ[key] = '1'
# Set this storage path independently before running.
os.environ['NUMBA_CACHE_DIR'] = str(ROOT)
# Set this storage path independently before running.
os.environ['MPLCONFIGDIR'] = str(ROOT)
os.environ['TMP'] = os.environ['TEMP'] = os.environ['TMPDIR'] = str(WORK)
import h5py
import numpy as np
import pandas as pd
from scipy import sparse,stats

SEED = 42
PANEL = ('Meis2','Npas3','Sall3','Tlx3','Tbx3','Onecut1','Pcdh19')
# Set this storage path independently before running. Early-data.h5. Produced by fig4_VGN_input_analysis.py.
DATA_FILE = ROOT
# Set this storage path independently before running. Trajectory.h5. Produced by fig4_VGN_trajectory_analysis.py.
RESULT_FILE = ROOT
SOURCE = 'GSE178931'


def array(node):
    if isinstance(node,h5py.Dataset):
        return node.asstr()[:] if node.dtype.kind in 'OUS' else node[:]
    codes = node['codes'][:]
    categories = array(node['categories'])
    out = np.full(len(codes),'',dtype=object)
    use = codes >= 0
    out[use] = categories[codes[use]]
    return out


def frame(group):
    return pd.DataFrame({key:array(group[key]) for key in group})


def put(h,key,value):
    if isinstance(value,(dict,list,tuple)):
        h.create_dataset(key,data=json.dumps(value,allow_nan=False),dtype=h5py.string_dtype())
        return
    a = np.asarray(value)
    if a.dtype.kind in 'OUS':
        h.create_dataset(key,data=a.astype(object),dtype=h5py.string_dtype())
    else:
        h.create_dataset(key,data=a,compression='gzip' if a.ndim else None,shuffle=bool(a.ndim))


def put_frame(h,key,value):
    node = h.create_group(key)
    node.attrs['columns'] = json.dumps(value.columns.tolist())
    for col in value:
        put(node,col,value[col].to_numpy())


def digest(path):
    x = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):
            x.update(block)
    return x.hexdigest()


#2. =========== Freeze the exact curated early cells and native count axis
def prepare_data():
    if DATA_FILE.exists():
        with h5py.File(DATA_FILE,'r') as h:
            assert len(h['cell_ids']) == 1681
        print('Reuse immutable Early-data.h5',flush=True)
        return
    # Set this storage path independently before running. RNA-screen.h5.
    with h5py.File(ROOT,'r') as h:
        old = frame(h['cells'])
    # Set this storage path independently before running. activity-screen.h5.
    with h5py.File(ROOT,'r') as h:
        annotation = frame(h['obs'])
    assert np.array_equal(old.cell_id,annotation['_index'])
    chosen = old.study_id.eq(SOURCE)
    assert int(chosen.sum()) == 1161
    assert old.loc[chosen,'source_state_evaluation_allowed'].all()
    assert not old.loc[chosen,'source_rna_mixture_review_flag'].any()
    obs = old.loc[chosen,['cell_id','library_id','stage_standardized','library_total']].copy()
    obs['type'] = np.where(annotation.loc[chosen,'cell_type_plot'].eq('Differentiating SGN'),'SGN','CVG')
    obs['original_annotation'] = annotation.loc[chosen,'cell_type_plot'].to_numpy()
    obs = obs.rename(columns={'library_total':'full_total'})
    # Set this storage path independently before running. VGN520-common-counts.npz.
    with np.load(ROOT,allow_pickle=False) as h:
        v = pd.DataFrame({'cell_id':h['cells'],'library_id':h['library_id'],
                           'stage_standardized':h['stage'],'full_total':h['full_endogenous_total'],
                           'type':'VGN','original_annotation':'Curated VGN520 reference'})
    obs = pd.concat([obs,v],ignore_index=True)
    assert obs.cell_id.is_unique and len(obs) == 1681
    assert obs.type.value_counts().to_dict() == {'SGN':671,'VGN':520,'CVG':490}
    assert obs.stage_standardized.value_counts().to_dict() == {'E13.5':1179,'E11.5':490,'E9.5':12}
    obs['study_id'] = SOURCE
    obs['age'] = obs.stage_standardized.str[1:].astype(float)
    blocks,block_ids,source_hashes = [],[],{}
    axis = None
    feature_flags = None
    qc_records = []
    for (stage,library),part in obs.groupby(['stage_standardized','library_id'],sort=True):
        # Set this storage path independently before running.
        path = ROOT/f'{stage.replace(".5","")}_{SOURCE}__{library}.h5ad'
        with h5py.File(path,'r') as h:
            ids = array(h['obs'][h['obs'].attrs['_index']])
            pos = pd.Index(ids).get_indexer(part.cell_id)
            assert (pos >= 0).all()
            genes = array(h['var/gene_id_analysis'])
            assert pd.Index(genes).is_unique
            flags = {k:array(h['var'][k]).astype(bool) for k in
                     ['eligible_for_within_dataset_analysis','mt_feature','technical_feature',
                      'external_spike_feature','reporter_or_vector_feature']}
            if axis is None:
                axis = genes
                feature_flags = flags
                keep = flags['eligible_for_within_dataset_analysis'] & ~flags['mt_feature']
                keep &= ~flags['technical_feature'] & ~flags['external_spike_feature'] & ~flags['reporter_or_vector_feature']
                keep &= ~pd.Series(genes).str.lower().str.startswith('mt-').to_numpy()
            else:
                np.testing.assert_array_equal(axis,genes)
                for k in flags:
                    np.testing.assert_array_equal(feature_flags[k],flags[k])
            assert array(h['obs/qc_pass'])[pos].all()
            total = array(h['obs/total_counts_endogenous'])[pos]
            np.testing.assert_array_equal(total,part.full_total)
            g = h['X']
            assert g.attrs['encoding-type'] == 'csr_matrix'
            matrix = sparse.csr_matrix((g['data'][:],g['indices'][:],g['indptr'][:]),shape=tuple(g.attrs['shape']))
            block = matrix[pos][:,keep].tocsr()
            assert (block.data >= 0).all() and np.array_equal(block.data,np.round(block.data))
            block = block.astype(np.int32)
            for key in ['n_genes_endogenous','pct_mito','pct_ribo','cell_type_curated',
                        'author_reference_cell_type','doublet_status','biological_replicate_id']:
                if key in h['obs']:
                    values = array(h['obs'][key])[pos]
                    for index,value in zip(part.index,values):
                        qc_records.append((index,key,value))
        blocks.append(block)
        block_ids.extend(part.cell_id)
        source_hashes[str(path.relative_to(ROOT))] = digest(path)
        print('Read native library:',library,'cells:',len(part),flush=True)
    merged = sparse.vstack(blocks,format='csr')
    order = pd.Index(block_ids).get_indexer(obs.cell_id)
    counts = merged[order].tocsr()
    genes = axis[keep]
    for index,key,value in qc_records:
        if key not in obs:
            obs[key] = '' if isinstance(value,(str,bytes)) else np.nan
        obs.loc[index,key] = value
    assert (pd.Index(genes).get_indexer(PANEL) >= 0).all()
    assert np.isfinite(obs.full_total).all() and obs.full_total.gt(0).all()
    cpm = counts.toarray().astype(np.float64) / obs.full_total.to_numpy()[:,None] * 1e6
    RNA = np.log10(1+cpm).astype(np.float32)
    norm_log = np.log1p(cpm/100).astype(np.float32)
    # Keep the curated cells; blood-cell metadata and ambient erythroid signal are audited.
    blood_annotation = obs.original_annotation.str.contains('eryth|blood|immune|macroph|leuk|hematop',case=False,regex=True)
    assert not blood_annotation.any()
    hb = pd.Series(genes).str.match(r'^Hb[ab]-',case=False).to_numpy()
    obs['hemoglobin_pct'] = counts[:,hb].sum(axis=1).A1 / obs.full_total.to_numpy() * 100
    for g in ['Ptprc','Lyz2','Lst1','Alas2','Gypa']:
        j = pd.Index(genes).get_indexer([g])[0]
        if j >= 0:
            obs['count_'+g] = counts[:,j].toarray().ravel()
    methods = {'source':SOURCE,'cells':1681,'curated_CVG':490,'curated_SGN':671,'curated_VGN':520,
        'stages':['E9.5','E11.5','E13.5'],'cross_source_E10_or_late_cells':False,
        'RNA':'log10(1 + raw_counts / full_endogenous_total * 1e6)',
        'norm_log':'log1p(raw_counts / full_endogenous_total * 1e4)',
        'denominator':'Full endogenous count total before mitochondrial feature exclusion; never recomputed from this filtered axis',
        'feature_filter':'Native within-dataset eligible; remove technical, external-spike, reporter/vector and mitochondrial features',
        'original_features':len(axis),'retained_features':len(genes),
        'cell_filter':'Reuse all 1681 curated QC-passing cells; no cell removed by candidate expression, cycle score or shape',
        'blood_audit':'No blood/immune original annotations; erythroid/immune counts retained for diagnostic review, not single-gene exclusion',
        'cell_order':'Existing source atlas cells, then disjoint VGN520 cells',
        'animal_replicates':'Metadata retained; library identity is not automatically animal identity'}
    with h5py.File(DATA_FILE,'w') as h:
        h.attrs.update(schema='VGN_BC_early_native_v1',immutable_input=True)
        put(h,'cell_ids',obs.cell_id.to_numpy())
        put(h,'genes',genes)
        g = h.create_group('counts')
        for key,value in [('data',counts.data),('indices',counts.indices),('indptr',counts.indptr),('shape',np.array(counts.shape))]:
            put(g,key,value)
        g.attrs['format'] = 'csr'
        h.create_dataset('RNA',data=RNA,compression='gzip',shuffle=True,chunks=(128,256))
        h.create_dataset('norm_log',data=norm_log,compression='gzip',shuffle=True,chunks=(128,256))
        put_frame(h,'obs',obs)
        put(h,'methods_json',methods)
        put(h,'source_sha256_json',source_hashes)
    with h5py.File(DATA_FILE,'r') as h:
        np.testing.assert_array_equal(h['cell_ids'].asstr()[:],obs.cell_id)
        np.testing.assert_allclose(h['RNA'][:],RNA,atol=0,rtol=0)
    print('Early-data ready:',str(DATA_FILE),'shape:',counts.shape,flush=True)


if __name__ == "__main__":
    prepare_data()


## Original standalone source: fig4_VGN_trajectory_analysis.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# Accepted 1,010-cell CVG/VGN trajectory: HVG3000, 40 nodes, lambda=3, sigma=0.095, 1000 steps, root=17.
# Original source label: Trajectory.py (fig4_VGN_trajectory_analysis)
# Original source SHA256: 75e43dfd37f186f204cf9a296ca9c349b672ebd88b8cfdb5186ecaff3e846582
# Only storage addresses and release comments were changed; analysis was not executed.

"""Actual CVG/VGN principal trees with the original SimplePPT projection method."""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Frozen inputs and native principal-tree runtime
from pathlib import Path
import os
import sys
import json
import hashlib
import time
import contextlib
import io
import warnings
import re

# Set this storage path independently before running.
ROOT = ROOT
OUT = Path(__file__).resolve().parent
# Set this storage path independently before running.
WORK = ROOT
WORK.mkdir(exist_ok=True)
# Set this storage path independently before running. Early-data.h5. Produced by fig4_VGN_input_analysis.py.
INPUT = ROOT
# Set this storage path independently before running.
PACKAGES = ROOT
# Set this storage path independently before running. Trajectory.h5. Produced by fig4_VGN_trajectory_analysis.py.
RESULT = ROOT
ANCHORS = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ[key] = '1'
# Set this storage path independently before running.
os.environ.update(MPLCONFIGDIR=str(ROOT), NUMBA_CACHE_DIR=str(ROOT))
sys.dont_write_bytecode = True
sys.path.insert(0, str(PACKAGES))
import h5py
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy import sparse
from scipy.special import softmax
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits
import simpleppt
threadpool_limits(1)
simpleppt.settings.verbosity = 0


def digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4194304), b''):
            value.update(block)
    return value.hexdigest()


def array_digest(value):
    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()


def readframe(node):
    columns = json.loads(node.attrs['columns']) if 'columns' in node.attrs else list(node)
    columns = [k for k in columns if k != '_index']
    table = pd.DataFrame({k: node[k].asstr()[:] if node[k].dtype.kind in 'OSU' else node[k][:] for k in columns})
    if '_index' in node:
        table.index = node['_index'].asstr()[:]
    return table


def writeframe(handle, name, table):
    node = handle.create_group(name)
    node.attrs['columns'] = json.dumps(table.columns.tolist())
    node.create_dataset('_index', data=table.index.astype(str).to_numpy(dtype=object), dtype=h5py.string_dtype())
    for key in table:
        value = table[key].to_numpy()
        if value.dtype.kind in 'OUS':
            node.create_dataset(key, data=table[key].fillna('').astype(str).to_numpy(dtype=object), dtype=h5py.string_dtype())
        else:
            node.create_dataset(key, data=value, compression='gzip', shuffle=True)


def put(handle, name, value):
    if isinstance(value, (dict, list, tuple)):
        handle.create_dataset(name, data=json.dumps(value, allow_nan=False), dtype=h5py.string_dtype())
        return
    a = np.asarray(value)
    if a.dtype.kind in 'OUS':
        handle.create_dataset(name, data=a.astype(object), dtype=h5py.string_dtype())
    else:
        handle.create_dataset(name, data=a, compression='gzip' if a.ndim else None)


def correlation(x, y):
    value = float(spearmanr(x, y).statistic)
    return value if np.isfinite(value) else None


#2. =========== Original unrestricted native fitting and numerical contraction
def fit_tree(x, nodes, lam, sigma, steps, seed):
    CONTRACTION_TOLERANCE = 1e-8
    FIXED_POINT_TOLERANCE = 1e-4
    started = time.perf_counter()
    with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):
        warnings.simplefilter('ignore')
        model = simpleppt.ppt(x, Nodes=nodes, lam=lam, sigma=sigma, metric='euclidean',
            nsteps=steps, err_cut=-1., seed=seed, init=None, device='cpu', progress=False)
    f, b, r = np.asarray(model.F), np.asarray(model.B), np.asarray(model.R)
    assert np.isfinite(f).all() and np.isfinite(r).all()
    np.testing.assert_allclose(r.sum(1), 1., atol=1e-6)
    soft = softmax(-cdist(x, f.T) / sigma, axis=1)
    updated = np.linalg.solve(lam * (np.diag(b.sum(0)) - b) + np.diag(soft.sum(0)), soft.T @ x).T
    residual = float(np.linalg.norm(updated - f, axis=0).max())
    parent = np.arange(len(b))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return int(i)

    for i, j in np.argwhere(np.triu(b, 1) > 0):
        if np.linalg.norm(f[:, i] - f[:, j]) < CONTRACTION_TOLERANCE:
            parent[find(j)] = find(i)
    groups = np.array([find(i) for i in range(len(b))])
    unique, codes = np.unique(groups, return_inverse=True)
    cf = np.column_stack([f[:, groups == key].mean(1) for key in unique])
    cb = np.zeros((len(unique), len(unique)), dtype=np.int8)
    for i, j in np.argwhere(np.triu(b, 1) > 0):
        if codes[i] != codes[j]:
            cb[codes[i], codes[j]] = cb[codes[j], codes[i]] = 1
    assert connected_components(csr_matrix(cb))[0] == 1
    assert cb.sum() == 2 * (len(cb) - 1)
    return {'F': cf, 'B': cb, 'raw_F': f, 'raw_B': b,
        'zero_length_nodes_contracted': int(len(b) - len(cb)), 'fixed_point_residual': residual,
        'numerically_settled': residual <= FIXED_POINT_TOLERANCE, 'seconds': time.perf_counter() - started}


#3. =========== Original nearest-edge alpha and rooted arc distance
def rooted_graph(f, b, root):
    parent = np.full(len(b), -1, int)
    distance = np.full(len(b), np.nan)
    parent[root], distance[root] = root, 0.
    order = [int(root)]
    for u in order:
        for v in np.flatnonzero(b[u]):
            if v == parent[u]:
                continue
            assert parent[v] == -1, 'Graph has a cycle'
            parent[v] = u
            distance[v] = distance[u] + np.linalg.norm(f[:, v] - f[:, u])
            order.append(int(v))
    assert len(order) == len(b)
    edges = np.array([(int(parent[v]), int(v)) for v in order[1:]], int)
    return parent, distance, edges


def graph_path(b, start, end):
    previous, queue = {int(start): None}, [int(start)]
    for u in queue:
        for v in np.flatnonzero(b[u]):
            if int(v) not in previous:
                previous[int(v)] = u
                queue.append(int(v))
    path = [int(end)]
    while path[-1] != start:
        path.append(previous[path[-1]])
    return path[::-1]


def nearest_edge_projection(x, f, edges):
    a = f[:, edges[:, 0]].T
    d = f[:, edges[:, 1]].T - a
    den = (d * d).sum(1)
    assert np.all(den > 1e-20)
    alpha = np.clip(np.einsum('ned,ed->ne', x[:, None, :] - a, d) / den, 0., 1.)
    residual = ((x[:, None, :] - a - alpha[:, :, None] * d) ** 2).sum(2)
    eid = residual.argmin(1)
    return eid, alpha[np.arange(len(x)), eid], np.sqrt(residual[np.arange(len(x)), eid])


def project_and_select(model, x, obs):
    f, b = model['F'], model['B']
    degree = b.sum(0)
    tips = np.flatnonzero(degree == 1)
    assert len(tips) >= 2, 'No natural root-to-tip path exists'
    nearest = cdist(x, f.T).argmin(1)
    age = obs.age.to_numpy(float)
    mass = np.bincount(nearest, minlength=len(b))
    mean_age = np.divide(np.bincount(nearest, weights=age, minlength=len(b)), mass,
        out=np.full(len(b), np.inf), where=mass > 0)
    occupied = tips[mass[tips] > 0]
    tip_only_root = int(min(occupied, key=lambda v: (mean_age[v], int(v))))
    earliest_age = float(age.min())
    earliest = np.flatnonzero(age == earliest_age)
    assert len(earliest) == 12 and obs.stage_standardized.iloc[earliest].eq('E9.5').all()
    root = int(cdist(x[earliest].mean(axis=0, keepdims=True), f.T).argmin())
    earliest_nearest = nearest[earliest]
    parent, rd, edges = rooted_graph(f, b, root)
    eid, alpha, residual = nearest_edge_projection(x, f, edges)
    lengths = np.linalg.norm(f[:, edges[:, 1]] - f[:, edges[:, 0]], axis=0)
    t = rd[edges[eid, 0]] + alpha * lengths[eid]
    edge_lookup = {tuple(sorted(pair)): i for i, pair in enumerate(edges)}
    segments, edge_segment = [], np.full(len(edges), -1, int)
    milestones = set(np.flatnonzero(degree != 2)) | {root}
    for start in sorted(milestones, key=lambda v: (rd[v], int(v))):
        for child in np.flatnonzero(parent == start):
            if child == start:
                continue
            path = [int(start), int(child)]
            while path[-1] not in milestones:
                onward = np.flatnonzero(parent == path[-1])
                assert len(onward) == 1
                path.append(int(onward[0]))
            eids = np.array([edge_lookup[tuple(sorted(pair))] for pair in zip(path[:-1], path[1:])], int)
            sid = len(segments)
            edge_segment[eids] = sid
            segments.append({'segment_id': sid, 'seg': f's{sid}', 'start': int(start), 'end': int(path[-1]),
                'node_path': path, 'edge_ids': eids.tolist(), 'length': float(lengths[eids].sum()),
                'start_t': float(rd[start]), 'end_t': float(rd[path[-1]])})
    assert (edge_segment >= 0).all()
    target = obs.type.eq('VGN').to_numpy() & obs.stage_standardized.eq('E13.5').to_numpy()
    tip_rows = []
    for tip in tips:
        path = graph_path(b, root, int(tip))
        path_edges = np.array([edge_lookup[tuple(sorted(pair))] for pair in zip(path[:-1], path[1:])], int)
        # Natural terminal branch: walk backwards from each tip to its first milestone.
        terminal = [int(tip)]
        previous = None
        u = int(tip)
        while True:
            neighbors = [int(v) for v in np.flatnonzero(b[u]) if v != previous]
            if not neighbors:
                break
            v = neighbors[0]
            terminal.append(v)
            if degree[v] != 2 or v == root:
                break
            previous, u = u, v
        terminal_edges = [edge_lookup[tuple(sorted(pair))] for pair in zip(terminal[:-1], terminal[1:])]
        terminal_cells = np.isin(eid, terminal_edges)
        k, n = int((terminal_cells & target).sum()), int(terminal_cells.sum())
        coverage = k / int(target.sum())
        purity = k / n if n else 0.
        score = 2 * coverage * purity / (coverage + purity) if coverage + purity else 0.
        path_cells = np.isin(eid, path_edges)
        row = {'tip': int(tip), 'is_root': bool(tip == root), 'nearest_node_cells': int(mass[tip]),
            'nearest_node_mean_age': float(mean_age[tip]) if mass[tip] else None,
            'nearest_node_E9_cells': int(((nearest == tip) & obs.stage_standardized.eq('E9.5').to_numpy()).sum()),
            'terminal_cells': n, 'terminal_E13_VGN': k, 'E13_VGN_coverage': coverage,
            'E13_VGN_purity': purity, 'E13_VGN_F1': score, 'root_to_tip_cells': int(path_cells.sum()),
            'root_to_tip_VGN': int((path_cells & obs.type.eq('VGN').to_numpy()).sum()),
            'root_to_tip_CVG': int((path_cells & obs.type.eq('CVG').to_numpy()).sum()),
            'terminal_edge_ids': terminal_edges, 'path_edge_ids': path_edges.tolist(), 'path_nodes': path}
        tip_rows.append(row)
    nonroot = [r for r in tip_rows if not r['is_root']]
    endpoint = max(nonroot, key=lambda r: (r['E13_VGN_F1'], r['terminal_E13_VGN'], r['E13_VGN_purity'], -r['tip']))
    old_point = (1 - alpha[:, None]) * f[:, edges[eid, 0]].T + alpha[:, None] * f[:, edges[eid, 1]].T
    old_t, old_residual = t.copy(), residual.copy()
    at_root = (edges[eid, 0] == root) & (alpha == 0.)
    canonicalized = at_root & ~np.isin(eid, endpoint['path_edge_ids'])
    eid[canonicalized] = endpoint['path_edge_ids'][0]
    alpha[canonicalized] = 0.
    new_point = (1 - alpha[:, None]) * f[:, edges[eid, 0]].T + alpha[:, None] * f[:, edges[eid, 1]].T
    np.testing.assert_array_equal(new_point, old_point)
    np.testing.assert_array_equal(t, old_t)
    np.testing.assert_array_equal(residual, old_residual)
    for row in tip_rows:
        mask = np.isin(eid, row['path_edge_ids'])
        row['root_to_tip_cells'] = int(mask.sum())
        row['root_to_tip_VGN'] = int((mask & obs.type.eq('VGN').to_numpy()).sum())
        row['root_to_tip_CVG'] = int((mask & obs.type.eq('CVG').to_numpy()).sum())
    primary = np.isin(eid, endpoint['path_edge_ids'])
    path_forks = [int(node) for node in endpoint['path_nodes'] if degree[node] >= 3]
    fork_t = float(rd[path_forks[0]]) if path_forks else np.nan
    seg = np.asarray([f's{sid}' for sid in edge_segment[eid]], dtype=object)
    assert np.isfinite(t).all() and primary.any()
    observed = np.sort(t[primary])
    gap_index = int(np.argmax(np.diff(observed)))
    gap = [float(observed[gap_index]), float(observed[gap_index + 1])]
    left = primary & (t <= gap[0])
    right = primary & (t >= gap[1])
    gap_groups = lambda m: obs.loc[m].groupby(['type', 'stage_standardized']).size().to_dict()
    gap_report = {'largest_unsampled_native_t_interval': gap, 'largest_unsampled_native_t_width': gap[1] - gap[0],
        'width_fraction_primary_range': float((gap[1] - gap[0]) / np.ptp(observed)),
        'left_cells': int(left.sum()), 'right_cells': int(right.sum()),
        'left_composition': {f'{k[0]}_{k[1]}': int(v) for k, v in gap_groups(left).items()},
        'right_composition': {f'{k[0]}_{k[1]}': int(v) for k, v in gap_groups(right).items()},
        'interpretation': 'An unsampled projection interval does not identify a developmental discontinuity; principal-tree edges do not add observed cells.'}
    edge_counts = np.bincount(eid, minlength=len(edges))
    path_edge_ids = np.asarray(endpoint['path_edge_ids'], int)
    gap_report['primary_edges_without_assigned_cells'] = path_edge_ids[edge_counts[path_edge_ids] == 0].tolist()
    summary = {'cells': len(x), 'nodes': len(b), 'natural_tips': tips.tolist(),
        'branchpoints': np.flatnonzero(degree > 2).tolist(), 'root': root,
        'root_nearest_mean_age': float(mean_age[root]), 'endpoint': int(endpoint['tip']),
        'root_degree': int(degree[root]), 'tip_only_root': tip_only_root,
        'root_rule': 'Existing principal node nearest the12 E9.5 PCA centroid; internal degree2 nodes are allowed',
        'E9_nearest_node_counts': {str(k): int(v) for k, v in zip(*np.unique(earliest_nearest, return_counts=True))},
        'all_E9_nearest_selected_root': bool((earliest_nearest == root).all()),
        'root_endpoint_canonicalized_cells': int(canonicalized.sum()),
        'root_endpoint_projection_unchanged': True,
        'primary_E9_cells': int((primary & obs.stage_standardized.eq('E9.5').to_numpy()).sum()),
        'first_primary_natural_fork': path_forks[0] if path_forks else None,
        'fork_t_native': fork_t if np.isfinite(fork_t) else None,
        'endpoint_terminal_E13_VGN': endpoint['terminal_E13_VGN'],
        'endpoint_terminal_E13_VGN_coverage': endpoint['E13_VGN_coverage'],
        'endpoint_terminal_E13_VGN_purity': endpoint['E13_VGN_purity'],
        'primary_cells': int(primary.sum()), 'primary_CVG': int((primary & obs.type.eq('CVG').to_numpy()).sum()),
        'primary_VGN': int((primary & obs.type.eq('VGN').to_numpy()).sum()),
        'age_t_rho_scope': correlation(age, t), 'age_t_rho_primary': correlation(age[primary], t[primary]),
        'VGN_age_t_rho': correlation(age[obs.type.eq('VGN')], t[obs.type.eq('VGN')]),
        'projection_RMS': float(np.sqrt(np.mean(residual ** 2))), 't_native_max': float(t.max()),
        'numerically_settled': model['numerically_settled'], 'fixed_point_residual': model['fixed_point_residual'],
        'zero_length_nodes_contracted': model['zero_length_nodes_contracted'], 'seconds': model['seconds'],
        'support_gap': gap_report}
    return {'root': root, 'endpoint': int(endpoint['tip']), 'F': f, 'B': b, 'parent': parent,
        'root_distance': rd, 'edges': edges, 'edge_id': eid, 'alpha': alpha, 'residual': residual,
        'edge_length': lengths, 'edge_cell_counts': edge_counts, 'nearest_node': nearest,
        't_native': t, 't': t.copy(), 't_display': 6 * t / t.max(), 'fork_t': fork_t,
        'seg': seg, 'primary_path': primary,
        'segments': segments, 'tip_audit': tip_rows, 'QA': summary}


#4. =========== Prespecified HVG and topology sensitivity, without marker-driven selection
def main():
    HVG_COUNTS = (2000, 3000, 5000)
    BASELINE_HVG = 3000
    MIN_HVG_CELLS, N_PCS, PCA_SEED = 3, 30, 42
    BLOOD_GENES = ('Hba-a1', 'Hba-a2', 'Hbb-bs', 'Hbb-bt', 'Hbb-y', 'Hbb-bh1',
        'Alas2', 'Gypa', 'Slc4a1', 'Pf4', 'Ppbp', 'Ptprc', 'Lyz2', 'Lst1')
    MT_PATTERN = r'^mt[-.]'
    TECH_PATTERN = r'^(?:ERCC[-_].*|Egfp|eGFP|GFP|Gfp|mCherry|tdTomato|Cre|CreERT2|Luc2|luciferase)$'
    NODES, LAMBDA, SIGMA, STEPS, SEED = 40, 3., .095, 1000, 1
    GRID = [(n, NODES, SEED) for n in HVG_COUNTS] + [(BASELINE_HVG, 20, SEED),
        (BASELINE_HVG, 60, SEED), (BASELINE_HVG, NODES, 42)]
    hashes = {str(p.relative_to(ROOT)): digest(p) for p in [INPUT,
        # Set this storage path independently before running. tree_results.pkl.
        ROOT, ROOT,
        # Set this storage path independently before running. VGN-lines.png.
        ROOT]}
    with h5py.File(INPUT, 'r') as h:
        all_ids = h['cell_ids'].asstr()[:]
        obs = readframe(h['obs'])
    scope = obs.type.isin(['CVG', 'VGN']).to_numpy()
    source_rows = np.flatnonzero(scope)
    ids = all_ids[scope]
    obs = obs.loc[scope].copy()
    obs.index = pd.Index(ids, name='cell_id_index')
    obs['cell_id'] = ids
    assert len(obs) == 1010 and obs.type.value_counts().to_dict() == {'VGN': 520, 'CVG': 490}
    import scanpy as sc
    import anndata as ad
    sc.settings.verbosity = 0
    sc.settings.n_jobs = 1
    with h5py.File(INPUT, 'r') as h:
        genes = h['genes'].asstr()[:]
        count_group = h['counts']
        counts = sparse.csr_matrix((count_group['data'][:], count_group['indices'][:], count_group['indptr'][:]),
            shape=tuple(count_group['shape'][:]))[scope]
        norm_log = h['norm_log'][source_rows, :]
    detected = np.asarray((counts > 0).sum(axis=0)).ravel()
    excluded = np.isin(genes, ANCHORS + BLOOD_GENES)
    excluded |= np.array([bool(re.search(MT_PATTERN, g, re.I) or re.fullmatch(TECH_PATTERN, g, re.I)) for g in genes])
    eligible = (detected >= MIN_HVG_CELLS) & ~excluded
    assert eligible.sum() >= max(HVG_COUNTS)
    base = ad.AnnData(norm_log[:, eligible].copy(), obs=obs.copy(),
        var=pd.DataFrame(index=pd.Index(genes[eligible], name='gene')))
    del norm_log, counts
    representations, feature_sets, pca_variance, pca_loadings = {}, {}, {}, {}
    for n in HVG_COUNTS:
        a = base.copy()
        sc.pp.highly_variable_genes(a, flavor='seurat', n_top_genes=n, subset=True)
        feature_sets[n] = a.var_names.to_numpy()
        assert not set(feature_sets[n]).intersection(ANCHORS + BLOOD_GENES)
        assert not any(re.search(MT_PATTERN, g, re.I) or re.fullmatch(TECH_PATTERN, g, re.I) for g in feature_sets[n])
        sc.pp.scale(a, zero_center=True, max_value=10)
        sc.tl.pca(a, n_comps=N_PCS, svd_solver='arpack', random_state=PCA_SEED)
        representations[n] = a.obsm['X_pca'].astype(float)
        pca_variance[n] = a.uns['pca']['variance_ratio'].copy()
        pca_loadings[n] = a.varm['PCs'].copy()
        assert representations[n].shape == (1010, 30)
        print('Prepared scope-only HVG/PCA:', n, len(feature_sets[n]), flush=True)
    del base, a
    results = {}
    for hvg, nodes, seed in GRID:
        key = f'hvg{hvg}_N{nodes}_seed{seed}'
        pcs = representations[hvg]
        center = pcs.mean(axis=0)
        radius = float(np.sqrt(np.mean(np.sum((pcs - center) ** 2, axis=1))))
        x = (pcs - center) / radius
        print('Fit', key, flush=True)
        model = fit_tree(x, nodes, LAMBDA, SIGMA, STEPS, seed)
        result = project_and_select(model, x, obs)
        result.update(PCs=pcs, center=center, global_rms=radius,
            raw_F=model['raw_F'], raw_B=model['raw_B'], hvg_genes=feature_sets[hvg],
            pca_variance_ratio=pca_variance[hvg], pca_loadings=pca_loadings[hvg])
        result['QA'].update(hvg=hvg, requested_nodes=nodes, seed=seed)
        results[key] = result
        print(json.dumps(result['QA'], allow_nan=False), flush=True)
    selected_key = f'hvg{BASELINE_HVG}_N{NODES}_seed{SEED}'
    selected = results[selected_key]
    comparisons = []
    for key, result in results.items():
        a, b = selected['primary_path'], result['primary_path']
        common = a & b
        comparisons.append({'branch': key, 'rho_scope': correlation(selected['t_native'], result['t_native']),
            'rho_common_primary': correlation(selected['t_native'][common], result['t_native'][common]) if common.sum() >= 5 else None,
            'primary_membership_jaccard': float((a & b).sum() / (a | b).sum()),
            'primary_cells': int(b.sum()), 'natural_tips': len(result['QA']['natural_tips']),
            'numerically_settled': result['QA']['numerically_settled']})
    obs['seg'] = selected['seg']
    obs['t_native'] = selected['t_native']
    obs['t'] = selected['t']
    obs['t_display'] = selected['t_display']
    methods = {'algorithm': 'Official simpleppt1.1.7 unrestricted Euclidean principal tree; cold random-cell initialization',
        # Set this storage path independently before running. ppt.py.
        'implementation': str(simpleppt.__file__), 'principal_tree_module_sha256': digest(ROOT),
        'representation': 'HVG selection, scaling, PCA30 and principal-tree fitting are estimated on only1010 real CVG/VGN cells; no SGN cell enters geometry estimation',
        'normalization': 'Center PCA coordinates across the1010 fitted cells and divide by one global RMS radius, as in the original principal-tree method',
        'upstream_PCA': 'Gene-wise scale and clip at10; arpack PCA30', 'PCA_seed': PCA_SEED,
        'upstream_HVG': 'Seurat HVG on existing log1p(CP10k); min3 detected fitted cells, as in the original local embedding function',
        'HVG_candidate_genes': int(eligible.sum()), 'minimum_HVG_detected_cells': MIN_HVG_CELLS,
        'blood_HVG_exclusions': BLOOD_GENES, 'mt_exclusion_pattern': MT_PATTERN,
        'technical_exclusion_pattern': TECH_PATTERN,
        'excluded_anchors': ANCHORS, 'HVG_COUNTS': HVG_COUNTS,
        'selected_key': selected_key, 'selection': 'Prespecified middle HVG3000/N40/seed1; no candidate RNA or program curves used',
        'N40_scope': 'Adapted cell-count-scale node count within the original20/40/60 local sensitivity grid',
        'lambda': LAMBDA, 'sigma': SIGMA, 'steps': STEPS, 'error_cut': -1.,
        'root_rule': 'Existing principal node nearest the12 E9.5 PCA centroid; degree2 internal nodes are allowed; geometry is fixed before metadata orientation',
        'root_reason': 'The original tip-only age rule can exclude the earliest sampled cells when their nearest node is degree2; metadata-grounded rooting avoids changing HVG or geometry for a desired plot',
        'endpoint_rule': 'Among nonroot natural tips, maximize terminal-branch F1 for real E13.5 VGN labels, then count, purity, and lower node index; seven anchor genes are excluded from both geometry and endpoint rule',
        'endpoint_limit': 'Post-fit metadata endpoint identification is descriptive and does not establish prospective lineage',
        'projection': 'Unrestricted nearest straight principal-tree edge; clipped alpha; t_native=root_distance(edge_start)+alpha*edge_length',
        'primary_path': 'Cells whose assigned edges lie on the natural graph path from root to selected VGN endpoint; all other fitted cells remain scope background',
        'segments': 'Natural maximal root/fork/tip graph chains named s0,s1,...; original type retained separately',
        'display_time': 't and t_native are native arc distances; t_display=6*t_native/max(t_native) over1010 scope is a separate linear display rescaling',
        'fork_line': 'First natural degree>=3 graph node on the selected root-to-VGN-endpoint route; native root distance, not an established fate-decision threshold',
        'UMAP': 'No UMAP calculated or altered; display mapping is handled separately using the original accepted atlas',
        'limits': ['Existing curated types are not lineage tracing.', 'Principal-tree edges do not add sampled intermediates.',
            'Only12 E9.5 cells and three ages are present; age, depth, library and proliferation can influence geometry.',
            'The previous DPT values, UMAP and1681-cell PCA/HVG were not used for this fit.',
            'One continuous GAM may interpolate through unsampled native-t intervals; this interpolation is not an observed developmental intermediate.']}
    assert {p: digest(ROOT / p) for p in hashes} == hashes
    with h5py.File(RESULT, 'w') as h:
        h.attrs.update(schema='actual_CVG_VGN_SimplePPT_v1', original_inputs_unchanged=True)
        put(h, 'cell_ids', ids)
        put(h, 'source_rows', source_rows)
        put(h, 'scope_CVG_VGN', np.ones(len(obs), bool))
        put(h, 'good', np.ones(len(obs), bool))
        put(h, 'selected_key', np.asarray(selected_key))
        put(h, 'selected_hvg', BASELINE_HVG)
        for field in ['PCs', 'F', 'B', 'parent', 'root_distance', 'edges', 'edge_id', 'alpha', 'residual',
            'edge_length', 'edge_cell_counts', 'nearest_node', 't_native', 't', 't_display', 'fork_t', 'primary_path', 'root', 'endpoint', 'seg']:
            put(h, field, selected[field])
        put(h, 'graph_segments_json', selected['segments'])
        put(h, 'tip_audit_json', selected['tip_audit'])
        put(h, 'path_QA_json', selected['QA'])
        put(h, 'methods_json', methods)
        put(h, 'input_sha256_json', hashes)
        writeframe(h, 'obs', obs)
        writeframe(h, 'sensitivity', pd.DataFrame(comparisons))
        for key, result in results.items():
            node = h.create_group(key)
            for field in ['PCs', 'F', 'B', 'raw_F', 'raw_B', 'center', 'global_rms', 'parent', 'root_distance',
                'edges', 'edge_id', 'alpha', 'residual', 'edge_length', 'edge_cell_counts', 'nearest_node',
                't_native', 't', 't_display', 'fork_t', 'primary_path', 'root', 'endpoint', 'seg', 'hvg_genes', 'pca_variance_ratio', 'pca_loadings']:
                put(node, field, result[field])
            put(node, 'graph_segments_json', result['segments'])
            put(node, 'tip_audit_json', result['tip_audit'])
            put(node, 'QA_json', result['QA'])
        put(h, 'executed_source', Path(__file__).read_text(encoding='utf-8'))
    print('Ready:', str(RESULT), flush=True)
    print(pd.DataFrame(comparisons).to_string(index=False), flush=True)


def reroot_saved():
    """Orient saved graphs without re-estimating HVG, PCA, F or B."""
    with h5py.File(RESULT, 'r') as h:
        obs = readframe(h['obs'])
        methods = json.loads(h['methods_json'].asstr()[()])
        selected_key = h['selected_key'].asstr()[()]
        hashes = json.loads(h['input_sha256_json'].asstr()[()])
        branches = [k for k in h if k.startswith('hvg')]
        results, graph_audit = {}, []
        for key in branches:
            g = h[key]
            old = json.loads(g['QA_json'].asstr()[()])
            model = {'F': g['F'][:], 'B': g['B'][:], 'numerically_settled': old['numerically_settled'],
                'fixed_point_residual': old['fixed_point_residual'],
                'zero_length_nodes_contracted': old['zero_length_nodes_contracted'], 'seconds': old['seconds']}
            x = (g['PCs'][:] - g['center'][:]) / g['global_rms'][()]
            result = project_and_select(model, x, obs)
            result['QA'].update(hvg=old['hvg'], requested_nodes=old['requested_nodes'], seed=old['seed'])
            results[key] = result
            graph_audit.append({'branch': key, 'old_root': old['root'], 'new_root': result['root'],
                'F_sha256': array_digest(model['F']), 'B_sha256': array_digest(model['B']),
                'graph_refitted': False})
    selected = results[selected_key]
    obs['seg'] = selected['seg']
    obs['t_native'] = selected['t_native']
    obs['t'] = selected['t_native']
    obs['t_display'] = selected['t_display']
    comparisons = []
    for key, result in results.items():
        a, b = selected['primary_path'], result['primary_path']
        common = a & b
        comparisons.append({'branch': key, 'rho_scope': correlation(selected['t_native'], result['t_native']),
            'rho_common_primary': correlation(selected['t_native'][common], result['t_native'][common]) if common.sum() >= 5 else None,
            'primary_membership_jaccard': float((a & b).sum() / (a | b).sum()),
            'primary_cells': int(b.sum()), 'natural_tips': len(result['QA']['natural_tips']),
            'numerically_settled': result['QA']['numerically_settled']})
    methods.update(root_rule='Existing principal node nearest the12 E9.5 PCA centroid; degree2 internal nodes are allowed; geometry is fixed before metadata orientation',
        root_reason='The original tip-only age rule can exclude the earliest sampled cells when their nearest node is degree2; metadata-grounded rooting avoids changing HVG or geometry for a desired plot',
        display_time='t and t_native are native arc distances; t_display=6*t_native/max(t_native) over1010 scope is a separate linear display rescaling',
        fork_line='First natural degree>=3 graph node on the selected root-to-VGN-endpoint route; native root distance, not an established fate-decision threshold',
        metadata_rooting_refit=False)
    fields = ['parent', 'root_distance', 'edges', 'edge_id', 'alpha', 'residual', 'edge_length',
        'edge_cell_counts', 'nearest_node', 't_native', 't', 't_display', 'fork_t', 'primary_path', 'root', 'endpoint', 'seg']
    assert {p: digest(ROOT / p) for p in hashes} == hashes
    with h5py.File(RESULT, 'r+') as h:
        for key, result in results.items():
            node = h[key]
            for field in fields:
                if field in node:
                    del node[field]
                put(node, field, result[field])
            for field, value in [('graph_segments_json', result['segments']), ('tip_audit_json', result['tip_audit']), ('QA_json', result['QA'])]:
                del node[field]
                put(node, field, value)
        for row in graph_audit:
            node = h[row['branch']]
            assert array_digest(node['F'][:]) == row['F_sha256']
            assert array_digest(node['B'][:]) == row['B_sha256']
        for field in fields:
            if field in h:
                del h[field]
            put(h, field, selected[field])
        for field, value in [('graph_segments_json', selected['segments']), ('tip_audit_json', selected['tip_audit']),
            ('path_QA_json', selected['QA']), ('methods_json', methods), ('metadata_rooting_audit_json', graph_audit),
            ('executed_source', Path(__file__).read_text(encoding='utf-8'))]:
            if field in h:
                del h[field]
            put(h, field, value)
        del h['obs'], h['sensitivity']
        writeframe(h, 'obs', obs)
        writeframe(h, 'sensitivity', pd.DataFrame(comparisons))
    print('Reoriented saved principal graphs; all F/B hashes unchanged.', flush=True)
    print(json.dumps(selected['QA'], allow_nan=False), flush=True)
    print(pd.DataFrame(comparisons).to_string(index=False), flush=True)


if __name__ == '__main__':
    if '--reroot' in sys.argv:
        reroot_saved()
    else:
        main()


## Original standalone source: fig4_VGN_network_analysis.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# VGN GRNBoost2/motif-supported network and AUCell worker producing Network.h5.
# Original source label: Network.py (fig4_VGN_network_analysis)
# Original source SHA256: d727c68b344d7f7145cc127046162ab8624e186ade56eedc38333c56029acdd3
# Only storage addresses and release comments were changed; analysis was not executed.

"""Actual CVG/VGN SCENIC network; full TF competition and explicit shared-axis scoring."""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

#1. =========== Shared inputs and reproducible settings
from pathlib import Path
import argparse
import hashlib
import importlib.metadata
import json
import os
import pickle
import re
import sys
import time
import warnings
# Set this storage path independently before running.
ROOT = ROOT
OUT = Path(__file__).resolve().parent
# Set this storage path independently before running.
WORK = ROOT
WORK.mkdir(exist_ok=True)
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ[key] = '1'
os.environ['TMP'] = os.environ['TEMP'] = str(WORK)
# Set this storage path independently before running.
os.environ['NUMBA_CACHE_DIR'] = str(ROOT)
# Set this storage path independently before running.
os.environ['MPLCONFIGDIR'] = str(ROOT)
# Set this storage path independently before running.
os.environ['PYTHONPYCACHEPREFIX'] = str(ROOT)
sys.dont_write_bytecode = True
warnings.filterwarnings('ignore', message='Not prepending group keys', category=FutureWarning, module='pyscenic.utils')
import dask
dask.config.set({'dataframe.query-planning': False})
import h5py
import numpy as np
import pandas as pd
from scipy import sparse
# Set this storage path independently before running. Early-data.h5. Produced by fig4_VGN_input_analysis.py.
INPUT = ROOT
# Set this storage path independently before running. expression.h5.
COMMON = ROOT
# Set this storage path independently before running.
RESOURCE = ROOT
# Set this storage path independently before running. Inputs.h5. Candidate input contract; supply the immutable input container (standalone producer not located).
PREPARED = ROOT
# Set this storage path independently before running. Network.h5. Produced by fig4_VGN_network_analysis.py.
RESULT = ROOT
ANCHORS = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
TF_CANDIDATES = ANCHORS[:-1]
TF_SUPPLEMENT = ('Npas3',)
SEED = 42
N_THREADS = 8
DATABASE_NAMES = ('mm10_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
                  'mm10_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather')
# Set this storage path independently before running.
ANNOTATIONS = ROOT


def digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def json_dataset(handle, name, value):
    handle.create_dataset(name, data=json.dumps(value, ensure_ascii=False, allow_nan=False), dtype=h5py.string_dtype())


def string_dataset(handle, name, value):
    handle.create_dataset(name, data=np.asarray(value, dtype=object), dtype=h5py.string_dtype())


def readframe(node):
    cols = json.loads(node.attrs['columns']) if 'columns' in node.attrs else list(node)
    return pd.DataFrame({k: node[k].asstr()[:] if node[k].dtype.kind in 'OSU' else node[k][:]
                         for k in cols if k != '_index'})


def writeframe(handle, name, frame):
    node = handle.create_group(name)
    node.attrs['columns'] = json.dumps(frame.columns.tolist())
    for key in frame:
        a = frame[key].to_numpy()
        if a.dtype.kind in 'OUS':
            string_dataset(node, key, frame[key].fillna('').astype(str).to_numpy())
        else:
            node.create_dataset(key, data=a, compression='gzip', shuffle=True)


def versions():
    return {k: importlib.metadata.version(k) for k in
            ('pyscenic', 'arboreto', 'ctxcore', 'numpy', 'pandas', 'scikit-learn', 'dask', 'joblib')}


#2. =========== Frozen actual CVG/VGN source and complete expressed inference axis

def prepare():
    MIN_DETECTED_CELLS = 3
    source_hashes = {str(p.relative_to(ROOT)): digest(p) for p in (INPUT, COMMON)}
    resource_hashes = {p.name: digest(p) for p in RESOURCE.iterdir() if p.is_file()}
    if PREPARED.exists():
        with h5py.File(PREPARED, 'r') as h:
            assert json.loads(h['source_sha256_json'].asstr()[()]) == source_hashes
            assert json.loads(h['resource_sha256_json'].asstr()[()]) == resource_hashes
            assert json.loads(h['methods_json'].asstr()[()])['normalization'] == 'log10(1+native raw count/full endogenous total*1e6); float64; paper Alignment expression unit'
            return h.attrs['signature']
    with h5py.File(INPUT, 'r') as h:
        obs = readframe(h['obs'])
        native_genes = h['genes'].asstr()[:]
        all_cells = h['cell_ids'].asstr()[:]
        counts = sparse.csr_matrix((h['counts/data'][:], h['counts/indices'][:], h['counts/indptr'][:]),
                                  shape=tuple(h['counts/shape'][:]))
    np.testing.assert_array_equal(all_cells, obs.cell_id)
    assert obs.type.value_counts().to_dict() == {'SGN': 671, 'VGN': 520, 'CVG': 490}
    assert obs.study_id.eq('GSE178931').all() and obs.stage_standardized.isin(['E9.5', 'E11.5', 'E13.5']).all()
    rows = np.flatnonzero(obs.type.isin(['CVG', 'VGN']))
    primary = obs.iloc[rows].reset_index(drop=True)
    assert primary.type.value_counts().to_dict() == {'VGN': 520, 'CVG': 490}
    assert primary.cell_id.is_unique and pd.Index(native_genes).is_unique
    totals = primary.full_total.to_numpy()
    assert (totals > 0).all() and (counts.data >= 0).all()
    # Original SCENIC source feature eligibility; full-endogenous denominators remain unchanged.
    excluded = np.array([bool(re.search(r'(?i)^mt-|^ERCC[-_]|^Hb[ab]-', g)) for g in native_genes])
    eligible = ~excluded & (native_genes != '')
    detection = np.asarray((counts[rows] > 0).sum(axis=0)).ravel()
    keep = eligible & (detection >= MIN_DETECTED_CELLS)
    genes = native_genes[keep]
    x = np.log10(1 + counts[rows][:, keep].toarray() / totals[:, None] * 1e6)
    # Set this storage path independently before running. allTFs_mm.txt.
    original_tfs = set((ROOT).read_text().splitlines())
    tf_set = original_tfs | set(TF_SUPPLEMENT)
    tf_genes = np.array([g for g in genes if g in tf_set])
    assert all(tf in tf_genes for tf in TF_CANDIDATES) and 'Pcdh19' not in tf_genes
    annotation = pd.read_csv(ANNOTATIONS, sep='\t', usecols=['#motif_id', 'gene_name', 'motif_similarity_qvalue',
                            'orthologous_identity', 'orthologous_gene_name', 'description'])
    supplement_evidence = annotation[annotation.gene_name.isin(TF_SUPPLEMENT)].to_dict('records')
    assert all(tf in set(annotation.gene_name) for tf in TF_SUPPLEMENT)
    with h5py.File(COMMON, 'r') as h:
        common_genes = h['gene_symbols'].asstr()[:]
        atlas_cells = h['cell_ids'].asstr()[:]
    assert len(common_genes) == 11467 and set(common_genes).issubset(set(native_genes))
    assert not set(primary.cell_id[primary.type.eq('VGN')]) & set(atlas_cells)
    signature_record = {'source_sha256': source_hashes, 'resource_sha256': resource_hashes,
        'cell_ids': primary.cell_id.tolist(), 'inference_genes': genes.tolist(), 'TFs': tf_genes.tolist(),
        'seed': SEED, 'minimum_detected_cells': MIN_DETECTED_CELLS,
        'normalization': 'log10(1+native raw count/full endogenous total*1e6); float64; paper Alignment expression unit',
        'TF_supplement': TF_SUPPLEMENT}
    signature = hashlib.sha256(json.dumps(signature_record, sort_keys=True).encode()).hexdigest()
    methods = {'primary_cells': 1010, 'types': {'CVG': 490, 'VGN': 520}, 'SGN_excluded_from_inference': 671,
        'native_measured_genes': len(native_genes), 'eligible_genes': int(eligible.sum()),
        'inference_genes': len(genes), 'resource_inference_TFs': sum(g in original_tfs for g in tf_genes),
        'inference_TFs': len(tf_genes), 'TF_supplement': list(TF_SUPPLEMENT),
        'TF_supplement_reason': 'Prespecified TF Npas3 is absent from the TF list but present in the supplied motif annotation; no competitor removed',
        'TF_supplement_motif_evidence': supplement_evidence,
        'minimum_detected_cells': MIN_DETECTED_CELLS, 'normalization': signature_record['normalization'],
        'full_endogenous_totals_before_feature_exclusion': True,
        'no_HVG_or_common_axis_restriction_for_GRN': True, 'shared_scoring_genes': len(common_genes),
        'missing_source_genes_imputed': False, 'seed': SEED, 'runtime_versions': versions()}
    with h5py.File(PREPARED, 'w') as h:
        h.attrs['signature'] = signature
        h.create_dataset('expression', data=np.ascontiguousarray(x), compression='gzip', shuffle=True)
        h.create_dataset('full_endogenous_total', data=totals)
        h.create_dataset('input_primary_rows', data=rows)
        h.create_dataset('native_detection', data=detection)
        string_dataset(h, 'genes', genes); string_dataset(h, 'TFs', tf_genes)
        string_dataset(h, 'cell_ids', primary.cell_id); string_dataset(h, 'eligible_genes', native_genes[eligible])
        string_dataset(h, 'native_genes', native_genes); string_dataset(h, 'common_genes', common_genes)
        writeframe(h, 'obs', primary)
        for name, value in [('methods_json', methods), ('source_sha256_json', source_hashes),
                            ('resource_sha256_json', resource_hashes), ('signature_record_json', signature_record)]:
            json_dataset(h, name, value)
    print('Prepared actual CVG/VGN:', json.dumps(methods), flush=True)
    return signature


#3. =========== Full-competition standard GRNBoost2 with resumable target checkpoints

def infer_target(index, expression, genes, tf_expression, tf_genes, seed):
    import dask
    dask.config.set({'dataframe.query-planning': False})
    from arboreto.core import EARLY_STOP_WINDOW_LENGTH, SGBM_KWARGS, infer_partial_network
    table = infer_partial_network(regressor_type='GBM', regressor_kwargs=SGBM_KWARGS,
        tf_matrix=tf_expression, tf_matrix_gene_names=tf_genes,
        target_gene_name=genes[index], target_gene_expression=expression[:, index],
        include_meta=False, early_stop_window_length=EARLY_STOP_WINDOW_LENGTH, seed=seed)
    return index, table


def grn(pilot=0):
    from joblib import Parallel, delayed
    CHECKPOINT_TARGETS = 250
    signature = prepare()
    # Set this storage path independently before running.
    folder = ROOT; folder.mkdir(exist_ok=True)
    with h5py.File(PREPARED, 'r') as h:
        x = np.ascontiguousarray(h['expression'][:], dtype=np.float64)
        genes = h['genes'].asstr()[:].tolist(); tf_genes = h['TFs'].asstr()[:].tolist()
    tf_expression = np.ascontiguousarray(x[:, pd.Index(genes).get_indexer(tf_genes)])
    targets = list(range(len(genes)))
    if pilot:
        targets = np.random.default_rng(SEED).permutation(targets)[:pilot].tolist()
    completed = set(); frames = []
    for p in sorted(folder.glob('batch_*.pkl')):
        with p.open('rb') as stream: saved = pickle.load(stream)
        assert saved['signature'] == signature and not completed.intersection(saved['targets'])
        completed.update(saved['targets']); frames.append(saved['adjacency'])
    if pilot:
        targets = [t for t in targets if t not in completed]
    else:
        targets = [t for t in targets if t not in completed]
    start = time.monotonic()
    print(f'GRNBoost2 PID={os.getpid()} normalization=log10(1+CPM); full competitors: 1010 cells, {len(genes)} targets, {len(tf_genes)} TFs, {N_THREADS} workers; pending={len(targets)}; pilot={pilot}', flush=True)
    pending_frames, pending_targets = [], []
    with Parallel(n_jobs=N_THREADS, return_as='generator_unordered', max_nbytes='1M',
                  # Set this storage path independently before running.
                  temp_folder=ROOT, batch_size=1) as executor:
        generated = executor(delayed(infer_target)(j, x, genes, tf_expression, tf_genes, SEED) for j in targets)
        for n, (j, table) in enumerate(generated, 1):
            pending_frames.append(table); pending_targets.append(j)
            if n % CHECKPOINT_TARGETS == 0 or n == len(targets):
                block = pd.concat(pending_frames, ignore_index=True)
                path = folder / f'batch_{len(completed):05d}.pkl'
                with path.with_suffix('.part').open('wb') as stream:
                    pickle.dump({'signature': signature, 'targets': pending_targets, 'adjacency': block}, stream, protocol=pickle.HIGHEST_PROTOCOL)
                path.with_suffix('.part').replace(path)
                completed.update(pending_targets); frames.append(block)
                pending_frames, pending_targets = [], []
                print(f'GRN targets complete={len(completed)}/{len(genes)}; new={n}/{len(targets)}; elapsed={time.monotonic()-start:.1f}s', flush=True)
    elapsed = time.monotonic() - start
    if pilot:
        print(json.dumps({'pilot_new_targets': len(targets), 'elapsed_seconds': elapsed,
            'projected_GRN_seconds_excluding_startup': elapsed / max(len(targets), 1) * len(genes),
            'full_competitors': len(tf_genes), 'full_result': False}), flush=True)
        return
    assert completed == set(range(len(genes)))
    adjacency = pd.concat(frames, ignore_index=True).sort_values('importance', ascending=False)
    assert not adjacency.duplicated(['TF', 'target']).any()
    assert np.isfinite(adjacency.importance).all() and (adjacency.importance > 0).all()
    assert set(adjacency.TF).issubset(tf_genes) and set(adjacency.target).issubset(genes)
    # Set this storage path independently before running. Adjacency.pkl.
    adjacency.to_pickle(ROOT)
    metadata = {'method': 'GRNBoost2', 'cells': 1010, 'genes': len(genes), 'TFs': len(tf_genes),
        'targets_fitted': len(completed), 'edges': len(adjacency), 'seed': SEED, 'workers': N_THREADS,
        'elapsed_this_run_seconds': elapsed, 'pilot_only': False, 'signature': signature,
        'all_TFs_competed_for_every_target': True, 'runtime_versions': versions()}
    # Set this storage path independently before running. GRN.json.
    (ROOT).write_text(json.dumps(metadata), encoding='utf-8')
    with h5py.File(PREPARED, 'r') as inp, h5py.File(RESULT, 'w') as h:
        h.attrs['schema'] = 'actual_CVG_VGN_SCENIC_v1'; h.attrs['stage'] = 'grn_complete'
        for name in ['genes', 'TFs', 'cell_ids', 'eligible_genes', 'native_genes', 'common_genes',
                     'full_endogenous_total', 'input_primary_rows', 'obs', 'methods_json', 'source_sha256_json', 'resource_sha256_json']:
            inp.copy(name, h)
        string_dataset(h, 'anchors', ANCHORS)
        json_dataset(h, 'grn_methods_json', metadata)
        h.create_dataset('grn/TF_index', data=pd.Index(tf_genes).get_indexer(adjacency.TF).astype(np.int32), compression='gzip', shuffle=True)
        h.create_dataset('grn/target_index', data=pd.Index(genes).get_indexer(adjacency.target).astype(np.int32), compression='gzip', shuffle=True)
        h.create_dataset('grn/importance', data=adjacency.importance.to_numpy(), compression='gzip', shuffle=True)
        h.create_dataset('grn/targets_fitted_mask', data=np.ones(len(genes), dtype=bool), compression='gzip')
    print('GRN COMPLETE:', json.dumps(metadata), flush=True)


#4. =========== Full-adjacency modules and candidate-only cisTarget motif pruning

def plain(value):
    if isinstance(value, np.generic): return plain(value.item())
    if isinstance(value, float): return value if np.isfinite(value) else None
    if isinstance(value, dict): return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (set, frozenset)): return [plain(v) for v in sorted(value, key=str)]
    if isinstance(value, (tuple, list, np.ndarray)): return [plain(v) for v in value]
    return value


def prune():
    from ctxcore.rnkdb import FeatherRankingDatabase
    from pyscenic.utils import modules_from_adjacencies
    from pyscenic.prune import prune2df, df2regulons
    MIN_MODULE_GENES = 20
    NES_THRESHOLD = 3.0
    MOTIF_AUC_THRESHOLD = 0.05
    RANK_THRESHOLD = 5000
    RHO_THRESHOLD = 0.03
    PRUNE_WORKERS = 4
    with h5py.File(RESULT, 'r') as h:
        meta = json.loads(h['grn_methods_json'].asstr()[()])
        signature = meta['signature']
        genes = h['genes'].asstr()[:]
        cells = h['cell_ids'].asstr()[:]
        resource_hashes = json.loads(h['resource_sha256_json'].asstr()[()])
        expression = pd.DataFrame(h['inference_expression'][:], index=cells, columns=genes) if 'inference_expression' in h else None
        # Set this storage path independently before running. Adjacency.pkl.
        if (ROOT).exists():
            # Set this storage path independently before running. Adjacency.pkl.
            adjacency = pd.read_pickle(ROOT)
        else:
            full_tfs = h['TFs'].asstr()[:]
            adjacency = pd.DataFrame({'TF': full_tfs[h['grn/TF_index'][:]],
                'target': genes[h['grn/target_index'][:]], 'importance': h['grn/importance'][:]})
    if expression is None:
        with h5py.File(PREPARED, 'r') as h:
            assert h.attrs['signature'] == signature
            np.testing.assert_array_equal(h['genes'].asstr()[:], genes)
            np.testing.assert_array_equal(h['cell_ids'].asstr()[:], cells)
            expression = pd.DataFrame(h['expression'][:], index=cells, columns=genes)
    assert not meta['pilot_only'] and meta['targets_fitted'] == len(genes)
    assert not adjacency.duplicated(['TF', 'target']).any() and len(adjacency) == meta['edges']
    # Set this storage path independently before running. Candidate_modules.pkl.
    module_cache = ROOT
    reuse_module_cache = False
    if module_cache.exists():
        with module_cache.open('rb') as stream: saved = pickle.load(stream)
        assert saved['signature'] == signature
        reuse_module_cache = saved.get('TF_candidates') == list(TF_CANDIDATES)
    if reuse_module_cache:
        modules, module_metadata = saved['modules'], saved['metadata']
    else:
        started = time.monotonic()
        # Crucial: the entire adjacency enters quantiles and target-wise top regulators.
        # Filter the six predeclared anchors only after modules for all competing TFs exist.
        all_modules = modules_from_adjacencies(adjacency, expression, min_genes=MIN_MODULE_GENES,
            thresholds=(0.75, 0.90), top_n_targets=(50,), top_n_regulators=(5, 10, 50),
            absolute_thresholds=False, rho_dichotomize=True, keep_only_activating=True,
            rho_threshold=RHO_THRESHOLD, rho_mask_dropouts=False)
        modules = [m for m in all_modules if m.transcription_factor in TF_CANDIDATES]
        module_metadata = {'full_adjacency_edges': len(adjacency), 'all_TF_modules': len(all_modules),
            'candidate_modules': len(modules), 'candidate_counts': {tf: sum(m.transcription_factor == tf for m in modules) for tf in TF_CANDIDATES},
            'all_adjacency_before_TF_filter': True, 'absolute_thresholds': False, 'module_creation_seconds': time.monotonic() - started,
            'thresholds': [0.75, 0.90], 'top_n_targets': [50], 'top_n_regulators': [5, 10, 50]}
        with module_cache.open('wb') as stream:
            pickle.dump({'signature': signature, 'TF_candidates': list(TF_CANDIDATES), 'modules': modules, 'metadata': module_metadata}, stream, protocol=pickle.HIGHEST_PROTOCOL)
        del all_modules
    print('cisTarget candidate modules:', json.dumps(module_metadata), flush=True)
    assert all(digest(RESOURCE / name) == value for name, value in resource_hashes.items())
    databases = [FeatherRankingDatabase(str(RESOURCE / name), name=name) for name in DATABASE_NAMES]
    coverage = [{'database': db.name, 'genes': db.total_genes,
                 'inference_gene_coverage': len(set(genes) & set(db.genes)) / len(genes)} for db in databases]
    started = time.monotonic()
    motif_table = prune2df(databases, modules, str(ANNOTATIONS), rank_threshold=RANK_THRESHOLD,
        auc_threshold=MOTIF_AUC_THRESHOLD, nes_threshold=NES_THRESHOLD,
        client_or_address='dask_multiprocessing', num_workers=PRUNE_WORKERS) if modules else pd.DataFrame()
    regulons = df2regulons(motif_table, save_columns=['NES', 'Annotation', 'MotifSimilarityQvalue', 'OrthologousIdentity']) if not motif_table.empty else []
    assert len({r.name for r in regulons}) == len(regulons)
    assert all(r.transcription_factor in TF_CANDIDATES and set(r.genes).issubset(set(genes)) for r in regulons)
    # Set this storage path independently before running. Motif_support.pkl.
    motif_table.to_pickle(ROOT)
    # Set this storage path independently before running. Regulons.pkl.
    with (ROOT).open('wb') as stream:
        pickle.dump(regulons, stream, protocol=pickle.HIGHEST_PROTOCOL)
    rows = []
    for index, row in motif_table.iterrows():
        record = {'TF': str(index[0]), 'MotifID': str(index[1])}
        record.update({'|'.join(map(str, key)) if isinstance(key, tuple) else str(key): plain(value) for key, value in row.items()})
        rows.append(record)
    definitions = {}
    by_tf = {r.transcription_factor: r for r in regulons}
    for anchor in ANCHORS:
        r = by_tf.get(anchor)
        d = {'anchor': anchor, 'anchor_is_TF': anchor != 'Pcdh19', 'native_anchor_measured': True,
             'regulon_name': None, 'gene2weight': {}, 'motif_supported': False,
             'status': 'not_TF_no_regulon' if anchor == 'Pcdh19' else 'no_motif_supported_regulon',
             'direct_binding_or_causality_claim_allowed': False}
        if r is not None:
            d.update({'regulon_name': r.name, 'gene2weight': plain(dict(r.gene2weight)),
                'motif_supported': True, 'status': 'motif_supported_regulon',
                'context': plain(r.context), 'NES': plain(r.nes), 'annotation': plain(r.annotation),
                'orthologous_identity': plain(r.orthologous_identity), 'similarity_qvalue': plain(r.similarity_qvalue),
                'native_targets': len(r), 'native_target_coverage': 1.0,
                'own_TF_in_regulon': anchor in r.genes,
                'gene_membership_sha256': hashlib.sha256(json.dumps(sorted(r.genes)).encode()).hexdigest(),
                'full_regulon_genes_preserved': True})
        definitions[anchor] = d
    methods = {**module_metadata, 'min_module_genes_before_motif_pruning': MIN_MODULE_GENES,
        'NES_threshold': NES_THRESHOLD, 'motif_auc_threshold': MOTIF_AUC_THRESHOLD,
        'rank_threshold': RANK_THRESHOLD,
        'rank_threshold_basis': 'Author pySCENIC0.11.2 CLI default5000, explicitly forwarded to prune2df; API default1500 is not used',
        'paper_CLI_source': 'https://raw.githubusercontent.com/aertslab/pySCENIC/0.11.2/src/pyscenic/cli/pyscenic.py',
        'Pearson_rho_threshold': RHO_THRESHOLD,
        'dropout_masking': False, 'activating_only': True, 'databases': coverage,
        'regulons': len(regulons), 'motif_rows': len(rows), 'pruning_seconds': time.monotonic() - started,
        'pruned_TF_family': list(TF_CANDIDATES), 'Pcdh19_is_not_TF': True,
        'resource_sha256': resource_hashes,
        'no_final_regulon_20_gene_gate': True,
        'limits': ['Motif-pruned coexpression regulons are inferred candidate targets, not demonstrated binding or causal regulation',
                   'Only predeclared TF modules are motif-pruned, after complete full-TF module competition']}
    with h5py.File(RESULT, 'a') as h:
        if 'inference_expression' not in h:
            h.create_dataset('inference_expression', data=expression.to_numpy(), compression='gzip', shuffle=True)
            h.attrs['inference_expression_sha256'] = hashlib.sha256(np.ascontiguousarray(expression.to_numpy()).tobytes()).hexdigest()
        if 'signature_record_json' not in h:
            with h5py.File(PREPARED, 'r') as inp:
                inp.copy('signature_record_json', h)
        if 'grn/targets_fitted_mask' not in h:
            h.create_dataset('grn/targets_fitted_mask', data=np.ones(len(genes), dtype=bool), compression='gzip')
        for key in ['regulons_json', 'motif_support_json', 'pruning_methods_json', 'candidate_modules_json']:
            if key in h: del h[key]
        json_dataset(h, 'regulons_json', definitions)
        json_dataset(h, 'motif_support_json', rows)
        json_dataset(h, 'pruning_methods_json', methods)
        json_dataset(h, 'candidate_modules_json', [plain({'TF': m.transcription_factor, 'name': m.name,
            'context': m.context, 'gene2weight': dict(m.gene2weight)}) for m in modules])
        h.attrs['stage'] = 'motif_complete'
    print('MOTIF COMPLETE:', json.dumps({a: {'status': d['status'], 'targets': len(d['gene2weight']), 'NES': d.get('NES')} for a, d in definitions.items()}), flush=True)


def inference_implementation_evidence():
    import ast
    import inspect
    import arboreto.core as core
    import sklearn.ensemble._gb as sklearn_gb
    from sklearn.ensemble import GradientBoostingRegressor
    sources = {'infer_target': inspect.getsource(infer_target),
               'arboreto_infer_partial_network': inspect.getsource(core.infer_partial_network),
               'arboreto_fit_model': inspect.getsource(core.fit_model)}
    hashes = {name: hashlib.sha256(ast.dump(ast.parse(code), include_attributes=False).encode()).hexdigest()
              for name, code in sources.items()}
    configuration = {'regressor_type': 'GBM', 'SGBM_KWARGS': core.SGBM_KWARGS,
        'early_stop_window_length': core.EARLY_STOP_WINDOW_LENGTH, 'seed_per_target': SEED,
        'effective_sklearn_parameters': GradientBoostingRegressor(**dict(core.SGBM_KWARGS, random_state=SEED)).get_params(),
        'arboreto_core_file_sha256': digest(Path(core.__file__)),
        'sklearn_GBM_file_sha256': digest(Path(sklearn_gb.__file__)), 'runtime_versions': versions(),
        'normalization': 'log10(1+native raw count/full endogenous total*1e6)',
        'scheduling_adaptation': 'Joblib8 workers instead of author CLI40; each target invokes the standard arboreto infer_partial_network with all1439 eligible TFs',
        'source_AST_sha256': hashes}
    return sources, plain(configuration)


#5. =========== Unweighted original-signature AUCell on the frozen shared 11,467-gene axis

def score():
    from pyscenic.aucell import aucell
    from ctxcore.genesig import Regulon
    AUC_THRESHOLD = 0.05
    MIN_SHARED_COVERAGE = 0.80
    SCORE_WORKERS = 4
    with h5py.File(RESULT, 'r') as h:
        definitions = json.loads(h['regulons_json'].asstr()[()])
        actual_cells = h['cell_ids'].asstr()[:]
        source_hashes = json.loads(h['source_sha256_json'].asstr()[()])
    regulons = [Regulon(name=d['regulon_name'], gene2weight=d['gene2weight'], gene2occurrence={},
        transcription_factor=anchor, context=frozenset(d.get('context', [])), nes=d.get('NES') or 0.0,
        annotation=d.get('annotation') or '', orthologous_identity=d.get('orthologous_identity') or 0.0,
        similarity_qvalue=d.get('similarity_qvalue') or 0.0)
        for anchor, d in definitions.items() if d['motif_supported']]
    with h5py.File(COMMON, 'r') as h:
        common_genes = h['gene_symbols'].asstr()[:]
        original_cells = h['cell_ids'].asstr()[:]
        original_rna = h['expression'][:]
    with h5py.File(INPUT, 'r') as h:
        obs = readframe(h['obs']); native_genes = h['genes'].asstr()[:]
        counts = sparse.csr_matrix((h['counts/data'][:], h['counts/indices'][:], h['counts/indptr'][:]),
                                  shape=tuple(h['counts/shape'][:]))
    positions = pd.Index(native_genes).get_indexer(common_genes)
    assert len(common_genes) == 11467 and (positions >= 0).all()
    vgn_rows = np.flatnonzero(obs.type.eq('VGN'))
    assert len(vgn_rows) == 520 and not set(obs.cell_id.iloc[vgn_rows]) & set(original_cells)
    vgn_rna = np.log10(1 + counts[vgn_rows][:, positions].toarray() / obs.full_total.to_numpy()[vgn_rows, None] * 1e6)
    cells = np.concatenate([original_cells, obs.cell_id.to_numpy()[vgn_rows]])
    expression = pd.DataFrame(np.vstack([original_rna, vgn_rna]), index=cells, columns=common_genes)
    assert expression.shape == (7875, 11467) and expression.index.is_unique
    assert np.isfinite(expression.to_numpy()).all() and (expression.to_numpy() >= 0).all()
    actual_positions = pd.Index(cells).get_indexer(actual_cells)
    assert (actual_positions >= 0).all()
    np.testing.assert_array_equal(cells[actual_positions], actual_cells)
    by_tf = {r.transcription_factor: r for r in regulons}
    ordered_regulons = [by_tf[a] for a in TF_CANDIDATES if a in by_tf]
    names = [r.name for r in ordered_regulons]
    common_set = set(common_genes); native_set = set(native_genes)
    coverage_rows, accepted = [], []
    for r in ordered_regulons:
        retained = set(r.genes) & common_set
        coverage = len(retained) / len(r)
        assert set(r.genes).issubset(native_set)
        accept = coverage >= MIN_SHARED_COVERAGE
        coverage_rows.append({'anchor': r.transcription_factor, 'regulon_name': r.name,
            'full_targets': len(r), 'native_targets_measured': len(r), 'native_coverage': 1.0,
            'common_targets_measured': len(retained), 'common_coverage': coverage,
            'common_AUCell_accepted': accept, 'own_TF_in_common_axis': r.transcription_factor in common_set})
        d = definitions[r.transcription_factor]
        d.update({'common_targets_measured': len(retained), 'common_target_coverage': coverage,
            'missing_common_targets': sorted(set(r.genes) - common_set), 'common_AUCell_available': accept,
            'status': 'motif_regulon_and_shared_AUCell_available' if accept else 'regulon_available_shared_AUCell_unavailable'})
        if accept: accepted.append(r)
    all_auc = pd.DataFrame(np.nan, index=cells, columns=names)
    started = time.monotonic()
    if accepted:
        # Full signatures are passed unchanged. No target cropping to evade ctxcore's 80% check.
        scored = aucell(expression, accepted, auc_threshold=AUC_THRESHOLD, seed=SEED,
                        noweights=True, normalize=False, num_workers=SCORE_WORKERS)
        np.testing.assert_array_equal(scored.index, cells)
        assert np.isfinite(scored.to_numpy()).all() and (scored.to_numpy() >= 0).all() and (scored.to_numpy() <= 1).all()
        all_auc.loc[:, scored.columns] = scored.to_numpy()
    anchor_auc = np.full((len(cells), len(ANCHORS)), np.nan)
    status_rows = []
    for j, anchor in enumerate(ANCHORS):
        d = definitions[anchor]
        if d.get('common_AUCell_available', False):
            anchor_auc[:, j] = all_auc[d['regulon_name']].to_numpy()
            primary_values = anchor_auc[actual_positions, j]
            d['actual_AUC_all_zero'] = bool((primary_values == 0).all())
            d['actual_AUC_variance'] = float(np.var(primary_values))
        status_rows.append({'anchor': anchor, 'anchor_is_TF': d['anchor_is_TF'], 'status': d['status'],
            'regulon_name': d['regulon_name'] or '', 'full_targets': len(d['gene2weight']),
            'native_anchor_measured': True, 'common_AUCell_available': bool(d.get('common_AUCell_available', False)),
            'common_target_coverage': d.get('common_target_coverage', np.nan)})
    positive = expression.gt(0).sum(axis=1).to_numpy()
    zero_tie_window_risk = positive < int(round(AUC_THRESHOLD * len(common_genes)))
    methods = {'ranking_cells': len(cells), 'original_atlas_cells': 7355, 'additional_actual_VGN_cells': 520,
        'actual_CVG_VGN_cells': 1010, 'shared_ranked_genes': len(common_genes),
        'gene_order': 'Exact original expression.h5 gene_symbols order; no native-axis expansion',
        'RNA_unit': 'log10(1+native count/full endogenous total*1e6)', 'auc_threshold': AUC_THRESHOLD,
        'seed': SEED, 'unweighted': True, 'noweights': True, 'normalize': False,
        'paper_CLI_default': 'pyscenic aucell without --weights -> noweights=True; API default differs',
        'paper_CLI_source': 'https://raw.githubusercontent.com/aertslab/pySCENIC/0.11.2/src/pyscenic/cli/pyscenic.py',
        'paper_CLI_evidence_lines': [269, 639, 645],
        'AUCell_0_11_2_to_0_12_1_compatibility': {
            'create_rankings_AST_identical': True,
            'dispatch_AST_equal_after_flatten_to_ravel_C_order_normalization': True,
            'fixture': '12 cells x 11467 genes; many zeros/ties; four signature sizes; both unweighted and weighted',
            'ranks_and_column_order_exact': True, 'AUC_max_absolute_difference': 0.0,
            'limit': 'Both versions used the current ctxcore; this does not establish historical ctxcore or database equivalence'},
        'runtime_version': importlib.metadata.version('pyscenic'),
        'minimum_full_signature_common_coverage': MIN_SHARED_COVERAGE,
        'full_signature_preserved': True, 'no_final_20_target_gate': True,
        'coverage_failure_value': 'NaN; never ctxcore all-zero placeholder',
        'zero_tie_window_risk_recorded_not_masked': True,
        'zero_tie_window_risk_actual_count': int(zero_tie_window_risk[actual_positions].sum()),
        'zero_tie_window_risk_original_count': int(zero_tie_window_risk[:7355].sum()),
        'scored_regulons': len(accepted), 'motif_regulons': len(regulons),
        'scoring_seconds': time.monotonic() - started,
        'B_test_family_adaptation': 'Actually motif-supported and shared-axis-scoreable signatures among the six user-prespecified TFs, not all TFs',
        'C_target_scope': 'Complete native motif-pruned signatures, with full native target measurability reported separately from shared B coverage',
        'limits': ['AUCell is inferred target-set expression enrichment, not proof of direct binding or causal regulation',
                   'All original 7355 atlas cell IDs retain their order; no original embedding is recomputed']}
    assert all(digest(ROOT / p) == v for p, v in source_hashes.items())
    implementation_sources, implementation_config = inference_implementation_evidence()
    with h5py.File(RESULT, 'a') as h:
        for name, code in implementation_sources.items():
            key = 'grn/' + name + '_executed_source'
            if key in h: del h[key]
            h.create_dataset(key, data=code, dtype=h5py.string_dtype())
        if 'grn/implementation_json' in h: del h['grn/implementation_json']
        json_dataset(h, 'grn/implementation_json', implementation_config)
        for key in ['regulons_json', 'regulon_names', 'original_cell_ids', 'joint_cell_ids', 'actual_score_rows',
                    'AUC_actual', 'AUC_original', 'AUC_joint', 'AUC_actual_anchors', 'AUC_original_anchors',
                    'positive_gene_count', 'zero_tie_window_risk', 'AUCell_actual_zero_tie_risk', 'AUCell_original_zero_tie_risk', 'coverage', 'anchor_status', 'aucell_methods_json', 'executed_source']:
            if key in h: del h[key]
        json_dataset(h, 'regulons_json', definitions)
        string_dataset(h, 'regulon_names', names); string_dataset(h, 'original_cell_ids', original_cells)
        string_dataset(h, 'joint_cell_ids', cells)
        for key, a in [('actual_score_rows', actual_positions), ('AUC_actual', all_auc.to_numpy()[actual_positions]),
                       ('AUC_original', all_auc.to_numpy()[:7355]), ('AUC_joint', all_auc.to_numpy()),
                       ('AUC_actual_anchors', anchor_auc[actual_positions]), ('AUC_original_anchors', anchor_auc[:7355]),
                       ('positive_gene_count', positive), ('zero_tie_window_risk', zero_tie_window_risk),
                       ('AUCell_actual_zero_tie_risk', zero_tie_window_risk[actual_positions]),
                       ('AUCell_original_zero_tie_risk', zero_tie_window_risk[:7355])]:
            h.create_dataset(key, data=a, compression='gzip', shuffle=True)
        writeframe(h, 'coverage', pd.DataFrame(coverage_rows))
        writeframe(h, 'anchor_status', pd.DataFrame(status_rows))
        json_dataset(h, 'aucell_methods_json', methods)
        h.create_dataset('executed_source', data=Path(__file__).read_text(encoding='utf-8'), dtype=h5py.string_dtype())
        h.attrs['stage'] = 'complete'
    with h5py.File(RESULT, 'r') as h:
        np.testing.assert_array_equal(h['cell_ids'].asstr()[:], actual_cells)
        np.testing.assert_array_equal(h['common_genes'].asstr()[:], common_genes)
        assert h['AUC_actual'].shape == (1010, len(names)) and h['AUC_original'].shape == (7355, len(names))
        assert json.loads(h['regulons_json'].asstr()[()]) == definitions
    print('AUCELL COMPLETE:', json.dumps(methods), flush=True)
    print(pd.DataFrame(status_rows).to_string(index=False), flush=True)
    print(pd.DataFrame(coverage_rows).to_string(index=False), flush=True)


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('stage', choices=['prepare', 'pilot', 'grn', 'prune', 'score'])
    parser.add_argument('--pilot-targets', type=int, default=64)
    args = parser.parse_args()
    if args.stage == 'prepare': prepare()
    elif args.stage == 'pilot': grn(args.pilot_targets)
    elif args.stage == 'grn': grn()
    elif args.stage == 'prune': prune()
    else: score()


## Original standalone source: fig4_VGN_modules_analysis.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
# VGN/Tbx3 signed associations, dynamic modules and model worker producing Modules.h5.
# Original source label: Modules.py (fig4_VGN_modules_analysis)
# Original source SHA256: cee845f88c9ca9611bc1e9c409210ba7da14db93af877cd7f21fbbebb8cdbe4a
# Only storage addresses and release comments were changed; analysis was not executed.

"""Original SCENIC target tests, GAMs and official PhenoGraph CPU communities."""

# Configure each file/directory placeholder independently; no storage addresses are shipped.
from pathlib import Path
ROOT = Path(".")

# Cleanup must never interpret the project/current directory as a temporary target.
def _release_cleanup_target(path):
    candidate = Path(path).resolve()
    current = Path.cwd().resolve()
    if candidate == current or candidate in current.parents or candidate == Path(candidate.anchor):
        raise RuntimeError("Configure the owned temporary cleanup path before running this section.")
    return path


#1. =========== Shared inputs and reproducible runtime
from pathlib import Path
import os
import sys
import json
import hashlib
import pickle
import subprocess
import importlib
import functools
import contextlib
import io
import random
import zipfile
import argparse
import tempfile

# Set this storage path independently before running.
ROOT = ROOT
OUT = Path(__file__).resolve().parent
# Set this storage path independently before running.
WORK = ROOT
WORK.mkdir(exist_ok=True)
# Set this storage path independently before running.
PACKAGES = ROOT
# Set this storage path independently before running. Network.h5. Produced by fig4_VGN_network_analysis.py.
NETWORK = ROOT
# Set this storage path independently before running. Early-data.h5. Produced by fig4_VGN_input_analysis.py.
DATA = ROOT
# Set this storage path independently before running. Trajectory.h5. Produced by fig4_VGN_trajectory_analysis.py.
TRAJECTORY = ROOT
# Set this storage path independently before running. tree_results-1.pkl. Accepted tree input produced by the released Figure 3 analysis.
OLD_TREE = ROOT
# Set this storage path independently before running. Gene expression plots.h5.
OLD_XY = ROOT
# Set this storage path independently before running. Fig4CD.h5.
OLD_META = ROOT
# Set this storage path independently before running. Modules.h5. Produced by fig4_VGN_modules_analysis.py.
RESULT = ROOT
# Set this storage path independently before running. Rscript.exe.
RSCRIPT = 'ROOT'
ANCHORS = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
SEED = 42
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
# Set this storage path independently before running.
os.environ.update(MPLCONFIGDIR=str(ROOT), NUMBA_CACHE_DIR=str(ROOT))
sys.dont_write_bytecode = True
import h5py
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import hypergeom, false_discovery_control


def digest(path):
    value = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(4194304), b''):
            value.update(block)
    return value.hexdigest()


def readframe(node):
    columns = json.loads(node.attrs['columns']) if 'columns' in node.attrs else list(node)
    columns = [key for key in columns if key != '_index']
    result = pd.DataFrame({key: node[key].asstr()[:] if node[key].dtype.kind in 'OSU' else node[key][:] for key in columns})
    if '_index' in node:
        result.index = node['_index'].asstr()[:]
    return result


def writeframe(handle, name, table):
    node = handle.create_group(name)
    node.attrs['columns'] = json.dumps(table.columns.tolist())
    node.create_dataset('_index', data=table.index.astype(str).to_numpy(dtype=object), dtype=h5py.string_dtype())
    for key in table:
        a = table[key].to_numpy()
        if a.dtype.kind in 'OUS':
            node.create_dataset(key, data=table[key].fillna('').astype(str).to_numpy(dtype=object), dtype=h5py.string_dtype())
        else:
            node.create_dataset(key, data=a, compression='gzip', shuffle=True)


def put(handle, name, value):
    if isinstance(value, (dict, list, tuple)):
        handle.create_dataset(name, data=json.dumps(value), dtype=h5py.string_dtype())
        return
    a = np.asarray(value)
    if a.dtype.kind in 'OUS':
        handle.create_dataset(name, data=a.astype(object), dtype=h5py.string_dtype())
    else:
        handle.create_dataset(name, data=a, compression='gzip' if a.ndim else None)


def rooted_paths(b, root):
    parent = np.full(len(b), -1, int)
    parent[root] = root
    queue = [int(root)]
    for u in queue:
        for v in np.flatnonzero(b[u]):
            if v == parent[u]:
                continue
            assert parent[v] == -1
            parent[v] = u
            queue.append(int(v))
    assert len(queue) == len(b)
    paths = []
    for tip in np.flatnonzero(b.sum(axis=0) == 1):
        if tip == root:
            continue
        path = [int(tip)]
        while path[-1] != root:
            path.append(int(parent[path[-1]]))
        paths.append(path[::-1])
    return paths


def original_graph_metadata():
    tree = pickle.load(OLD_TREE.open('rb'))
    with h5py.File(OLD_META, 'r') as h:
        obs = readframe(h['obs'])
    np.testing.assert_array_equal(obs.index.astype(str), tree['obs'].index.astype(str))
    obs['cell_id'] = obs.index.astype(str)
    obs['t'] = tree['projection']['t']
    b = tree['model']['B']
    root = int(tree['projection']['root'])
    edges = tree['projection']['edges']
    eid = tree['projection']['edge_id']
    parent = tree['projection']['parent']
    rd = tree['projection']['root_distance']
    lookup = {tuple(sorted(pair)): j for j, pair in enumerate(edges)}
    milestones = set(np.flatnonzero(b.sum(axis=0) != 2)) | {root}
    segments, edge_segment = [], np.full(len(edges), -1, int)
    for start in sorted(milestones, key=lambda node: (rd[node], int(node))):
        for child in np.flatnonzero(parent == start):
            if child == start:
                continue
            path = [int(start), int(child)]
            while path[-1] not in milestones:
                onward = np.flatnonzero(parent == path[-1])
                assert len(onward) == 1
                path.append(int(onward[0]))
            ids = [lookup[tuple(sorted(pair))] for pair in zip(path[:-1], path[1:])]
            edge_segment[ids] = len(segments)
            segments.append({'seg': f's{len(segments)}', 'start': int(start), 'end': int(path[-1]), 'edges': ids})
    assert (edge_segment >= 0).all()
    obs['natural_seg'] = np.asarray([f's{value}' for value in edge_segment[eid]])
    branch_rows = []
    for path in rooted_paths(b, root):
        ids = [lookup[tuple(sorted(pair))] for pair in zip(path[:-1], path[1:])]
        rows = np.flatnonzero(np.isin(eid, ids))
        branch_rows.extend({'row': int(row), 'branch': f'{root}_{path[-1]}'} for row in rows)
    branches = pd.DataFrame(branch_rows)
    assert branches.row.nunique() == len(obs) == 7355
    with h5py.File(OLD_XY, 'r') as h:
        np.testing.assert_array_equal(h['cell_ids'].asstr()[:], obs.index.astype(str))
        xy = h['X_umap'][:]
    return obs, branches, xy, segments


#2. =========== Read complete motif-pruned signatures and native expression
def prepare():
    protected = [NETWORK, DATA, TRAJECTORY, OLD_TREE, OLD_XY, OLD_META]
    hashes = {str(path.relative_to(ROOT)): digest(path) for path in protected}
    with h5py.File(TRAJECTORY, 'r') as h:
        cells = h['cell_ids'].asstr()[:]
        obs = readframe(h['obs'])
        primary = h['primary_path'][:]
        native_t = h['t_native'][:]
        fork_t = float(h['fork_t'][()])
    obs['t'] = native_t
    obs.index = pd.Index(cells, name='cell_id_index')
    assert len(cells) == 1010 and primary.sum() == 478 and obs.loc[primary & obs.stage_standardized.eq('E9.5').to_numpy()].shape[0] == 12
    with h5py.File(NETWORK, 'r') as h:
        assert h.attrs.get('stage') == 'complete', 'Wait for completed motif pruning and AUCell scoring'
        np.testing.assert_array_equal(h['cell_ids'].asstr()[:], cells)
        names = h['regulon_names'].asstr()[:]
        auc_actual = h['AUC_actual'][:]
        auc_original = h['AUC_original'][:]
        original_ids = h['original_cell_ids'].asstr()[:]
        signatures = json.loads(h['regulons_json'].asstr()[()])
        actual_risk = h['AUCell_actual_zero_tie_risk'][:].astype(bool)
        original_risk = h['AUCell_original_zero_tie_risk'][:].astype(bool)
    assert actual_risk.shape == (1010,) and original_risk.shape == (7355,)
    assert auc_actual.shape == (1010, len(names)) and auc_original.shape == (7355, len(names))
    with h5py.File(DATA, 'r') as h:
        all_ids = h['cell_ids'].asstr()[:]
        rows = pd.Index(all_ids).get_indexer(cells)
        assert (rows >= 0).all()
        genes = h['genes'].asstr()[:]
        native_obs = readframe(h['obs']).iloc[rows]
        totals = native_obs.full_total.to_numpy(float)
        g = h['counts']
        counts = sparse.csr_matrix((g['data'][:], g['indices'][:], g['indptr'][:]), shape=tuple(g['shape'][:]))[rows]
    gene_index = pd.Index(genes)
    target_sets, statuses = {}, {}
    for anchor in ANCHORS:
        d = signatures.get(anchor, {})
        weights = d.get('gene2weight', {})
        measured = all(g in gene_index for g in weights)
        if anchor == 'Pcdh19':
            status = 'not_applicable_non_TF'
        elif not weights:
            status = d.get('status', 'unavailable_no_motif_pruned_activating_regulon')
        elif not measured:
            status = 'unavailable_incomplete_native_signature'
        else:
            status = 'available'
        target_sets[anchor] = list(weights) if status == 'available' else []
        statuses[anchor] = {'anchor': anchor, 'C_status': status, 'motif_targets': len(weights),
            'native_measured_targets': sum(g in gene_index for g in weights),
            'regulon_name': d.get('regulon_name', anchor + '(+)'), 'RNA_association_fallback': False}
        statuses[anchor].update(network_status=d.get('status', 'unavailable_no_network_record'),
            motif_supported=bool(d.get('motif_supported', False)),
            common_AUCell_available=bool(d.get('common_AUCell_available', False)),
            common_target_coverage=d.get('common_target_coverage'),
            missing_common_targets=d.get('missing_common_targets', []))
    union = sorted({g for value in target_sets.values() for g in value})
    ix = gene_index.get_indexer(union)
    raw = np.log10(1. + counts[:, ix].toarray().astype(float) / totals[:, None] * 1e6)
    minimum = raw.min(axis=0) if len(union) else np.empty(0)
    span = raw.max(axis=0) - minimum if len(union) else np.empty(0)
    family_sizes = np.asarray([min(len(target_sets[tf]) for tf in ANCHORS if g in target_sets[tf]) for g in union])
    old_obs, old_branches, old_xy, old_segments = original_graph_metadata()
    np.testing.assert_array_equal(original_ids, old_obs.index.astype(str))
    original_allowed = (old_obs.source_state_evaluation_allowed.to_numpy(bool)
        & ~old_obs.source_rna_mixture_review_flag.to_numpy(bool) & ~original_risk)
    actual_allowed = ~actual_risk
    assert obs.loc[primary].groupby('seg').t.nunique().min() >= 5
    assert old_obs.groupby('seg').t.nunique().min() >= 5
    methods = {'official_reference': 'https://github.com/LouisFaure/sgnfates_paper/blob/main/make_fig3.ipynb',
        'official_version': 'scFates0.4.0', 'C_start': 'Actual motif-pruned activating SCENIC regulon membership, without related-gene substitution',
        'C_scope': 'Official root/leaves restriction to the478 assigned-edge real CVG-to-VGN cells; remaining1010 scope cells are external RNA-score background',
        'C_expression': 'float64 log10(1 + native count/full_endogenous_total*1e6)',
        'expression_limit': 'Author X also uses log10(1+CPM), but the author cleared ERCC/glial genes before normalization; project full-endogenous denominators need not be identical',
        'test': 'Per-natural-segment unweighted mgcv gam(exp~s(t,k=5)); global gam(exp~1); exact scFates0.4.0 extra-minus-nsegments F statistic',
        'adjustment': 'Bonferroni in actual0.4.0 code, despite its BH docstring; C family is each complete regulon separately',
        'A_cut_targets': .3, 'adjusted_p_cut': 1e-4, 'n_map': 1, 'st_cut': .8,
        'constant_features': 'Globally constant responses remain in the complete multiple-testing family with undefined p-value, amplitude0 and no selection; real model errors are retained as errors',
        'fit': "mgcv gam(exp~s(t,bs='ts'),gamma=1.5), GCV.Cp, unweighted; all root-to-tip branches, shared-cell predictions averaged",
        'AUC_test': 'A_cut=.025; actual478 path and original7355 native tree are separate candidate-regulon families, not whole-genome regulon discovery',
        'B_test_segments': 'Existing accepted nine atlas labels; natural chains retained separately for root-to-tip branch fitting',
        'B_QC': 'Existing source_state_evaluation_allowed and not source_rna_mixture_review_flag, plus exclusion of new AUCell zero-tie-risk rows from fits and colored points; original coordinates retained',
        'cluster': 'Official scFates0.4.0 CPU branch: PhenoGraph1.5.7 on fitted.T, correlation kNN15, Jaccard graph and Louvain; min_cluster_size10, only communities strictly larger than10 retained',
        'cluster_limits': 'Official CPU backend instead of paper GPU grapheno; nearest-neighbor ties and random modularity optimization need not yield bitwise-identical partitions',
        'C_score': 'Original generate_module(layer=None) formula on a separate full1010 native RNA object: unweighted gene-wise minmax raw RNA mean; test/cluster/black-line fit remain478',
        'C_normalization_scope_adaptation': 'Author called generate_module on adata_neurod1 after an unrestricted whole-tree fit, so its cell range was whole tree; this panel retains the equivalent fullactual1010 background range while restricting dynamic testing to the requested478-cell route',
        'display_rule': 'Retain every official community; display up to two largest qualifying communities, then order by fitted native-t peak; this prespecified new-panel rule differs from author manual clusters1/3',
        'Pcdh19': 'Non-TF; no regulon or target modules will be fabricated',
        'geometry': 'Frozen actual Trajectory.h5 for C and frozen original tree/UMAP for B; no UMAP or tree refit'}
    assert {key: digest(ROOT / key) for key in hashes} == hashes
    with h5py.File(RESULT, 'w') as h:
        for name, value in [('cell_ids', cells), ('anchors', np.asarray(ANCHORS)), ('target_genes', np.asarray(union, dtype=object)),
            ('target_RNA', raw), ('target_min', minimum), ('target_span', span), ('minimum_family_size', family_sizes),
            ('primary_path', primary), ('t_native', native_t), ('fork_t', fork_t), ('regulon_names', names),
            ('B_actual_allowed', actual_allowed), ('B_original_allowed', original_allowed),
            ('AUCell_actual_zero_tie_risk', actual_risk), ('AUCell_original_zero_tie_risk', original_risk),
            ('AUC_actual', auc_actual), ('AUC_original', auc_original), ('original_cell_ids', original_ids),
            ('X_umap_original', old_xy), ('target_sets_json', target_sets), ('definitions_json', statuses),
            ('network_regulons_json', signatures), ('methods_json', methods), ('input_sha256_json', hashes),
            ('original_graph_segments_json', old_segments)]:
            put(h, name, value)
        writeframe(h, 'obs', obs)
        writeframe(h, 'original_obs', old_obs)
        writeframe(h, 'original_branches', old_branches)
        put(h, 'executed_source', Path(__file__).read_text(encoding='utf-8'))
    print('Prepared complete motif signatures:', {key: len(value) for key, value in target_sets.items()}, flush=True)


#3. =========== Exact scFates0.4.0 statistics and branch GAMs through native R
R_SOURCE = "\nsuppressPackageStartupMessages({library(mgcv);library(rhdf5)})\na<-commandArgs(trailingOnly=TRUE); input<-a[1];work<-a[2];mode<-a[3]\nobs<-read.delim(file.path(work,'actual_obs.tsv'),check.names=FALSE,stringsAsFactors=FALSE)\nold<-read.delim(file.path(work,'original_obs.tsv'),check.names=FALSE,stringsAsFactors=FALSE)\nbranches<-read.delim(file.path(work,'original_branches.tsv'),check.names=FALSE,stringsAsFactors=FALSE)\nprimary<-which(tolower(as.character(obs$primary)) %in% c('true','1'))\nread_matrix<-function(name,n){\n info<-h5ls(input,recursive=FALSE);d<-info$dim[match(name,info$name)]\n sizes<-as.integer(strsplit(gsub(' ','',d),'x',fixed=TRUE)[[1]])\n if(length(sizes)==2 && any(sizes==0))return(matrix(numeric(),nrow=n,ncol=0))\n x<-h5read(input,name)\n if(is.null(dim(x))){if(!length(x))return(matrix(numeric(),nrow=n,ncol=0));x<-matrix(x,nrow=n)}\n if(ncol(x)==n)x<-t(x);stopifnot(nrow(x)==n);x\n}\nfit_template<-function(G,y,gamma=1){G$y<-y;G$model$exp<-y;gam(G=G,gamma=gamma,method='GCV.Cp')}\nmake_templates<-function(d,final=FALSE){\n d$exp<-0\n if(final)return(gam(exp~s(t,bs='ts'),data=d,fit=FALSE))\n stopifnot(length(unique(d$t))>=5)\n gam(exp~s(t,k=5),data=d,fit=FALSE)\n}\ntest_matrix<-function(X,meta){\n groups<-split(seq_len(nrow(meta)),factor(meta$seg,levels=unique(meta$seg)))\n if(any(vapply(groups,function(i)length(unique(meta$t[i]))<5,logical(1)))){\n  return(data.frame(p_val=rep(NA_real_,ncol(X)),A=NA_real_,Fstat=NA_real_,df_num=NA_real_,df_den=NA_real_,\n   deviance=NA_real_,null_deviance=NA_real_,status='error',error='A fixed k5 segment has fewer than5 unique supported times after the declared scope/QC',warnings=''))\n }\n templates<-lapply(groups,function(i)make_templates(meta[i,,drop=FALSE]))\n d<-meta;d$exp<-0;null<-gam(exp~1,data=d,fit=FALSE)\n ans<-vector('list',ncol(X))\n for(j in seq_len(ncol(X))){\n  warns<-character()\n  ans[[j]]<-tryCatch(withCallingHandlers({\n   y<-X[,j];stopifnot(all(is.finite(y)))\n   if(length(unique(y))==1){\n    data.frame(p_val=NA_real_,A=0,Fstat=NA_real_,df_num=NA_real_,df_den=NA_real_,\n     deviance=0,null_deviance=0,status='constant',error='',stringsAsFactors=FALSE)\n   }else{\n   mdl<-lapply(seq_along(groups),function(k)fit_template(templates[[k]],y[groups[[k]]]))\n   D<-sum(vapply(mdl,function(m)as.numeric(m$deviance),numeric(1)))\n   odf<-sum(vapply(mdl,function(m)as.numeric(m$df.residual),numeric(1)))-length(mdl)\n   m0<-fit_template(null,y);D0<-as.numeric(m0$deviance);df0<-as.numeric(m0$df.residual)\n   Fstat<-if(D==0)0 else (D0-D)/(df0-odf)/(D/odf)\n   p<-pf(Fstat,df0-odf,odf,lower.tail=FALSE)\n   pr<-unlist(lapply(mdl,function(m)as.numeric(predict(m))))\n   data.frame(p_val=p,A=max(pr)-min(pr),Fstat=Fstat,df_num=df0-odf,df_den=odf,\n     deviance=D,null_deviance=D0,status='ok',error='',stringsAsFactors=FALSE)\n   }\n  },warning=function(w){warns<<-c(warns,conditionMessage(w));invokeRestart('muffleWarning')}),\n  error=function(e)data.frame(p_val=NA_real_,A=NA_real_,Fstat=NA_real_,df_num=NA_real_,df_den=NA_real_,\n    deviance=NA_real_,null_deviance=NA_real_,status='error',error=conditionMessage(e),stringsAsFactors=FALSE))\n  ans[[j]]$warnings<-paste(unique(warns),collapse=' | ')\n }\n do.call(rbind,ans)\n}\nbranch_fit<-function(X,meta,membership){\n groups<-split(membership$row+1,factor(membership$branch,levels=unique(membership$branch)))\n templates<-lapply(groups,function(i)make_templates(meta[i,,drop=FALSE],TRUE))\n result<-matrix(NA_real_,nrow(X),ncol(X));counts<-tabulate(unlist(groups),nbins=nrow(X))\n for(j in seq_len(ncol(X))){\n  sums<-numeric(nrow(X))\n  for(k in seq_along(groups)){\n   i<-groups[[k]];m<-fit_template(templates[[k]],X[i,j],1.5)\n   sums[i]<-sums[i]+as.numeric(predict(m))\n  }\n  result[counts>0,j]<-sums[counts>0]/counts[counts>0]\n }\n result\n}\nwrite_numeric<-function(value,name,path){if(length(value))h5write(value,path,name)}\nout<-file.path(work,paste0(mode,'-result.h5'));if(file.exists(out))file.remove(out)\nh5createFile(out)\nif(mode=='features'){\n RNA<-read_matrix('target_RNA',nrow(obs))[primary,,drop=FALSE]\n if(ncol(RNA)){\n  stats<-test_matrix(RNA,obs[primary,,drop=FALSE]);write.table(stats,file.path(work,'target_stats.tsv'),sep='\\t',quote=TRUE,row.names=FALSE,na='NA')\n  families<-as.numeric(h5read(input,'minimum_family_size'))\n  take<-which(stats$status=='ok'&is.finite(stats$p_val)&stats$p_val*families<1e-4&stats$A>.3)\n  if(length(take)){\n   paths<-data.frame(row=seq_along(primary)-1,branch='primary')\n   fit<-branch_fit(RNA[,take,drop=FALSE],obs[primary,,drop=FALSE],paths)\n   write_numeric(fit,'target_fitted',out);write_numeric(take-1,'target_fitted_gene_indices',out)\n  }\n }\n for(scope in c('actual','original')){\n  metadata<-if(scope=='actual')obs[primary,,drop=FALSE] else old\n  matrix<-read_matrix(paste0('AUC_',scope),if(scope=='actual')nrow(obs) else nrow(old))\n  if(scope=='actual')matrix<-matrix[primary,,drop=FALSE]\n  allowed<-which(tolower(as.character(metadata$fit_allowed)) %in% c('true','1'))\n  source_rows<-if(scope=='actual')primary[allowed] else allowed\n  metadata<-metadata[allowed,,drop=FALSE];matrix<-matrix[allowed,,drop=FALSE]\n  if(!length(allowed))next\n  valid<-which(colSums(!is.finite(matrix))==0)\n  if(!length(valid))next\n  stats<-test_matrix(matrix[,valid,drop=FALSE],metadata)\n  stats$regulon_index<-valid-1\n  stats$family_size<-length(valid)\n  stats$fit_cells<-nrow(metadata)\n  stats$adjusted_p<-pmin(stats$p_val*length(valid),1)\n  stats$signi<-stats$status=='ok'&is.finite(stats$adjusted_p)&stats$adjusted_p<1e-4&stats$A>.025\n  write.table(stats,file.path(work,paste0('AUC_',scope,'_stats.tsv')),sep='\\t',quote=TRUE,row.names=FALSE,na='NA')\n  take<-which(stats$signi)\n  if(length(take)){\n   if(scope=='actual')members<-data.frame(row=seq_len(nrow(matrix))-1,branch='primary') else {\n    members<-branches[(branches$row+1) %in% allowed,,drop=FALSE]\n    members$row<-match(members$row+1,allowed)-1\n   }\n   fitted<-branch_fit(matrix[,valid[take],drop=FALSE],metadata,members)\n   write_numeric(fitted,paste0('AUC_',scope,'_fitted'),out)\n   write_numeric(valid[take]-1,paste0('AUC_',scope,'_fitted_indices'),out)\n   write_numeric(source_rows-1,paste0('AUC_',scope,'_fitted_row_indices'),out)\n  }\n }\n}else if(mode=='modules'){\n X<-read_matrix('module_scores',nrow(obs));grid<-seq(min(obs$t[primary]),max(obs$t[primary]),length.out=250)\n fitted<-matrix(NA_real_,length(primary),ncol(X));pred<-matrix(NA_real_,length(grid),ncol(X))\n d<-obs[primary,,drop=FALSE];d$exp<-0;G<-make_templates(d,TRUE)\n for(j in seq_len(ncol(X))){m<-fit_template(G,X[primary,j],1.5);fitted[,j]<-as.numeric(predict(m));pred[,j]<-as.numeric(predict(m,newdata=data.frame(t=grid)))}\n write_numeric(fitted,'module_fitted',out);write_numeric(pred,'module_grid_fitted',out);write_numeric(grid,'fit_grid',out)\n}else stop('Unknown R stage')\nh5write(as.character(packageVersion('mgcv')),out,'mgcv_version');h5write(R.version.string,out,'R_version');h5closeAll()\n"


def run_r(mode):
    with h5py.File(RESULT, 'r') as h:
        obs = readframe(h['obs'])
        obs['primary'] = h['primary_path'][:]
        obs['fit_allowed'] = h['B_actual_allowed'][:]
        original = readframe(h['original_obs'])
        original['fit_allowed'] = h['B_original_allowed'][:]
        branches = readframe(h['original_branches'])
        empty_modules = mode == 'modules' and h['module_scores'].shape[1] == 0
        native_t = h['t_native'][:]
        primary = h['primary_path'][:]
    if empty_modules:
        # Set this storage path independently before running. modules-result.h5.
        path = ROOT
        with h5py.File(path, 'w') as h:
            put(h, 'fit_grid', np.linspace(native_t[primary].min(), native_t[primary].max(), 250))
            put(h, 'module_fitted', np.empty((0, int(primary.sum()))))
            put(h, 'module_grid_fitted', np.empty((0, 250)))
        return
    if mode == 'features':
        for name in ['target_stats.tsv', 'AUC_actual_stats.tsv', 'AUC_original_stats.tsv']:
            path = WORK / name
            if path.exists():
                _release_cleanup_target(path).unlink()
    # Set this storage path independently before running. actual_obs.tsv.
    obs[['t', 'seg', 'primary', 'fit_allowed']].to_csv(ROOT, sep='\t', index=False)
    # Set this storage path independently before running. original_obs.tsv.
    original[['t', 'seg', 'fit_allowed']].to_csv(ROOT, sep='\t', index=False)
    # Set this storage path independently before running. original_branches.tsv.
    branches.to_csv(ROOT, sep='\t', index=False)
    # Set this storage path independently before running. Modules.R.
    script = ROOT
    script.write_text(R_SOURCE, encoding='utf-8')
    env = os.environ.copy()
    env.update(TMP=str(WORK), TEMP=str(WORK), TMPDIR=str(WORK), LC_ALL='C', LANG='C')
    subprocess.run([RSCRIPT, '--vanilla', str(script), str(RESULT), str(WORK), mode], check=True, env=env)


def collect_features():
    with h5py.File(RESULT, 'r') as h:
        names = h['regulon_names'].asstr()[:]
        target_genes = h['target_genes'].asstr()[:]
        obs = readframe(h['obs'])
        primary = h['primary_path'][:]
        definitions = json.loads(h['definitions_json'].asstr()[()])
    tables, fitted, meta = {}, {}, {}
    # Set this storage path independently before running. target_stats.tsv.
    target_statistics_path = ROOT
    target_statistics = pd.read_csv(target_statistics_path, sep='\t') if target_statistics_path.exists() else pd.DataFrame()
    target_statistics.index = pd.Index(target_genes, name='gene')
    # Set this storage path independently before running. features-result.h5.
    with h5py.File(ROOT, 'r') as r:
        target_fit_indices = r['target_fitted_gene_indices'][:].astype(int) if 'target_fitted_gene_indices' in r else np.empty(0, int)
        target_fit = r['target_fitted'][:].T if len(target_fit_indices) else np.empty((primary.sum(), 0))
        assert target_fit.shape == (primary.sum(), len(target_fit_indices))
        for scope, n_cells in [('actual', 1010), ('original', 7355)]:
            path = WORK / f'AUC_{scope}_stats.tsv'
            table = pd.read_csv(path, sep='\t') if path.exists() else pd.DataFrame(
                columns=['p_val', 'A', 'status', 'regulon_index', 'adjusted_p', 'signi', 'family_size', 'fit_cells'])
            if len(table):
                table.insert(0, 'regulon_name', names[table.regulon_index.to_numpy(int)])
            tables[scope] = table
            matrix = np.full((n_cells, len(names)), np.nan)
            key = f'AUC_{scope}_fitted'
            if key in r:
                rows = r[key + '_row_indices'][:].astype(int)
                columns = r[key + '_indices'][:].astype(int)
                block = r[key][:].T
                assert block.shape == (len(rows), len(columns)) and np.isfinite(block).all()
                matrix[np.ix_(rows, columns)] = block
            fitted[scope] = matrix
        for key in ['mgcv_version', 'R_version']:
            meta[key] = str(np.asarray(r[key].asstr()[()]).ravel()[0])
    reference = primary & obs.type.eq('VGN').to_numpy() & obs.stage_standardized.eq('E13.5').to_numpy()
    vmax = np.full(len(names), np.nan)
    modes, reference_cells = [], []
    for j in range(len(names)):
        values = fitted['actual'][reference, j]
        values = values[np.isfinite(values)]
        reference_cells.append(len(values))
        if len(values) and values.max() > 0:
            vmax[j] = values.max()
            modes.append('actual_primary_E13_VGN_fitted_max')
        else:
            values = fitted['original'][:, j]
            values = values[np.isfinite(values)]
            if len(values) and values.max() > 0:
                vmax[j] = values.max()
                modes.append('fallback_original_atlas_fitted_max_no_valid_E13_reference')
            else:
                modes.append('unavailable_no_positive_fitted_range')
    by_name = pd.Index(names)
    anchor_vmax = np.full(len(ANCHORS), np.nan)
    anchor_modes = np.full(len(ANCHORS), 'unavailable', dtype=object)
    anchor_fitted = {'original': np.full((7355, len(ANCHORS)), np.nan),
        'actual': np.full((1010, len(ANCHORS)), np.nan)}
    for i, anchor in enumerate(ANCHORS):
        d = definitions[anchor]
        j = by_name.get_indexer([d['regulon_name']])[0]
        d['B_regulon_index'] = int(j)
        d['B_status'] = 'not_applicable_non_TF' if anchor == 'Pcdh19' else 'unavailable_no_scored_regulon'
        d['B_actual_status'] = d['B_status']
        if anchor != 'Pcdh19' and (not d['motif_supported'] or not d['common_AUCell_available']):
            d['B_status'] = d['network_status']
            d['B_actual_status'] = d['network_status']
        if j >= 0:
            for scope, field in [('original', 'B_status'), ('actual', 'B_actual_status')]:
                if not d['motif_supported'] or not d['common_AUCell_available']:
                    continue
                table = tables[scope]
                z = table.loc[table.regulon_index.eq(j)]
                if len(z):
                    d[f'B_{scope}_A'] = float(z.A.iloc[0]) if np.isfinite(z.A.iloc[0]) else None
                    d[f'B_{scope}_adjusted_p'] = float(z.adjusted_p.iloc[0]) if np.isfinite(z.adjusted_p.iloc[0]) else None
                    d[f'B_{scope}_p_value'] = float(z.p_val.iloc[0]) if np.isfinite(z.p_val.iloc[0]) else None
                    d[f'B_{scope}_failed_gates'] = (["A>.025"] if not z.A.iloc[0] > .025 else []) + (["adjusted_p<1e-4"] if not z.adjusted_p.iloc[0] < 1e-4 else [])
                if not len(z):
                    d[field] = 'unavailable_no_testable_AUC_after_declared_QC'
                elif z.status.iloc[0] == 'error':
                    d[field] = 'unavailable_official_AUC_test_error'
                elif not np.isfinite(z.adjusted_p.iloc[0]) and z.status.iloc[0] != 'constant':
                    d[field] = 'unavailable_undefined_AUC_association_statistic'
                elif bool(z.signi.iloc[0]):
                    d[field] = 'available'
                elif np.isfinite(z.adjusted_p.iloc[0]) and z.adjusted_p.iloc[0] < 1e-4 and z.A.iloc[0] <= .025:
                    d[field] = 'AUC_amplitude_below_0.025'
                else:
                    d[field] = 'no_significant_AUC_constant_on_scope' if z.status.iloc[0] == 'constant' else 'no_significant_AUC_at_original_cutoffs'
            d['B_vmax'] = float(vmax[j]) if np.isfinite(vmax[j]) else None
            d['B_vmax_mode'] = modes[j]
            d['B_reference_cells'] = reference_cells[j]
            anchor_vmax[i] = vmax[j]
            anchor_modes[i] = modes[j]
            for scope in ['original', 'actual']:
                anchor_fitted[scope][:, i] = fitted[scope][:, j]
    with h5py.File(RESULT, 'r+') as h:
        for name in ['B_original_test', 'B_actual_test', 'B_original_fitted', 'B_actual_fitted', 'B_vmax',
            'B_vmax_mode', 'B_reference_cells', 'B_reference_mask', 'B_anchor_vmax', 'definitions_json', 'R_fit_metadata_json', 'R_source']:
            if name in h:
                del h[name]
        for name in ['target_raw_statistics', 'target_fitted_primary', 'target_fitted_gene_indices']:
            if name in h:
                del h[name]
        writeframe(h, 'target_raw_statistics', target_statistics)
        put(h, 'target_fitted_primary', target_fit)
        put(h, 'target_fitted_gene_indices', target_fit_indices)
        for name in ['B_all_regulon_vmax', 'B_all_regulon_vmax_mode', 'B_fitted_original', 'B_fitted_actual',
            'B_status', 'B_display_allowed', 'B_metric']:
            if name in h:
                del h[name]
        writeframe(h, 'B_original_test', tables['original'])
        writeframe(h, 'B_actual_test', tables['actual'])
        put(h, 'B_original_fitted', fitted['original'])
        put(h, 'B_actual_fitted', fitted['actual'])
        put(h, 'B_all_regulon_vmax', vmax)
        put(h, 'B_all_regulon_vmax_mode', np.asarray(modes, dtype=object))
        put(h, 'B_vmax', anchor_vmax)
        put(h, 'B_vmax_mode', anchor_modes)
        put(h, 'B_fitted_original', anchor_fitted['original'])
        put(h, 'B_fitted_actual', anchor_fitted['actual'])
        put(h, 'B_status', np.asarray([definitions[a]['B_status'] for a in ANCHORS], dtype=object))
        put(h, 'B_display_allowed', np.isfinite(anchor_fitted['original']))
        put(h, 'B_metric', np.asarray(['Regulon AUC'] * len(ANCHORS), dtype=object))
        put(h, 'B_reference_cells', np.asarray(reference_cells))
        put(h, 'B_reference_mask', reference)
        put(h, 'B_anchor_vmax', anchor_vmax)
        put(h, 'definitions_json', definitions)
        put(h, 'R_fit_metadata_json', meta)
        put(h, 'R_source', R_SOURCE)


#4. =========== Official PhenoGraph CPU backend, without global installation
def official_phenograph():
    WHEEL_SHA256 = '62a0fb88e13e4409875da0d382f0783a63faa4913b79f61b99528eb79c93c2e2'
    # Set this storage path independently before running.
    wheel = ROOT
    assert digest(wheel) == WHEEL_SHA256
    packages = PACKAGES
    packages.mkdir(exist_ok=True)
    with zipfile.ZipFile(wheel) as archive:
        for name in archive.namelist():
            if not name.startswith('phenograph/') or name.endswith('/'):
                continue
            target = (packages / name).resolve()
            assert target.is_relative_to(packages.resolve())
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
            assert digest(target) == hashlib.sha256(archive.read(name)).hexdigest(), 'Official runtime source changed'
    sys.path.insert(0, str(packages))
    module = importlib.import_module('phenograph.cluster')
    assert Path(module.__file__).resolve().is_relative_to(packages.resolve()), 'A different PhenoGraph was already imported'
    # Limit the official final size-sorting worker pool; no graph or mathematical parameter changes.
    if not isinstance(module.sort_by_size, functools.partial):
        module.sort_by_size = functools.partial(module.sort_by_size, n_jobs=1)
    return importlib.import_module('phenograph'), WHEEL_SHA256


def cluster_targets():
    KNN, MIN_CLUSTER_SIZE = 15, 10
    phenograph, wheel_hash = official_phenograph()
    with h5py.File(RESULT, 'r') as h:
        cells = h['cell_ids'].asstr()[:]
        genes = h['target_genes'].asstr()[:]
        raw, primary = h['target_RNA'][:], h['primary_path'][:]
        minimum, span = h['target_min'][:], h['target_span'][:]
        sets = json.loads(h['target_sets_json'].asstr()[()])
        definitions = json.loads(h['definitions_json'].asstr()[()])
        t = h['t_native'][:]
        stats = readframe(h['target_raw_statistics'])
        fitted_indices = h['target_fitted_gene_indices'][:].astype(int)
        fitted = h['target_fitted_primary'][:]
    assert fitted.shape == (primary.sum(), len(fitted_indices))
    lookup = pd.Index(genes)
    fit_lookup = pd.Index(fitted_indices)
    evidence, modules, values, selected, community_records, graphs = [], [], [], {}, [], {}
    for anchor in ANCHORS:
        selected[anchor] = []
        pool = sets[anchor]
        if not pool:
            continue
        indices = lookup.get_indexer(pool)
        z = stats.iloc[indices].copy()
        z.insert(0, 'gene', pool)
        z.insert(0, 'anchor', anchor)
        z['family_size'] = len(pool)
        z['adjusted_p'] = np.minimum(z.p_val * len(pool), 1.)
        z['signi'] = z.status.eq('ok') & z.adjusted_p.lt(1e-4) & z.A.gt(.3)
        evidence.append(z)
        if z.status.eq('error').any():
            definitions[anchor]['C_status'] = 'unavailable_target_test_error'
            continue
        changing = indices[z.signi.to_numpy()]
        definitions[anchor]['dynamic_targets'] = len(changing)
        if len(changing) <= KNN:
            definitions[anchor]['C_status'] = 'unavailable_insufficient_dynamic_targets_for_knn15'
            continue
        fcols = fit_lookup.get_indexer(changing)
        assert (fcols >= 0).all() and np.isfinite(fitted[:, fcols]).all()
        np.random.seed(SEED)
        random.seed(SEED)
        # Set this storage path independently before running.
        previous = ROOT
        official_log = io.StringIO()
        try:
            with tempfile.TemporaryDirectory(prefix=f'phenograph-{anchor}-', dir=WORK) as folder:
                assert Path(folder).resolve().is_relative_to(WORK.resolve())
                try:
                    os.chdir(folder)
                    with contextlib.redirect_stdout(official_log):
                        communities, graph, modularity = phenograph.cluster(fitted[:, fcols].T,
                            primary_metric='correlation', k=KNN, n_jobs=1, min_cluster_size=MIN_CLUSTER_SIZE)
                finally:
                    os.chdir(previous)
        except Exception as error:
            definitions[anchor]['C_status'] = 'unavailable_official_clustering_error'
            definitions[anchor]['official_clustering_error'] = str(error)
            definitions[anchor]['official_clustering_stdout'] = official_log.getvalue()
            print(anchor, 'official clustering error:', str(error), flush=True)
            continue
        finally:
            os.chdir(previous)
        for j, gene_index in enumerate(changing):
            community_records.append({'anchor': anchor, 'gene': genes[gene_index], 'community': int(communities[j])})
        graphs[anchor] = {'genes': genes[changing], 'communities': communities,
            'graph': graph.tocsr(), 'modularity': float(modularity), 'stdout': official_log.getvalue()}
        groups = sorted([(int(label), changing[communities == label]) for label in np.unique(communities) if label >= 0],
            key=lambda value: (-len(value[1]), value[0]))
        definitions[anchor]['C_status'] = 'available' if groups else 'unavailable_no_qualifying_official_community'
        definitions[anchor]['official_modularity'] = float(modularity)
        print(anchor, 'motif targets', len(pool), 'dynamic targets', len(changing),
            'official community sizes', [len(members) for _, members in groups], flush=True)
        for rank, (label, members) in enumerate(groups):
            assert len(members) > MIN_CLUSTER_SIZE and (span[members] > 0).all()
            mid = len(modules)
            score = ((raw[:, members] - minimum[members]) / span[members]).mean(axis=1)
            names = genes[members].tolist()
            modules.append({'module_id': mid, 'TF': anchor, 'anchor': anchor, 'community': label,
                'genes': names, 'n_genes': len(names), 'display': rank < 2, 'kind': 'SCENIC_motif_target_community',
                'title': 'SCENIC targets', 'title_basis': 'Pending actual target composition review',
                'gene_sha256': hashlib.sha256('\n'.join(sorted(names)).encode()).hexdigest()})
            values.append(score)
            if rank < 2:
                selected[anchor].append(mid)
    with h5py.File(RESULT, 'r+') as h:
        for name in ['module_definitions_json', 'selected_modules_json', 'module_scores', 'definitions_json', 'target_association', 'official_communities', 'cluster_methods_json']:
            if name in h:
                del h[name]
        if 'clusters' in h:
            del h['clusters']
        put(h, 'module_definitions_json', modules)
        put(h, 'selected_modules_json', selected)
        put(h, 'module_scores', np.column_stack(values) if values else np.empty((len(cells), 0)))
        put(h, 'definitions_json', definitions)
        writeframe(h, 'target_association', pd.concat(evidence, ignore_index=True) if evidence else pd.DataFrame())
        writeframe(h, 'official_communities', pd.DataFrame(community_records))
        for anchor, result in graphs.items():
            node = h.create_group('clusters/' + anchor)
            put(node, 'genes', result['genes'])
            put(node, 'communities', result['communities'])
            put(node, 'modularity', result['modularity'])
            put(node, 'official_stdout', result['stdout'])
            graph = result['graph']
            for name, value in [('data', graph.data), ('indices', graph.indices), ('indptr', graph.indptr), ('shape', np.array(graph.shape))]:
                put(node, 'graph/' + name, value)
        put(h, 'cluster_methods_json', {'backend': 'Official scFates0.4.0 CPU PhenoGraph1.5.7 call',
            'wheel_sha256': wheel_hash, 'knn': KNN, 'metric': 'correlation', 'min_cluster_size': MIN_CLUSTER_SIZE,
            'sorting_pool_workers': 1, 'mathematical_code_modified': False,
            'randomness': 'Python/NumPy seeded42; original C++ Louvain has no seed CLI, so stored partitions are authoritative and rerun stochastic variation is possible'})
    print('Official communities:', {anchor: selected[anchor] for anchor in ANCHORS}, flush=True)


def collect_module_fit():
    with h5py.File(RESULT, 'r') as h:
        primary = h['primary_path'][:]
        modules = json.loads(h['module_definitions_json'].asstr()[()])
        selected = json.loads(h['selected_modules_json'].asstr()[()])
    # Set this storage path independently before running. modules-result.h5.
    with h5py.File(ROOT, 'r') as r:
        grid = r['fit_grid'][:] if 'fit_grid' in r else np.linspace(0, 1, 250)
        fitted = r['module_fitted'][:].T if 'module_fitted' in r else np.empty((primary.sum(), 0))
        grid_fitted = r['module_grid_fitted'][:].T if 'module_grid_fitted' in r else np.empty((len(grid), 0))
    assert fitted.shape == (primary.sum(), len(modules)) and np.isfinite(fitted).all()
    assert grid_fitted.shape == (len(grid), len(modules)) and np.isfinite(grid_fitted).all()
    for d in modules:
        j = d['module_id']
        y = grid_fitted[:, j]
        d['peak_t_native'] = float(grid[np.argmax(y)])
        d['fitted_range'] = float(np.ptp(y))
        thirds = [float(part.mean()) for part in np.array_split(y, 3)]
        if thirds[1] > max(thirds[0], thirds[2]) + .05:
            title = 'Transient targets'
        elif thirds[2] - thirds[0] > .05:
            title = 'Increasing targets'
        elif thirds[0] - thirds[2] > .05:
            title = 'Decreasing targets'
        else:
            title = 'Weakly varying targets'
        d.update(title=title, descriptive_title=title,
            title_basis='Descriptive raw-target module trend; no functional enrichment implied', profile_thirds=thirds)
    selected = {anchor: sorted(ids, key=lambda mid: modules[mid]['peak_t_native']) for anchor, ids in selected.items()}
    with h5py.File(RESULT, 'r+') as h:
        for name in ['module_fitted_primary', 'module_grid_fitted', 'fit_grid', 'module_definitions_json', 'selected_modules_json', 'executed_source']:
            if name in h:
                del h[name]
        put(h, 'module_fitted_primary', fitted)
        put(h, 'module_grid_fitted', grid_fitted)
        put(h, 'fit_grid', grid)
        put(h, 'module_definitions_json', modules)
        put(h, 'selected_modules_json', selected)
        put(h, 'executed_source', Path(__file__).read_text(encoding='utf-8'))


#5. =========== Name actual target communities with auditable local GO evidence
GO_R_SOURCE = r'''
args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages({library(AnnotationDbi);library(org.Mm.eg.db);library(GO.db);library(jsonlite)})
background <- unique(scan(args[1],what='character',quiet=TRUE))
valid <- intersect(background,keys(org.Mm.eg.db,keytype='SYMBOL'))
z <- suppressMessages(AnnotationDbi::select(org.Mm.eg.db,keys=valid,keytype='SYMBOL',columns=c('GOALL','ONTOLOGYALL','EVIDENCEALL')))
z <- z[!is.na(z$GOALL)&z$ONTOLOGYALL=='BP'&z$GOALL!='GO:0008150'&!is.na(z$EVIDENCEALL)&z$EVIDENCEALL!='ND',]
pairs <- unique(z[c('SYMBOL','GOALL')])
terms <- suppressMessages(AnnotationDbi::select(GO.db,keys=unique(pairs$GOALL),keytype='GOID',columns=c('TERM','ONTOLOGY')))
terms <- unique(terms[!is.na(terms$TERM)&terms$ONTOLOGY=='BP',c('GOID','TERM')])
pairs <- pairs[pairs$GOALL %in% terms$GOID,]
versions <- vapply(c('AnnotationDbi','org.Mm.eg.db','GO.db','jsonlite'),function(p)as.character(packageVersion(p)),character(1))
result <- list(pairs=pairs,terms=terms,versions=as.list(versions),R_version=R.version.string,
 database_files=list(org.Mm.eg.db=AnnotationDbi::dbfile(org.Mm.eg.db),GO.db=AnnotationDbi::dbfile(GO.db)),
 evidence_codes=sort(unique(z$EVIDENCEALL)))
write_json(result,args[2],auto_unbox=TRUE,na='null',dataframe='rows')
'''


def annotate():
    MIN_TERM_SIZE = 10
    MAX_TERM_SIZE = 500
    MIN_NAMING_HITS = 3
    FDR_THRESHOLD = .05
    with h5py.File(NETWORK,'r') as h:
        background = set(h['genes'].asstr()[:])
    with h5py.File(RESULT,'r') as h:
        all_module_defs = json.loads(h['module_definitions_json'].asstr()[()])
    module_defs = [d for d in all_module_defs if d['display']]
    if not module_defs:
        return
    with tempfile.TemporaryDirectory(prefix='program-go-',dir=OUT) as temporary:
        work = Path(temporary)
        # Set this storage path independently before running. background.txt.
        (ROOT).write_text('\n'.join(sorted(background)),encoding='utf-8')
        # Set this storage path independently before running. annotation.R.
        (ROOT).write_text(GO_R_SOURCE,encoding='utf-8')
        env = os.environ.copy()
        env.update(TMP=str(work),TEMP=str(work),TMPDIR=str(work),LC_ALL='C',LANG='C')
        # Set this storage path independently before running. annotation.R.
        subprocess.run([str(RSCRIPT),'--vanilla',str(ROOT),str(ROOT),
                        # Set this storage path independently before running. annotation.json.
                        str(ROOT)],check=True,env=env)
        # Set this storage path independently before running. annotation.json.
        annotation = json.loads((ROOT).read_text(encoding='utf-8'))
    pairs = pd.DataFrame(annotation['pairs']).drop_duplicates()
    term_names = pd.DataFrame(annotation['terms']).set_index('GOID').TERM.to_dict()
    term_genes = {term:set(part.SYMBOL) for term,part in pairs.groupby('GOALL')}
    annotated = set(pairs.SYMBOL)
    records = []
    for d in module_defs:
        universe = background
        u_bp = universe&annotated
        g = set(d['genes'])
        assert g<=universe
        g_bp = g&annotated
        for term,members in term_genes.items():
            in_background = members&universe
            K = len(in_background)
            if K<MIN_TERM_SIZE or K>MAX_TERM_SIZE:
                continue
            hit = sorted(g&in_background)
            k,n,N = len(hit),len(g),len(universe)
            p = float(hypergeom.sf(k-1,N,K,n))
            p_bp = float(hypergeom.sf(k-1,len(u_bp),K,len(g_bp)))
            records.append({'module_id':d['module_id'],'anchor':d['anchor'],'GO_ID':term,
                'Description':term_names[term],'Count':k,'input_n':n,'term_background_n':K,'background_n':N,
                'BP_input_n':len(g_bp),'BP_background_n':len(u_bp),'FoldEnrichment':k/n/(K/N),
                'PValue':p,'PValue_BP':p_bp,'hit_genes':';'.join(hit)})
    result = pd.DataFrame(records)
    result['FDR'] = false_discovery_control(result.PValue.to_numpy())
    result['FDR_BP'] = false_discovery_control(result.PValue_BP.to_numpy())
    result['naming_support'] = ((result.Count>=MIN_NAMING_HITS)&(result.FDR<FDR_THRESHOLD)
        &(result.FDR_BP<FDR_THRESHOLD)&(result.FoldEnrichment>1))
    assert result.loc[result.Count.eq(0),'PValue'].eq(1).all()
    title_categories = [
        ('axon guidance','Axon guidance-targets'),
        ('neuron projection','Neurite development-targets'),
        ('axon development','Axon development-targets'),
        ('neuron differentiation','Neuron differentiation-targets'),
        ('neurogenesis','Neurogenesis-targets'),
        ('synap','Synapse-targets'),
        ('cell adhesion','Cell adhesion-targets'),
        ('cytoskeleton','Cytoskeleton-targets'),
        ('ion transport','Ion transport-targets'),
        ('cell cycle','Cell cycle-targets'),
        ('rna processing','RNA processing-targets')]
    summaries = []
    for d in module_defs:
        z = result.loc[result.module_id.eq(d['module_id'])]
        supported = z.loc[z.naming_support].sort_values(['FDR','PValue','GO_ID'])
        d['GO_significant_terms'] = int(z.FDR.lt(FDR_THRESHOLD).sum())
        d['GO_robust_naming_terms'] = len(supported)
        d['GO_title_term'] = ''
        d['GO_title_FDR'] = None
        chosen = None
        # Choose one supported, non-generic biological description; never infer function from an uncorrected term.
        for row in supported.itertuples():
            for keyword,title in title_categories:
                if keyword in row.Description.lower():
                    chosen = (row,title)
                    break
            if chosen:
                break
        if chosen:
            row,title = chosen
            title = title.replace('-targets', ' targets')
            d['title'] = title
            d['title_basis'] = 'GO BP enrichment with joint BH and BP-annotated background sensitivity'
            d['GO_title_term'] = row.GO_ID+': '+row.Description
            d['GO_title_FDR'] = float(row.FDR)
        summaries.append({'module_id':d['module_id'],'anchor':d['anchor'],'genes':d['n_genes'],
            'display':d['display'],'title':d['title'],'title_basis':d['title_basis'],
            'GO_significant_terms':d['GO_significant_terms'],'GO_robust_naming_terms':d['GO_robust_naming_terms'],
            'minimum_FDR':float(z.FDR.min()),'minimum_naming_FDR':float(supported.FDR.min()) if len(supported) else np.nan})
    parameters = {'ontology':'BP','annotation':'GOALL ancestors, all non-ND evidence, unique symbol/GO pairs; BP root excluded',
        'background':'Complete native SCENIC inference-gene universe; unmapped symbols retained, no own-TF exclusion',
        'background_symbols':len(background),'min_term_size':MIN_TERM_SIZE,'max_term_size':MAX_TERM_SIZE,
        'test':'One-sided hypergeometric over-representation','BH_family':len(result),
        'BH_scope':'All eligible terms across every displayed official community, including zero-overlap terms',
        'BP_sensitivity':'Same hypotheses, BP-annotated symbol universe; BH separately across all module-term tests',
        'naming_rule':'At least3 hits, fold>1, main q<.05 and sensitivity q<.05; functional vocabulary bound to term text',
        'limits':'Exploratory annotation of correlated motif-pruned target communities; not proof of direct regulation, independent validation or fate conversion',
        'versions':annotation['versions'],'R_version':annotation['R_version'],
        'database_files':annotation['database_files'],
        'database_sha256':{k:digest(v) for k,v in annotation['database_files'].items()},
        'evidence_codes':annotation['evidence_codes']}
    with h5py.File(RESULT,'r+') as h:
        for key in ['GO/results','GO/summary','GO/methods_json','module_definitions_json','executed_source']:
            if key in h:del h[key]
        writeframe(h,'GO/results',result);writeframe(h,'GO/summary',pd.DataFrame(summaries));put(h,'GO/methods_json',parameters)
        put(h,'module_definitions_json',all_module_defs);put(h,'executed_source',Path(__file__).read_text(encoding='utf-8'))
    print(pd.DataFrame(summaries).loc[lambda x:x.display,['anchor','module_id','title','GO_robust_naming_terms']].to_string(index=False),flush=True)


#6. =========== Verify final evidence, fixed geometry and display contracts
def finalize():
    with h5py.File(RESULT, 'r') as h:
        hashes = json.loads(h['input_sha256_json'].asstr()[()])
        definitions = json.loads(h['definitions_json'].asstr()[()])
        modules = json.loads(h['module_definitions_json'].asstr()[()])
        selected = json.loads(h['selected_modules_json'].asstr()[()])
        sets = json.loads(h['target_sets_json'].asstr()[()])
        raw = h['target_RNA'][:]
        genes = pd.Index(h['target_genes'].asstr()[:])
        scores = h['module_scores'][:]
        b_old = h['B_fitted_original'][:]
        b_actual = h['B_fitted_actual'][:]
        primary = h['primary_path'][:]
        minimum, span = h['target_min'][:], h['target_span'][:]
        allowed_old, allowed_actual = h['B_original_allowed'][:], h['B_actual_allowed'][:]
        assert b_old.shape == (7355, 7) and b_actual.shape == (1010, 7)
        assert scores.shape == (1010, len(modules)) and np.isfinite(scores).all()
        assert h['module_grid_fitted'].shape == (250, len(modules))
        assert h['module_fitted_primary'].shape == (478, len(modules))
        assert np.isnan(b_old[~allowed_old]).all()
        assert np.isnan(b_actual[~(allowed_actual & primary)]).all()
        assert np.isnan(b_old[:, -1]).all() and np.isnan(b_actual[:, -1]).all() and not selected['Pcdh19']
        assert all(len(selected[a]) <= 2 for a in ANCHORS)
        if len(genes):
            np.testing.assert_array_equal(raw.min(axis=0), minimum)
            np.testing.assert_array_equal(raw.max(axis=0) - minimum, span)
        for d in modules:
            assert d['n_genes'] > 10 and set(d['genes']) <= set(sets[d['TF']])
            ix = genes.get_indexer(d['genes'])
            assert (ix >= 0).all() and (span[ix] > 0).all()
            np.testing.assert_allclose(scores[:, d['module_id']], ((raw[:, ix] - minimum[ix]) / span[ix]).mean(axis=1), rtol=0, atol=0)
        for anchor in ANCHORS:
            definitions[anchor]['displayed_modules'] = len(selected[anchor])
    assert {key: digest(ROOT / key) for key in hashes} == hashes
    # Set this storage path independently before running.
    reference = ROOT
    # Set this storage path independently before running. Sources.json.
    sources = json.loads((ROOT).read_text(encoding='utf-8'))
    code = {path.name: path.read_text(encoding='utf-8') for path in reference.glob('*.py')}
    checks = {'protected_inputs_unchanged': True, 'B_original_shape': list(b_old.shape),
        'B_actual_shape': list(b_actual.shape), 'C_module_shape': list(scores.shape),
        'native_primary_cells': int(primary.sum()), 'B_RNA_fallback': False,
        'C_related_gene_fallback': False, 'Pcdh19_not_applicable': True,
        'C_minmax_scope': 1010, 'C_test_cluster_fit_scope': 478,
        'C_scores_recomputed_exactly': True, 'community_minimum': 'Strictly more than10, official PhenoGraph default',
        'official_kNN': 15, 'official_CPU_runtime_preflight': 'Passed on40 existing biological AUC profiles',
        'R_direct_template_preflight': 'F statistics and final GAMs matched direct mgcv exactly on real native RNA and an existing AUC profile; these pilot inputs are not the final SCENIC data'}
    with h5py.File(RESULT, 'r+') as h:
        for name in ['definitions_json', 'checks_json', 'official_sources_json', 'official_code_json', 'executed_source']:
            if name in h:
                del h[name]
        put(h, 'definitions_json', definitions)
        put(h, 'checks_json', checks)
        put(h, 'official_sources_json', sources)
        put(h, 'official_code_json', code)
        put(h, 'executed_source', Path(__file__).read_text(encoding='utf-8'))
    print(pd.DataFrame([definitions[a] for a in ANCHORS])[['anchor', 'B_status', 'C_status', 'displayed_modules']].to_string(index=False), flush=True)




if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('command', choices=['prepare', 'features', 'cluster', 'modules', 'annotate', 'finalize', 'all'])
    args = parser.parse_args()
    if args.command in ['prepare', 'all']:
        prepare()
    if args.command in ['features', 'all']:
        run_r('features')
        collect_features()
    if args.command in ['cluster', 'all']:
        cluster_targets()
    if args.command in ['modules', 'all']:
        run_r('modules')
        collect_module_fit()

    if args.command in ['annotate', 'all']:
        annotate()
    if args.command in ['finalize', 'all']:
        finalize()


## Original standalone source: fig4_validate_figure_output.py

This complete existing program retains its original command-line entry point and runtime assumptions. Use it in its original independent script context. Executing the cell in Jupyter would invoke the unchanged entry point with Jupyter arguments; this section is not a new notebook dispatcher.


In [ ]:
#!/usr/bin/env python3
"""Fail-closed validation for 600-DPI PNGs and editable figure PDFs.

The default PDF profile is intentionally strict. It is designed for figures that
must remain straightforward to edit after opening in Adobe Illustrator:

* no more than four explicit, nonzero-winding clipping paths per page;
* clipping paths are sibling, axis-aligned rectangles (never nested);
* every clipping path is explicitly scoped by q/Q and ends with W n;
* no raster images, soft/image masks, inline images, text clipping, shading,
  or uninspected recursive Form XObjects;
* live, embedded, Unicode-mapped Arial/SimSun text; and
* at least one painted vector path.

The program writes its report to stdout only. Poppler render files are created in
a task-local temporary directory and removed before exit unless the caller uses
``--render-dir``; that option always creates and retains a new unique child
directory, so existing files are never overwritten.
"""

from __future__ import annotations

import argparse
import math
import os
import re
import secrets
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Iterable, Sequence

try:
    from PIL import Image
    from pypdf import PdfReader
    from pypdf.generic import ArrayObject, ContentStream, DictionaryObject, IndirectObject
except ImportError as exc:  # pragma: no cover - exercised only in a broken runtime
    print(f"[NOT CERTIFIED] Required Python dependency is unavailable: {exc}")
    raise SystemExit(1)


EPSILON = 1e-7
PATH_BUILDING = {b"m", b"l", b"c", b"v", b"y", b"h", b"re"}
PATH_ENDING = {b"S", b"s", b"f", b"F", b"f*", b"B", b"B*", b"b", b"b*", b"n"}
TEXT_SHOWING = {b"Tj", b"TJ", b"'", b'"'}
FILLING = {b"f", b"F", b"f*"}
STROKING = {b"S", b"s"}
FILL_STROKING = {b"B", b"B*", b"b", b"b*"}
INLINE_IMAGE_OPS = {b"BI", b"ID", b"EI", b"INLINE IMAGE"}
SUBSET_PREFIX = re.compile(r"^[A-Z]{6}\+")
CJK_RE = re.compile(
    "[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff\U00020000-\U0002fa1f]"
)
ASCII_ALNUM_RE = re.compile(r"[A-Za-z0-9]")


@dataclass
class Issue:
    code: str
    message: str
    outcome: str = "FAIL"


@dataclass
class ClipEvent:
    number: int
    scope: str
    depth: int
    bbox: tuple[float, float, float, float] | None
    kinds: Counter[str] = field(default_factory=Counter)


@dataclass
class FontUse:
    scope: str
    resource_name: str
    base_font: str
    subtype: str
    embedded: bool
    to_unicode: bool


@dataclass
class GraphicsState:
    ctm: tuple[float, float, float, float, float, float] = (1, 0, 0, 1, 0, 0)
    clip_depth: int = 0
    active_clips: tuple[int, ...] = ()
    font_resource: str | None = None
    font_object: Any = None
    text_render_mode: int = 0


@dataclass
class PdfPageStats:
    page_number: int
    crop_size_points: tuple[float, float] = (0.0, 0.0)
    media_size_points: tuple[float, float] = (0.0, 0.0)
    rotation: int = 0
    clip_events: list[ClipEvent] = field(default_factory=list)
    max_clip_depth: int = 0
    q_count: int = 0
    Q_count: int = 0
    form_calls: int = 0
    image_count: int = 0
    inline_image_count: int = 0
    text_show_count: int = 0
    text_object_count: int = 0
    vector_paint_count: int = 0
    vector_ops: Counter[str] = field(default_factory=Counter)
    font_uses: dict[tuple[str, str, str], FontUse] = field(default_factory=dict)


@dataclass
class ValidationResult:
    path: Path
    kind: str
    issues: list[Issue] = field(default_factory=list)
    details: list[str] = field(default_factory=list)

    @property
    def passed(self) -> bool:
        return not self.issues

    @property
    def status(self) -> str:
        if any(issue.outcome == "FAIL" for issue in self.issues):
            return "FAIL"
        if self.issues:
            return "NOT CERTIFIED"
        return "PASS"

    def fail(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "FAIL"))

    def not_certified(self, code: str, message: str) -> None:
        self.issues.append(Issue(code, message, "NOT CERTIFIED"))


def dereference(value: Any) -> Any:
    try:
        return value.get_object()
    except AttributeError:
        return value


def dictionary(value: Any) -> DictionaryObject | dict:
    value = dereference(value)
    return value if isinstance(value, (DictionaryObject, dict)) else {}


def numeric(value: Any) -> float:
    return float(dereference(value))


def normalize_font_name(value: Any) -> str:
    name = str(value or "").lstrip("/")
    return SUBSET_PREFIX.sub("", name)


def font_family(base_font: str) -> str | None:
    compact = re.sub(r"[^a-z0-9]", "", normalize_font_name(base_font).lower())
    if compact.startswith("arial"):
        return "Arial"
    if compact.startswith("simsun") or compact.startswith("nsimsun"):
        return "SimSun"
    return None


def font_descriptor(font_obj: Any) -> dict:
    font = dictionary(font_obj)
    subtype = str(font.get("/Subtype", ""))
    if subtype == "/Type0":
        descendants = dereference(font.get("/DescendantFonts", []))
        if descendants:
            descendant = dictionary(descendants[0])
            return dictionary(descendant.get("/FontDescriptor", {}))
    return dictionary(font.get("/FontDescriptor", {}))


def font_is_embedded(font_obj: Any) -> bool:
    descriptor = font_descriptor(font_obj)
    return any(key in descriptor for key in ("/FontFile", "/FontFile2", "/FontFile3"))


def object_identity(value: Any) -> tuple[Any, ...]:
    if isinstance(value, IndirectObject):
        return ("indirect", value.idnum, value.generation)
    obj = dereference(value)
    ref = getattr(obj, "indirect_reference", None)
    if isinstance(ref, IndirectObject):
        return ("indirect", ref.idnum, ref.generation)
    return ("direct", id(obj))


def matrix_multiply(
    first: tuple[float, float, float, float, float, float],
    second: tuple[float, float, float, float, float, float],
) -> tuple[float, float, float, float, float, float]:
    """Compose affine transforms so result(point) == first(second(point))."""
    a1, b1, c1, d1, e1, f1 = first
    a2, b2, c2, d2, e2, f2 = second
    return (
        a1 * a2 + c1 * b2,
        b1 * a2 + d1 * b2,
        a1 * c2 + c1 * d2,
        b1 * c2 + d1 * d2,
        a1 * e2 + c1 * f2 + e1,
        b1 * e2 + d1 * f2 + f1,
    )


def transform_point(
    matrix: tuple[float, float, float, float, float, float], x: float, y: float
) -> tuple[float, float]:
    a, b, c, d, e, f = matrix
    return (a * x + c * y + e, b * x + d * y + f)


def normalized_bbox(points: Sequence[tuple[float, float]]) -> tuple[float, float, float, float]:
    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return (left, bottom, right - left, top - bottom)


def nearly_equal(left: float, right: float, tolerance: float) -> bool:
    return abs(left - right) <= tolerance


def boxes_match(
    observed: tuple[float, float, float, float],
    expected: tuple[float, float, float, float],
) -> bool:
    tolerance = max(0.5, max(abs(value) for value in expected[2:]) * 1e-4)
    return all(nearly_equal(a, b, tolerance) for a, b in zip(observed, expected))


def image_blank_reason(image: Image.Image) -> str | None:
    """Return a concrete blank-page reason without allocating a full RGBA copy."""
    extrema: Any = image.getextrema()
    if not isinstance(extrema, tuple):
        extrema = (extrema,)
    elif extrema and not isinstance(extrema[0], tuple):
        extrema = (extrema,)
    bands = image.getbands()
    if "A" in bands and extrema[bands.index("A")] == (0, 0):
        return "all pixels are fully transparent"
    if all(low == high for low, high in extrema):
        return "all pixels have one uniform color"
    return None


def parse_simple_rectangle(
    path_ops: Sequence[tuple[bytes, list[Any]]],
    ctm: tuple[float, float, float, float, float, float],
) -> tuple[tuple[float, float, float, float] | None, str | None]:
    if len(path_ops) == 1 and path_ops[0][0] == b"re":
        operands = path_ops[0][1]
        if len(operands) != 4:
            return None, "rectangle operator does not have four operands"
        x, y, width, height = (numeric(value) for value in operands)
        if abs(width) <= EPSILON or abs(height) <= EPSILON:
            return None, "rectangle has zero width or height"
        raw_points = [
            (x, y),
            (x + width, y),
            (x + width, y + height),
            (x, y + height),
        ]
    else:
        operators = [operator for operator, _ in path_ops]
        if not operators or operators[0] != b"m" or any(
            operator not in {b"m", b"l", b"h"} for operator in operators
        ):
            return None, "clip path is not one re rectangle or one m/l/h rectangle"
        if operators.count(b"m") != 1 or operators.count(b"h") > 1:
            return None, "clip path contains multiple subpaths"
        raw_points: list[tuple[float, float]] = []
        for operator, operands in path_ops:
            if operator in {b"m", b"l"}:
                if len(operands) != 2:
                    return None, "rectangle vertex does not have two coordinates"
                raw_points.append((numeric(operands[0]), numeric(operands[1])))
        if len(raw_points) == 5 and all(
            nearly_equal(a, b, 1e-6) for a, b in zip(raw_points[0], raw_points[-1])
        ):
            raw_points.pop()
        if len(raw_points) != 4 or (operators[-1] != b"h" and len(path_ops) != 5):
            return None, "m/l/h clip does not contain exactly four rectangle vertices"

    points = [transform_point(ctm, x, y) for x, y in raw_points]
    tolerance = max(1e-5, max(abs(value) for point in points for value in point) * 1e-7)
    for start, end in zip(points, points[1:] + points[:1]):
        dx, dy = abs(end[0] - start[0]), abs(end[1] - start[1])
        if not (dx <= tolerance < dy or dy <= tolerance < dx):
            return None, "clip rectangle is rotated, skewed, or self-degenerate"
    unique_x = []
    unique_y = []
    for x, y in points:
        if not any(nearly_equal(x, item, tolerance) for item in unique_x):
            unique_x.append(x)
        if not any(nearly_equal(y, item, tolerance) for item in unique_y):
            unique_y.append(y)
    if len(unique_x) != 2 or len(unique_y) != 2:
        return None, "clip path does not form one axis-aligned rectangle"
    return normalized_bbox(points), None


class PdfAuditor:
    def __init__(
        self,
        path: Path,
        result: ValidationResult,
        max_clips: int,
        expected_clips: int | None,
        declared_clip_boxes: list[tuple[float, float, float, float]],
        allow_raster_images: bool,
        allow_form_xobjects: bool,
        allow_no_text: bool,
        allow_no_vector: bool,
    ) -> None:
        self.path = path
        self.result = result
        self.max_clips = max_clips
        self.expected_clips = expected_clips
        self.declared_clip_boxes = declared_clip_boxes
        self.allow_raster_images = allow_raster_images
        self.allow_form_xobjects = allow_form_xobjects
        self.allow_no_text = allow_no_text
        self.allow_no_vector = allow_no_vector
        self.reader: PdfReader | None = None
        self._scanned_resources: set[tuple[Any, ...]] = set()
        self._reported_fonts: set[tuple[Any, ...]] = set()

    def audit(self) -> list[PdfPageStats]:
        try:
            self.reader = PdfReader(str(self.path), strict=True)
        except Exception as exc:
            self.result.not_certified("PDF_PARSE", f"PDF cannot be parsed strictly: {exc}")
            return []
        if self.reader.is_encrypted:
            self.result.not_certified("PDF_ENCRYPTED", "encrypted PDFs cannot be inspected")
            return []
        if not self.reader.pages:
            self.result.fail("PDF_EMPTY", "PDF contains no pages")
            return []

        page_stats: list[PdfPageStats] = []
        for page_number, page in enumerate(self.reader.pages, start=1):
            crop_box = page.cropbox
            media_box = page.mediabox
            user_unit = float(page.get("/UserUnit", 1) or 1)
            rotation = int(page.get("/Rotate", 0) or 0) % 360
            stats = PdfPageStats(
                page_number,
                crop_size_points=(
                    float(crop_box.width) * user_unit,
                    float(crop_box.height) * user_unit,
                ),
                media_size_points=(
                    float(media_box.width) * user_unit,
                    float(media_box.height) * user_unit,
                ),
                rotation=rotation,
            )
            page_stats.append(stats)
            resources = dictionary(page.get("/Resources", {}))
            page_boxes = [
                (
                    float(crop_box.left),
                    float(crop_box.bottom),
                    float(crop_box.width),
                    float(crop_box.height),
                ),
                (
                    float(media_box.left),
                    float(media_box.bottom),
                    float(media_box.width),
                    float(media_box.height),
                ),
            ]
            allowed_boxes = page_boxes + self.declared_clip_boxes
            contents = page.get_contents()
            if contents is None:
                self.result.fail("PDF_NO_CONTENT", f"page {page_number} has no content stream")
                continue
            try:
                content_stream = ContentStream(contents, self.reader)
            except Exception as exc:
                self.result.not_certified(
                    "CONTENT_PARSE", f"page {page_number} content stream cannot be parsed: {exc}"
                )
                continue
            self._scan_resources(resources, f"page {page_number}", stats)
            self._analyze_operations(
                content_stream.operations,
                resources,
                f"page {page_number}",
                stats,
                GraphicsState(),
                allowed_boxes,
                form_stack=set(),
                implicit_scope=False,
            )
            self._validate_page_text(page, stats)
            self._validate_page_summary(stats)
        return page_stats

    def _scan_resources(self, resources: dict, scope: str, stats: PdfPageStats) -> None:
        identity = object_identity(resources)
        if identity in self._scanned_resources:
            return
        self._scanned_resources.add(identity)

        ext_states = dictionary(resources.get("/ExtGState", {}))
        for name, reference in ext_states.items():
            ext_state = dictionary(reference)
            if "/SMask" in ext_state and str(dereference(ext_state["/SMask"])) != "/None":
                self.result.fail(
                    "SOFT_MASK", f"{scope} ExtGState {name} contains a non-None /SMask"
                )

        patterns = dictionary(resources.get("/Pattern", {}))
        if patterns:
            self.result.fail(
                "PATTERN_RESOURCE",
                f"{scope} contains Pattern resources; their Illustrator editability is not certified",
            )
        shadings = dictionary(resources.get("/Shading", {}))
        if shadings:
            self.result.fail(
                "SHADING_RESOURCE",
                f"{scope} contains Shading resources; their Illustrator editability is not certified",
            )

        xobjects = dictionary(resources.get("/XObject", {}))
        for name, reference in xobjects.items():
            xobject = dictionary(reference)
            subtype = str(xobject.get("/Subtype", ""))
            if "/SMask" in xobject:
                self.result.fail("SOFT_MASK", f"{scope} XObject {name} contains /SMask")
            if "/Mask" in xobject:
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} contains /Mask")
            if bool(xobject.get("/ImageMask", False)):
                self.result.fail("IMAGE_MASK", f"{scope} XObject {name} is an image mask")
            if subtype == "/Image":
                stats.image_count += 1
                if not self.allow_raster_images:
                    self.result.fail("RASTER_IMAGE", f"{scope} contains image XObject {name}")
            elif subtype == "/Form" and not self.allow_form_xobjects:
                self.result.fail(
                    "FORM_XOBJECT",
                    f"{scope} contains Form XObject {name}; use --allow-form-xobjects only for an intentionally reviewed Form workflow",
                )
            elif subtype not in {"/Form", ""}:
                self.result.fail(
                    "UNSUPPORTED_XOBJECT", f"{scope} contains unsupported XObject {name} ({subtype})"
                )

    def _record_clip_content(self, stats: PdfPageStats, state: GraphicsState, kind: str) -> None:
        for event_number in state.active_clips:
            stats.clip_events[event_number - 1].kinds[kind] += 1

    def _register_font(
        self, scope: str, resource_name: str, font_reference: Any, stats: PdfPageStats
    ) -> None:
        font = dictionary(font_reference)
        base_font = normalize_font_name(font.get("/BaseFont", ""))
        subtype = str(font.get("/Subtype", ""))
        embedded = font_is_embedded(font)
        to_unicode = "/ToUnicode" in font
        key = (scope, resource_name, base_font)
        stats.font_uses[key] = FontUse(
            scope, resource_name, base_font, subtype, embedded, to_unicode
        )
        identity = object_identity(font_reference)
        if identity in self._reported_fonts:
            return
        self._reported_fonts.add(identity)
        family = font_family(base_font)
        if family is None:
            self.result.fail(
                "FONT_FAMILY",
                f"{scope} font {resource_name} uses {base_font or '<unnamed>'}; only Arial and SimSun are allowed",
            )
        if not embedded:
            self.result.fail(
                "FONT_NOT_EMBEDDED",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) is not embedded",
            )
        if not to_unicode:
            self.result.fail(
                "FONT_NO_TOUNICODE",
                f"{scope} font {resource_name} ({base_font or '<unnamed>'}) has no /ToUnicode map",
            )

    def _analyze_operations(
        self,
        operations: Sequence[tuple[list[Any], bytes]],
        resources: dict,
        scope: str,
        stats: PdfPageStats,
        initial_state: GraphicsState,
        allowed_boxes: list[tuple[float, float, float, float]],
        form_stack: set[tuple[Any, ...]],
        implicit_scope: bool,
    ) -> None:
        state = replace(initial_state)
        base_clip_depth = initial_state.clip_depth
        stack: list[GraphicsState] = []
        path_ops: list[tuple[bytes, list[Any]]] = []
        pending_clip = False
        in_text = False
        text_objects_at_start = stats.text_object_count

        for index, (operands, operator) in enumerate(operations):
            operands = list(operands)
            if operator == b"q":
                stats.q_count += 1
                stack.append(replace(state))
                continue
            if operator == b"Q":
                stats.Q_count += 1
                if not stack:
                    self.result.fail("Q_UNDERFLOW", f"{scope} has Q without a matching q at operation {index}")
                else:
                    state = stack.pop()
                path_ops = []
                pending_clip = False
                continue
            if operator == b"cm":
                if len(operands) != 6:
                    self.result.fail("BAD_CTM", f"{scope} has malformed cm at operation {index}")
                else:
                    matrix = tuple(numeric(value) for value in operands)
                    state.ctm = matrix_multiply(state.ctm, matrix)  # type: ignore[arg-type]
                continue

            if operator in PATH_BUILDING:
                path_ops.append((operator, operands))
                stats.vector_ops[operator.decode("ascii")] += 1
                continue
            if operator in {b"W", b"W*"}:
                if pending_clip:
                    self.result.fail(
                        "CLIP_PENDING", f"{scope} has consecutive clipping operators at operation {index}"
                    )
                if operator == b"W*":
                    self.result.fail(
                        "CLIP_EVEN_ODD",
                        f"{scope} uses W* at operation {index}; the strict profile requires nonzero-winding W",
                    )
                pending_clip = True
                continue
            if operator in PATH_ENDING:
                if pending_clip:
                    if operator != b"n":
                        self.result.fail(
                            "CLIP_PAINTED_PATH",
                            f"{scope} applies clipping with W/W* followed by {operator.decode('latin1')}; strict clipping must end with n",
                        )
                    bbox, reason = parse_simple_rectangle(path_ops, state.ctm)
                    event_number = len(stats.clip_events) + 1
                    if reason:
                        self.result.fail(
                            "CLIP_NOT_RECTANGLE",
                            f"{scope} clip {event_number} is invalid: {reason}",
                        )
                    elif bbox is not None and not any(
                        boxes_match(bbox, candidate) for candidate in allowed_boxes
                    ):
                        pretty = ", ".join(f"{value:.3f}" for value in bbox)
                        self.result.fail(
                            "CLIP_BOX_UNDECLARED",
                            f"{scope} clip {event_number} box ({pretty}) is neither the page box nor a declared --clip-box",
                        )
                    if not stack and not implicit_scope:
                        self.result.fail(
                            "CLIP_NOT_SCOPED",
                            f"{scope} clip {event_number} is not enclosed by an explicit q/Q scope",
                        )
                    state.clip_depth += 1
                    state.active_clips = state.active_clips + (event_number,)
                    stats.max_clip_depth = max(stats.max_clip_depth, state.clip_depth)
                    stats.clip_events.append(
                        ClipEvent(event_number, scope, state.clip_depth, bbox)
                    )
                    if state.clip_depth > 1:
                        self.result.fail(
                            "CLIP_NESTED",
                            f"{scope} clip {event_number} reaches effective clipping depth {state.clip_depth}",
                        )
                    pending_clip = False
                if operator in FILLING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill")
                elif operator in STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-stroke")
                elif operator in FILL_STROKING:
                    stats.vector_paint_count += 1
                    self._record_clip_content(stats, state, "vector-fill+stroke")
                path_ops = []
                continue

            if operator == b"BT":
                if in_text:
                    self.result.fail("TEXT_NESTED", f"{scope} contains nested BT operators")
                in_text = True
                stats.text_object_count += 1
                continue
            if operator == b"ET":
                if not in_text:
                    self.result.fail("TEXT_ET_UNMATCHED", f"{scope} contains ET without BT")
                in_text = False
                continue
            if operator == b"Tr":
                try:
                    state.text_render_mode = int(numeric(operands[0]))
                except Exception:
                    self.result.fail("TEXT_RENDER_MODE", f"{scope} contains malformed Tr")
                    continue
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} uses clipping text rendering mode {state.text_render_mode}",
                    )
                continue
            if operator == b"Tf":
                if len(operands) < 2:
                    self.result.fail("FONT_SELECT", f"{scope} contains malformed Tf")
                    continue
                resource_name = str(operands[0])
                fonts = dictionary(resources.get("/Font", {}))
                font_reference = fonts.get(resource_name)
                if font_reference is None:
                    self.result.fail(
                        "FONT_MISSING", f"{scope} selects missing font resource {resource_name}"
                    )
                    state.font_resource = None
                    state.font_object = None
                else:
                    state.font_resource = resource_name
                    state.font_object = font_reference
                continue
            if operator in TEXT_SHOWING:
                stats.text_show_count += 1
                if not in_text:
                    self.result.fail("TEXT_OUTSIDE_BT", f"{scope} shows text outside BT/ET")
                if state.text_render_mode in {3, 7}:
                    self.result.fail(
                        "INVISIBLE_TEXT", f"{scope} contains invisible text rendering mode {state.text_render_mode}"
                    )
                if state.text_render_mode in {4, 5, 6, 7}:
                    self.result.fail(
                        "TEXT_CLIPPING",
                        f"{scope} shows text in clipping rendering mode {state.text_render_mode}",
                    )
                if not state.font_resource or state.font_object is None:
                    self.result.fail("TEXT_NO_FONT", f"{scope} shows text without a resolved font")
                else:
                    self._register_font(scope, state.font_resource, state.font_object, stats)
                self._record_clip_content(stats, state, "text")
                continue

            if operator == b"Do":
                if not operands:
                    self.result.fail("XOBJECT_CALL", f"{scope} has malformed Do operator")
                    continue
                resource_name = str(operands[0])
                xobjects = dictionary(resources.get("/XObject", {}))
                reference = xobjects.get(resource_name)
                if reference is None:
                    self.result.fail(
                        "XOBJECT_MISSING", f"{scope} invokes missing XObject {resource_name}"
                    )
                    continue
                xobject = dictionary(reference)
                subtype = str(xobject.get("/Subtype", ""))
                if subtype == "/Image":
                    self._record_clip_content(stats, state, "image")
                    continue
                if subtype != "/Form":
                    self.result.fail(
                        "UNSUPPORTED_XOBJECT",
                        f"{scope} invokes unsupported XObject {resource_name} ({subtype})",
                    )
                    continue
                stats.form_calls += 1
                self._record_clip_content(stats, state, "form")
                identity = object_identity(reference)
                if identity in form_stack:
                    self.result.fail(
                        "FORM_RECURSION", f"{scope} has recursive Form XObject {resource_name}"
                    )
                    continue
                form_resources = dictionary(xobject.get("/Resources", resources))
                self._scan_resources(form_resources, f"{scope}/Form {resource_name}", stats)
                form_matrix_raw = dereference(xobject.get("/Matrix", [1, 0, 0, 1, 0, 0]))
                try:
                    form_matrix = tuple(numeric(value) for value in form_matrix_raw)
                    if len(form_matrix) != 6:
                        raise ValueError
                except Exception:
                    self.result.fail(
                        "FORM_MATRIX", f"{scope}/Form {resource_name} has malformed /Matrix"
                    )
                    form_matrix = (1, 0, 0, 1, 0, 0)
                form_state = replace(state)
                form_state.ctm = matrix_multiply(state.ctm, form_matrix)  # type: ignore[arg-type]
                try:
                    form_stream = ContentStream(dereference(reference), self.reader)
                except Exception as exc:
                    self.result.not_certified(
                        "FORM_PARSE", f"{scope}/Form {resource_name} cannot be parsed: {exc}"
                    )
                    continue
                self._analyze_operations(
                    form_stream.operations,
                    form_resources,
                    f"{scope}/Form {resource_name}",
                    stats,
                    form_state,
                    allowed_boxes,
                    form_stack | {identity},
                    implicit_scope=True,
                )
                continue

            if operator in INLINE_IMAGE_OPS:
                stats.inline_image_count += 1
                self.result.fail("INLINE_IMAGE", f"{scope} contains an inline image")
                self._record_clip_content(stats, state, "inline-image")
                continue
            if operator == b"sh":
                self.result.fail("SHADING", f"{scope} paints a shading object")
                self._record_clip_content(stats, state, "shading")
                continue

        if pending_clip:
            self.result.fail("CLIP_UNFINISHED", f"{scope} ends before its W/W* clip is applied")
        if path_ops:
            self.result.fail("PATH_UNFINISHED", f"{scope} ends with an unfinished path")
        if in_text:
            self.result.fail("TEXT_UNFINISHED", f"{scope} ends before ET")
        if stack:
            self.result.fail(
                "Q_UNBALANCED", f"{scope} has {len(stack)} q operator(s) without matching Q"
            )
        if state.clip_depth != base_clip_depth and not implicit_scope:
            self.result.fail(
                "CLIP_NOT_RESTORED",
                f"{scope} ends at clipping depth {state.clip_depth}, expected {base_clip_depth}",
            )
        if stats.text_object_count < text_objects_at_start:
            self.result.not_certified("INTERNAL_TEXT_COUNT", f"{scope} text-object accounting failed")

    def _validate_page_text(self, page: Any, stats: PdfPageStats) -> None:
        fragments: list[tuple[str, float, float, float, str]] = []
        parsed_font_names = {
            normalize_font_name(font.base_font) for font in stats.font_uses.values()
        }

        def visitor(text: str, cm: Sequence[float], tm: Sequence[float], font: Any, size: float) -> None:
            cleaned = text.strip()
            if not cleaned:
                return
            base_font = normalize_font_name(dictionary(font).get("/BaseFont", ""))
            try:
                x = float(cm[0]) * float(tm[4]) + float(cm[2]) * float(tm[5]) + float(cm[4])
                y = float(cm[1]) * float(tm[4]) + float(cm[3]) * float(tm[5]) + float(cm[5])
            except Exception:
                x = y = 0.0
            fragments.append((cleaned, x, y, float(size or 0), base_font))
            family = font_family(base_font)
            # pypdf can report a parent-page default font for text inside a Form
            # XObject. The content-stream pass above resolves the Form's actual
            # font. Apply script-specific checks only when the visitor font is a
            # font that the operation parser observed on a text-show operation.
            visitor_font_is_resolved = base_font in parsed_font_names
            if visitor_font_is_resolved and ASCII_ALNUM_RE.search(cleaned) and family != "Arial":
                self.result.fail(
                    "ASCII_FONT",
                    f"page {stats.page_number} text {cleaned!r} uses {base_font or '<unknown>'}, not Arial",
                )
            if visitor_font_is_resolved and CJK_RE.search(cleaned) and family != "SimSun":
                self.result.fail(
                    "CJK_FONT",
                    f"page {stats.page_number} Chinese text {cleaned!r} uses {base_font or '<unknown>'}, not SimSun",
                )

        try:
            page.extract_text(visitor_text=visitor)
        except Exception as exc:
            self.result.not_certified(
                "TEXT_EXTRACT", f"page {stats.page_number} live text cannot be decoded: {exc}"
            )
            return

        # Detect the common bad export in which a word or label becomes one text
        # object per character. Single tick labels are excluded by requiring a run.
        single_fragments = [
            item
            for item in fragments
            if len(item[0]) == 1 and (item[0].isalpha() or CJK_RE.search(item[0]))
        ]
        for index, current in enumerate(single_fragments):
            run = [current]
            for candidate in single_fragments[index + 1 :]:
                prior = run[-1]
                same_line = abs(candidate[2] - prior[2]) <= max(0.75, prior[3] * 0.35)
                same_font = candidate[4] == prior[4] and abs(candidate[3] - prior[3]) <= 0.25
                gap = candidate[1] - prior[1]
                nearby = -0.25 * max(prior[3], 1) <= gap <= 2.5 * max(prior[3], 1)
                if same_line and same_font and nearby:
                    run.append(candidate)
                    if len(run) >= 3:
                        word = "".join(item[0] for item in run)
                        self.result.fail(
                            "TEXT_FRAGMENTED",
                            f"page {stats.page_number} appears to split label {word!r} into individual character objects",
                        )
                        return
                elif candidate[2] < prior[2] - max(1.0, prior[3]):
                    break

    def _validate_page_summary(self, stats: PdfPageStats) -> None:
        clip_count = len(stats.clip_events)
        if clip_count > self.max_clips:
            self.result.fail(
                "TOO_MANY_CLIPS",
                f"page {stats.page_number} contains {clip_count} clipping paths; maximum is {self.max_clips}",
            )
        if self.expected_clips is not None and clip_count != self.expected_clips:
            self.result.fail(
                "CLIP_COUNT_EXPECTED",
                f"page {stats.page_number} contains {clip_count} clipping paths; expected exactly {self.expected_clips}",
            )
        if stats.max_clip_depth > 1:
            self.result.fail(
                "CLIP_DEPTH",
                f"page {stats.page_number} reaches clipping depth {stats.max_clip_depth}; maximum is 1",
            )
        for event in stats.clip_events:
            if not event.kinds:
                self.result.fail(
                    "EMPTY_CLIP",
                    f"page {stats.page_number} clip {event.number} contains no painted vector or live text content",
                )
        if stats.q_count != stats.Q_count:
            self.result.fail(
                "Q_COUNT",
                f"page {stats.page_number} recursively contains q={stats.q_count}, Q={stats.Q_count}",
            )
        if not self.allow_no_text and stats.text_show_count == 0:
            self.result.fail("NO_LIVE_TEXT", f"page {stats.page_number} contains no live text")
        if not self.allow_no_vector and stats.vector_paint_count == 0:
            self.result.fail("NO_VECTOR", f"page {stats.page_number} contains no painted vector path")


def validate_png(
    path: Path,
    target_dpi: float,
    dpi_tolerance: float,
    expected_size: tuple[float, float] | None,
) -> ValidationResult:
    result = ValidationResult(path, "PNG")
    if path.suffix.lower() != ".png":
        result.fail("PNG_EXTENSION", "filename does not end in .png")
    try:
        with Image.open(path) as image:
            if image.format != "PNG":
                result.fail("PNG_FORMAT", f"file content is {image.format}, not PNG")
            width, height = image.size
            dpi = image.info.get("dpi")
            image.load()
            blank_reason = image_blank_reason(image)
            if blank_reason:
                result.fail("PNG_BLANK", blank_reason)
    except Exception as exc:
        result.not_certified("PNG_DECODE", f"PNG cannot be decoded: {exc}")
        return result

    result.details.append(f"pixels={width}x{height}")
    if not dpi or len(dpi) < 2:
        result.fail("PNG_DPI_MISSING", "PNG has no two-axis physical-resolution metadata")
    else:
        x_dpi, y_dpi = float(dpi[0]), float(dpi[1])
        result.details.append(f"dpi={x_dpi:.3f}x{y_dpi:.3f}")
        if abs(x_dpi - target_dpi) > dpi_tolerance or abs(y_dpi - target_dpi) > dpi_tolerance:
            result.fail(
                "PNG_DPI",
                f"PNG resolution is {x_dpi:.3f}x{y_dpi:.3f} DPI; expected {target_dpi:g}+/-{dpi_tolerance:g}",
            )
        if expected_size:
            expected_width = expected_size[0] * target_dpi
            expected_height = expected_size[1] * target_dpi
            pixel_tolerance = max(1.0, target_dpi * 0.005)
            if abs(width - expected_width) > pixel_tolerance or abs(height - expected_height) > pixel_tolerance:
                result.fail(
                    "PNG_PIXEL_SIZE",
                    f"pixels {width}x{height} do not match {expected_size[0]:g}x{expected_size[1]:g} inches at {target_dpi:g} DPI",
                )
            result.details.append(
                f"expected-physical-size={expected_size[0]:g}x{expected_size[1]:g}in"
            )
        else:
            result.details.append(
                f"inferred-physical-size={width / x_dpi:.3f}x{height / y_dpi:.3f}in"
            )
    return result


def find_pdftoppm(explicit: str | None) -> str | None:
    if explicit:
        candidate = Path(explicit)
        return str(candidate) if candidate.is_file() else None
    return shutil.which("pdftoppm")


def _render_size_matches(
    observed: tuple[int, int], expected_points: tuple[float, float], dpi: float, rotation: int
) -> bool:
    width_points, height_points = expected_points
    if rotation in {90, 270}:
        width_points, height_points = height_points, width_points
    expected = (width_points * dpi / 72.0, height_points * dpi / 72.0)
    # Poppler normally rounds page boundaries outward. Accept at most one pixel
    # of normal raster-grid rounding in either dimension.
    return all(abs(actual - target) <= 1.05 for actual, target in zip(observed, expected))


def _prepare_render_directory(
    path: Path, result: ValidationResult, requested_base: Path | None
) -> tuple[Path | None, tempfile.TemporaryDirectory[str] | None]:
    if requested_base is not None:
        try:
            base = requested_base.expanduser().resolve(strict=False)
            if base.exists() and not base.is_dir():
                result.not_certified(
                    "RENDER_DIR_NOT_DIRECTORY", f"--render-dir is not a directory: {base}"
                )
                return None, None
            base.mkdir(parents=True, exist_ok=True)
            safe_stem = re.sub(r"[^A-Za-z0-9._-]+", "-", path.stem).strip("-.") or "figure"
            # Always create a normal child directory so Windows applies the
            # parent's inherited ACL. Path.mkdir(exist_ok=False) is atomic and
            # cannot overwrite an existing entry; the retry loop handles the
            # extremely unlikely random-name collision.
            output = None
            for _ in range(20):
                candidate = base / f"{safe_stem[:60]}-render-{secrets.token_hex(4)}"
                try:
                    candidate.mkdir(exist_ok=False)
                except FileExistsError:
                    continue
                output = candidate
                break
            if output is None:
                result.not_certified(
                    "RENDER_DIR_COLLISION",
                    "could not allocate a unique render child after 20 attempts",
                )
                return None, None
        except OSError as exc:
            result.not_certified(
                "RENDER_DIR_CREATE", f"cannot create a unique child under --render-dir: {exc}"
            )
            return None, None
        result.details.append(f"render-output-dir={output} (retained; unique child, no overwrite)")
        return output, None

    task_tmp_root = Path.cwd() / "tmp"
    try:
        task_tmp_root.mkdir(parents=True, exist_ok=True)
        temporary = tempfile.TemporaryDirectory(prefix="validate-figure-", dir=task_tmp_root)
    except OSError:
        try:
            temporary = tempfile.TemporaryDirectory(prefix="validate-figure-")
        except OSError as exc:
            result.not_certified("RENDER_DIR_CREATE", f"cannot create a temporary render directory: {exc}")
            return None, None
    return Path(temporary.name), temporary


def render_pdf(
    path: Path,
    result: ValidationResult,
    page_stats: Sequence[PdfPageStats],
    executable: str | None,
    render_dpi: float,
    render_dir: Path | None,
) -> None:
    pdftoppm = find_pdftoppm(executable)
    if not pdftoppm:
        result.not_certified("POPPLER_MISSING", "pdftoppm was not found; visual rendering is mandatory")
        return
    output_dir, temporary = _prepare_render_directory(path, result, render_dir)
    if output_dir is None:
        return
    try:
        prefix = output_dir / "page"
        command = [
            pdftoppm,
            "-png",
            "-r",
            f"{render_dpi:g}",
            "-f",
            "1",
            "-l",
            str(len(page_stats)),
            str(path),
            str(prefix),
        ]
        try:
            completed = subprocess.run(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=max(60, len(page_stats) * 30),
                check=False,
            )
        except Exception as exc:
            result.not_certified("POPPLER_RUN", f"Poppler rendering could not run: {exc}")
            return
        if completed.returncode != 0:
            detail = completed.stderr.strip() or f"exit code {completed.returncode}"
            result.not_certified("POPPLER_RENDER", f"Poppler failed to render the PDF: {detail}")
            return
        rendered: dict[int, Path] = {}
        for candidate in output_dir.glob("page-*.png"):
            match = re.fullmatch(r"page-(\d+)\.png", candidate.name, flags=re.IGNORECASE)
            if match:
                rendered[int(match.group(1))] = candidate
        if len(rendered) != len(page_stats) or set(rendered) != set(
            range(1, len(page_stats) + 1)
        ):
            result.not_certified(
                "POPPLER_PAGE_COUNT",
                f"Poppler produced page numbers {sorted(rendered)}, expected 1..{len(page_stats)}",
            )
            return
        for stats in page_stats:
            output = rendered[stats.page_number]
            try:
                with Image.open(output) as image:
                    image.verify()
                with Image.open(output) as image:
                    image.load()
                    observed_size = image.size
                    blank_reason = image_blank_reason(image)
            except Exception as exc:
                result.not_certified("POPPLER_OUTPUT", f"rendered page {output.name} is invalid: {exc}")
                continue
            if stats.rotation not in {0, 90, 180, 270}:
                result.not_certified(
                    "PAGE_ROTATION",
                    f"page {stats.page_number} has unsupported /Rotate={stats.rotation}",
                )
                matched_box = None
            elif _render_size_matches(
                observed_size, stats.crop_size_points, render_dpi, stats.rotation
            ):
                matched_box = "CropBox"
            elif _render_size_matches(
                observed_size, stats.media_size_points, render_dpi, stats.rotation
            ):
                matched_box = "MediaBox"
            else:
                matched_box = None
                result.fail(
                    "RENDER_PIXEL_SIZE",
                    f"page {stats.page_number} rendered to {observed_size[0]}x{observed_size[1]} px at {render_dpi:g} DPI; "
                    f"this matches neither CropBox {stats.crop_size_points[0]:.3f}x{stats.crop_size_points[1]:.3f} pt "
                    f"nor MediaBox {stats.media_size_points[0]:.3f}x{stats.media_size_points[1]:.3f} pt",
                )
            if blank_reason:
                result.fail(
                    "RENDER_BLANK",
                    f"page {stats.page_number} rendered blank: {blank_reason}",
                )
            match_text = matched_box or "no-box-match"
            content_text = "blank" if blank_reason else "nonblank"
            result.details.append(
                f"render-page={stats.page_number} pixels={observed_size[0]}x{observed_size[1]} "
                f"dpi={render_dpi:g} box={match_text} content={content_text}"
            )
        result.details.append(f"poppler-rendered-pages={len(rendered)}")
    finally:
        if temporary is not None:
            temporary.cleanup()


def validate_pdf(path: Path, args: argparse.Namespace) -> ValidationResult:
    result = ValidationResult(path, "PDF")
    if path.suffix.lower() != ".pdf":
        result.fail("PDF_EXTENSION", "filename does not end in .pdf")
    auditor = PdfAuditor(
        path,
        result,
        max_clips=args.max_clips,
        expected_clips=args.expected_clips,
        declared_clip_boxes=args.clip_box,
        allow_raster_images=args.allow_raster_images,
        allow_form_xobjects=args.allow_form_xobjects,
        allow_no_text=args.allow_no_text,
        allow_no_vector=args.allow_no_vector,
    )
    page_stats = auditor.audit()
    if page_stats:
        for stats in page_stats:
            fonts = sorted({font.base_font for font in stats.font_uses.values()})
            clip_summary = "; ".join(
                f"#{event.number} depth={event.depth} kinds={'+'.join(sorted(event.kinds)) or 'empty'}"
                for event in stats.clip_events
            )
            result.details.append(
                f"page={stats.page_number} clips={len(stats.clip_events)} max-depth={stats.max_clip_depth} "
                f"q/Q={stats.q_count}/{stats.Q_count} text-shows={stats.text_show_count} "
                f"vector-paints={stats.vector_paint_count} images={stats.image_count + stats.inline_image_count} "
                f"forms={stats.form_calls} fonts={','.join(fonts) or 'none'}"
            )
            if clip_summary:
                result.details.append(f"page={stats.page_number} clip-groups: {clip_summary}")
        render_pdf(
            path,
            result,
            page_stats,
            args.pdftoppm,
            args.render_dpi,
            args.render_dir,
        )
    return result


def parse_box(value: str) -> tuple[float, float, float, float]:
    try:
        fields = tuple(float(part.strip()) for part in value.split(","))
    except ValueError as exc:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height") from exc
    if len(fields) != 4 or fields[2] <= 0 or fields[3] <= 0:
        raise argparse.ArgumentTypeError("clip box must be x,y,width,height with positive size")
    return fields  # type: ignore[return-value]


def parse_size(value: str) -> tuple[float, float]:
    match = re.fullmatch(r"\s*([0-9]*\.?[0-9]+)\s*[xX×]\s*([0-9]*\.?[0-9]+)\s*", value)
    if not match:
        raise argparse.ArgumentTypeError("expected size must be WIDTHxHEIGHT in inches")
    return (float(match.group(1)), float(match.group(2)))


def positive_float(value: str) -> float:
    try:
        parsed = float(value)
    except ValueError as exc:
        raise argparse.ArgumentTypeError("value must be a positive number") from exc
    if not math.isfinite(parsed) or parsed <= 0:
        raise argparse.ArgumentTypeError("value must be a finite positive number")
    return parsed


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Validate 600-DPI PNGs or Illustrator-editable vector figure PDFs."
    )
    parser.add_argument("files", nargs="+", type=Path)
    parser.add_argument("--target-dpi", type=float, default=600.0)
    parser.add_argument("--dpi-tolerance", type=float, default=1.0)
    parser.add_argument(
        "--expected-size-in",
        type=parse_size,
        metavar="WIDTHxHEIGHT",
        help="also verify PNG pixel dimensions against a physical size in inches",
    )
    parser.add_argument("--max-clips", type=int, default=4, choices=range(0, 5))
    parser.add_argument(
        "--expected-clips",
        type=int,
        choices=range(0, 5),
        help="require exactly this many clipping paths on every PDF page (use 3 for the full-period regression profile)",
    )
    parser.add_argument(
        "--clip-box",
        type=parse_box,
        action="append",
        default=[],
        metavar="X,Y,W,H",
        help="declare an allowed non-page rectangular PDF clipping box (repeatable)",
    )
    parser.add_argument(
        "--allow-raster-images",
        action="store_true",
        help="allow intentional image XObjects; masks and inline images remain forbidden",
    )
    parser.add_argument(
        "--allow-form-xobjects",
        action="store_true",
        help="allow recursively inspected Form XObjects; the strict default rejects all Forms",
    )
    parser.add_argument("--allow-no-text", action="store_true")
    parser.add_argument("--allow-no-vector", action="store_true")
    parser.add_argument("--pdftoppm", help="explicit path to Poppler pdftoppm")
    parser.add_argument(
        "--render-dpi",
        type=positive_float,
        default=72.0,
        help="Poppler verification resolution (default: 72 DPI)",
    )
    parser.add_argument(
        "--render-dir",
        type=Path,
        help="retain rendered pages in a newly created unique child of DIR; existing contents are never overwritten",
    )
    return parser


def print_result(result: ValidationResult) -> None:
    print(f"[{result.status}] {result.path}")
    for detail in result.details:
        print(f"  INFO {detail}")
    for issue in result.issues:
        label = "ERROR" if issue.outcome == "FAIL" else "BLOCKER"
        print(f"  {label} [{issue.code}] {issue.message}")


def main(argv: Sequence[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)
    if args.expected_clips is not None and args.expected_clips > args.max_clips:
        parser.error("--expected-clips cannot exceed --max-clips")
    results: list[ValidationResult] = []
    for raw_path in args.files:
        path = raw_path.resolve()
        if not path.is_file():
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_MISSING", "file does not exist or is not a regular file")
        elif path.suffix.lower() == ".png":
            result = validate_png(path, args.target_dpi, args.dpi_tolerance, args.expected_size_in)
        elif path.suffix.lower() == ".pdf":
            result = validate_pdf(path, args)
        else:
            result = ValidationResult(path, "UNKNOWN")
            result.not_certified("FILE_TYPE", "only .png and .pdf are supported")
        results.append(result)
        print_result(result)
    statuses = Counter(result.status for result in results)
    passed = statuses["PASS"]
    print(
        f"SUMMARY passed={passed} failed={statuses['FAIL']} "
        f"not-certified={statuses['NOT CERTIFIED']} total={len(results)}"
    )
    return 0 if passed == len(results) else 1


if __name__ == "__main__":
    raise SystemExit(main())


## Existing panel drawing: unchanged source and layout

The following existing notebook cells are copied unchanged, including original shared figure outputs. Use this section in its original context; similarly named globals in other sections are not a new shared execution API.


# Figure S5A: Seven source-balanced VGN candidate expression curves

Source: `05_fig4B-D_VGN.ipynb`, using the accepted `VGN-lines.png` output. This file reads the saved `VGN-results.h5` and draws only this panel. It does not rebuild expression models, score regulons or modify the saved result.

Replace every `ROOT` placeholder with the required input, output, cache or font file path. Keep the original cached arrays and gene order. Both final figure copies, `FS5_VGN.png` and `FS4-2_VGN.png`, use the same content.


In [ ]:
#1. =========== Input, output and font paths
from pathlib import Path
import os, sys, json, hashlib
from urllib.parse import unquote
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
# Set the input file path. Uses VGN-results.h5 generated by 05_fig4B-D_VGN.ipynb.
RESULT_FILE = ROOT
# Set the PNG output file path for this panel.
PNG_FILE = ROOT
# Set the editable PDF output file path for this panel.
PDF_FILE = ROOT
# Set the Matplotlib cache directory.
CACHE_DIR = ROOT
os.environ['MPLCONFIGDIR'] = str(CACHE_DIR)
sys.dont_write_bytecode = True
SEED, DPI = 42, 600
GENES = ('Meis2', 'Npas3', 'Sall3', 'Tlx3', 'Tbx3', 'Onecut1', 'Pcdh19')
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
for file_path in (PNG_FILE, PDF_FILE):
    file_path.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
#2. =========== Read the accepted plotting data
def read_value(node):
    kind = node.attrs['kind']
    if kind == 'dataframe':
        columns = json.loads(node.attrs['columns'])
        out = pd.DataFrame({key: read_value(node[key]) for key in columns})
        out.index = read_value(node['_index']); return out
    if kind == 'dict': return {unquote(key): read_value(node[key]) for key in node}
    if kind == 'json': return json.loads(node.asstr()[()])
    return node.asstr()[:] if node.dtype.kind in 'OUS' else node[:]

with h5py.File(RESULT_FILE, 'r') as handle:
    assert handle.attrs['schema'] == 'VGN_fig4BD_v1'
    data = read_value(handle['data'])
np.testing.assert_array_equal(data['genes'], GENES)
assert len(data['line_units']) == 302 and len(data['line_points']) == 91
assert data['C_primary'].sum() == 478 and np.isfinite(data['D_atlas']).sum() == 168
assert data['network_edges'].shape[0] == 11 and len(data['module_genes']) == 12


In [ ]:
"""Strict vector PDF export for the Fig4 B/C/D Matplotlib canvas.

No analysis is performed. Each native Matplotlib label becomes one live Arial
text object, including the TF italic run in the two supported mathtext titles.
All geometry is written directly, without XObjects, glyph outlines or images.
The caller declares at most four data axes; their clip rectangles are collected
into one sibling viewport group per axis. Other artists must fit their axes.
"""

from pathlib import Path

import re

import sys

import math

import numpy as np

from matplotlib.backend_bases import RendererBase

from matplotlib.path import Path as MplPath

from reportlab.pdfgen import canvas

from reportlab.pdfbase import pdfmetrics

from reportlab.pdfbase.ttfonts import TTFont

_FONTS = {'normal':'FigArial','italic':'FigArialItalic',
          'bold':'FigArialBold','bolditalic':'FigArialBoldItalic'}

def _register():
    # Set the regular, italic, bold and bold-italic TrueType font file paths.
    for key, file in [('normal', ROOT), ('italic', ROOT),
                      ('bold', ROOT), ('bolditalic', ROOT)]:
        if _FONTS[key] not in pdfmetrics.getRegisteredFontNames():
            pdfmetrics.registerFont(TTFont(_FONTS[key], str(file)))

def _font(prop):
    bold=prop.get_weight() in ('bold','heavy','black','semibold','demibold') or isinstance(prop.get_weight(),(int,float)) and prop.get_weight()>=600
    italic=prop.get_style() in ('italic','oblique')
    return _FONTS['bolditalic' if bold and italic else 'bold' if bold else 'italic' if italic else 'normal']

def _runs(text,prop,ismath):
    if not ismath:
        if '$' in text: raise ValueError('Unexpected dollar in plain label: '+text)
        return [(text,_font(prop))]
    # Current Fig4 titles use only this explicitly audited mathtext form.
    found=re.fullmatch(r'\$\\it\{([^{}]+)\}([^$]*)\$(.*)',text)
    if found is None: raise ValueError('Unsupported math label: '+text)
    tf,inside,after=found.groups()
    return [(tf,_FONTS['italic']),(inside+after,_FONTS['normal'])]

class _Recorder(RendererBase):
    def __init__(self,fig,agg,viewports,halo_labels):
        super().__init__()
        self.dpi=fig.dpi;self.unit=72/fig.dpi;self.agg=agg
        self.width,self.height=fig.bbox.size;self.ops=[];self.viewports=viewports;self.halo_labels=halo_labels
    def points_to_pixels(self,p): return p/self.unit
    def flipy(self):return False
    def get_canvas_width_height(self):return self.width,self.height
    def get_text_width_height_descent(self,s,prop,ismath):
        runs=_runs(s,prop,ismath)
        width=sum(pdfmetrics.stringWidth(t,f,prop.get_size_in_points()) for t,f in runs)/self.unit
        # Match the source Arial baseline/ascent geometry; width uses the same
        # embedded font as the PDF, including the semantic mixed italic title.
        _,h,d=self.agg.get_text_width_height_descent(re.sub(r'\$|\\it\{|\}', '',s),prop,False)
        return width,h,d
    def _clip(self,gc):
        if gc.get_clip_path()[0] is not None:
            raise ValueError('Nonrectangular artist clip is unsupported')
        b=gc.get_clip_rectangle()
        if b is None:return None
        for i,box in enumerate(self.viewports):
            if np.allclose(b.bounds,box,rtol=0,atol=1e-5):return i
        return None
    def draw_path(self,gc,path,transform,rgbFace=None):
        if gc.get_hatch():raise ValueError('Hatching/Pattern is forbidden')
        p=transform.transform_path(path)
        self.ops.append(('path',self._clip(gc),p,gc.get_rgb(),rgbFace,
                         gc.get_linewidth(),gc.get_alpha(),gc.get_forced_alpha(),
                         gc.get_dashes(),gc.get_capstyle(),gc.get_joinstyle()))
    def draw_text(self,gc,x,y,s,prop,angle,ismath=False,mtext=None):
        self.ops.append(('text',None,float(x),float(y),str(s),prop.copy(),float(angle),
                         ismath,gc.get_rgb(),gc.get_alpha(),id(mtext) in self.halo_labels))
    def draw_image(self,*args,**kwargs):raise ValueError('Raster image is forbidden')
    def start_rasterizing(self):pass
    def stop_rasterizing(self):pass
    def start_filter(self):raise ValueError('Raster filter is forbidden')
    def stop_filter(self,*args):raise ValueError('Raster filter is forbidden')

def _pdf_path(pdf,path,unit):
    out=pdf.beginPath();current=None;start=None
    for vertices,code in path.iter_segments(remove_nans=True,simplify=False,curves=True):
        v=np.asarray(vertices).reshape(-1,2)*unit
        if code==MplPath.MOVETO:
            current=v[0];start=current;out.moveTo(*current)
        elif code==MplPath.LINETO:
            current=v[0];out.lineTo(*current)
        elif code==MplPath.CURVE3:
            control,end=v
            a=current+(control-current)*2/3;b=end+(control-end)*2/3
            out.curveTo(*a,*b,*end);current=end
        elif code==MplPath.CURVE4:
            out.curveTo(*v.ravel());current=v[-1]
        elif code==MplPath.CLOSEPOLY:
            out.close();current=start
        elif code==MplPath.STOP:break
        else:raise ValueError('Unknown path opcode '+str(code))
    return out

def strict_editable_pdf(fig,destination,*,data_axes=(),live_halo_width=.20):
    """Write a strict vector PDF. Return declared clip boxes in PDF points.

    The artist canvas remains unchanged after return. Rasterization flags and
    white text path effects are replaced by a narrow live text fill+stroke
    (render mode 2) in the same single label object; no outlined font paths are
    emitted. The width is narrower than Agg's halo to preserve the black stems.
    """
    _register()
    if len(data_axes)>4:raise ValueError('At most four sibling data viewports')
    fig.canvas.draw();agg=fig.canvas.get_renderer()
    viewports=[tuple(ax.bbox.bounds) for ax in data_axes]
    effects=[]
    for artist in fig.findobj():
        if hasattr(artist,'get_path_effects'):
            old=artist.get_path_effects()
            if old:effects.append((artist,old));artist.set_path_effects([])
    rec=_Recorder(fig,agg,viewports,{id(a) for a,e in effects})
    try:fig.draw(rec)
    finally:
        for a,e in effects:a.set_path_effects(e)
    dst=Path(destination);dst.parent.mkdir(parents=True,exist_ok=True)
    pdf=canvas.Canvas(str(dst),pagesize=tuple(fig.get_size_inches()*72),
                      pageCompression=1,invariant=1,initialFontName=_FONTS['normal'])
    unit=rec.unit
    def emit(op):
        if op[0]=='text':
            _,_,x,y,s,prop,angle,ismath,color,alpha,halo=op
            if not s:return
            pdf.saveState();pdf.setFillColorRGB(*color[:3]);pdf.setFillAlpha(alpha)
            text=pdf.beginText();rad=math.radians(angle)
            if halo and live_halo_width>0:
                pdf.setStrokeColorRGB(1,1,1);pdf.setStrokeAlpha(alpha)
                pdf.setLineWidth(float(live_halo_width));text.setTextRenderMode(2)
            text.setTextTransform(math.cos(rad),math.sin(rad),-math.sin(rad),math.cos(rad),x*unit,y*unit)
            for value,font in _runs(s,prop,ismath):
                if value:text.setFont(font,prop.get_size_in_points());text.textOut(value)
            pdf.drawText(text);pdf.restoreState();return
        _,_,path,stroke,face,lw,alpha,forced,dashes,cap,join=op
        pdf.saveState();pdf.setLineWidth(lw)
        pdf.setLineCap({'butt':0,'round':1,'projecting':2}[cap])
        pdf.setLineJoin({'miter':0,'round':1,'bevel':2}[join])
        offset,pattern=dashes
        pdf.setDash([] if pattern is None else [float(v) for v in pattern],0.0 if offset is None else float(offset))
        if stroke is not None:
            pdf.setStrokeColorRGB(*stroke[:3]);pdf.setStrokeAlpha(alpha if forced else (stroke[3] if len(stroke)>3 else alpha))
        if face is not None:
            pdf.setFillColorRGB(*face[:3]);pdf.setFillAlpha(alpha if forced else (face[3] if len(face)>3 else alpha))
        pdf.drawPath(_pdf_path(pdf,path,unit),stroke=int(stroke is not None and lw>0),fill=int(face is not None),fillMode=1)
        pdf.restoreState()
    done=set()
    for op in rec.ops:
        key=op[1]
        if key is None:emit(op);continue
        if key in done:continue
        done.add(key);pdf.saveState()
        p=pdf.beginPath();p.rect(*(np.asarray(viewports[key])*unit))
        pdf.clipPath(p,stroke=0,fill=0,fillMode=1)
        for candidate in rec.ops:
            if candidate[1]==key:emit(candidate)
        pdf.restoreState()
    pdf.showPage();pdf.save()
    if done!=set(range(len(data_axes))):raise AssertionError('Declared clip has no artist')
    return {'expected_clips':len(done),'clip_boxes_pt':(np.asarray(viewports)*unit).tolist(),
            'editable_label_objects':sum(op[0]=='text' for op in rec.ops),
            'label_strings':[op[4] for op in rec.ops if op[0]=='text'],
            'vector_paths':sum(op[0]=='path' for op in rec.ops),
            'text_effects':'White live text fill+stroke, single object; stroke width '+str(live_halo_width)+' pt.',
            'halo_labels':len(effects)}

In [ ]:
#4. =========== Original panel drawing
import matplotlib.patheffects as pe
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath
from matplotlib.colors import LinearSegmentedColormap, Normalize

def check_figure(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    clipped = []
    for text in fig.findobj(matplotlib.text.Text):
        if text.get_visible() and text.get_text():
            box = text.get_window_extent(renderer)
            if not fig.bbox.contains(*box.p0) or not fig.bbox.contains(*box.p1):
                clipped.append(text.get_text())
    assert not clipped, clipped
    return {'clipped_labels': clipped}

def save_figure(fig, stem):
    qa = check_figure(fig)
    png, pdf = PNG_FILE, PDF_FILE
    fig.savefig(png, dpi=DPI, facecolor='white')
    qa.update(strict_editable_pdf(fig, pdf, data_axes=()))
    from PIL import Image
    with Image.open(png) as im:
        assert im.size == tuple(round(v * DPI) for v in fig.get_size_inches())
        assert all(abs(v - DPI) < .1 for v in im.info['dpi'])
        qa.update({'pixels': list(im.size), 'dpi': list(im.info['dpi'])})
    qa.update({'png': str(png), 'pdf': str(pdf)})
    plt.close(fig)
    return qa

def render_lines(data):
    FIGSIZE = (15.2, 6.4)
    X_LIMITS, Y_LIMITS = (9.1, 18.9), (-.10, 4.10)
    X_TICKS, Y_TICKS = (9.5, 11.5, 13.5, 16.5, 18.5), (0, 1, 2, 3, 4)
    MEAN_LINE_WIDTH, MEAN_MARKER_SIZE = 1.65, 3.8
    LIBRARY_AREA, LIBRARY_ALPHA = 10.0, .25
    ORDER = ('CVG', 'VGN', 'SGN')
    COLORS = {'CVG': '#8D77A1', 'VGN': '#5B85A7', 'SGN': '#69A18B'}
    units, points = data['line_units'], data['line_points']
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 9,
        'text.color': '#30343A', 'axes.labelcolor': '#30343A',
        'xtick.color': '#50555B', 'ytick.color': '#50555B', 'savefig.facecolor': 'white'})
    fig = plt.figure(figsize=FIGSIZE)
    grid = fig.add_gridspec(2, 4, left=.058, right=.985, bottom=.13, top=.94,
                           hspace=.53, wspace=.25)
    axes = []
    for j, gene in enumerate(GENES):
        row, col = (0, j) if j < 3 else (1, j - 3)
        ax = fig.add_subplot(grid[row, col])
        axes.append(ax)
        for identity in ORDER:
            u = units.loc[units.gene.eq(gene) & units.display_type.eq(identity)]
            p = points.loc[points.gene.eq(gene) & points.display_type.eq(identity)].sort_values('age')
            ax.scatter(u.age, u.expr, s=LIBRARY_AREA, color=COLORS[identity],
                       alpha=LIBRARY_ALPHA, linewidths=0, zorder=2)
            line, = ax.plot(p.age, p.expr, color=COLORS[identity], lw=MEAN_LINE_WIDTH,
                           marker='o', ms=MEAN_MARKER_SIZE, markeredgewidth=0, zorder=3)
            np.testing.assert_array_equal(line.get_ydata(), p.expr)
        ax.set_xlim(X_LIMITS); ax.set_ylim(Y_LIMITS)
        ax.set_xticks(X_TICKS, [f'E{x:g}' for x in X_TICKS])
        ax.xaxis.set_minor_locator(FixedLocator(np.arange(9.5, 19)))
        ax.set_yticks(Y_TICKS)
        if col: ax.tick_params(labelleft=False)
        ax.tick_params(which='major', length=2.6, width=.6, labelsize=8.5, pad=3)
        ax.tick_params(axis='x', which='minor', length=1.5, width=.45, color='#A5AAB0')
        ax.grid(axis='y', color='#EBECEE', linewidth=.55, zorder=0)
        for side in ('top', 'right'): ax.spines[side].set_visible(False)
        for side in ('left', 'bottom'):
            ax.spines[side].set_color('#A3A7AD'); ax.spines[side].set_linewidth(.65)
        ax.set_title(gene, fontstyle='italic', fontsize=12, pad=8)
    legend_ax = fig.add_subplot(grid[0, 3]); legend_ax.axis('off')
    handles = [Line2D([0], [0], color=COLORS[t], lw=MEAN_LINE_WIDTH, marker='o',
                      ms=4, markeredgewidth=0, label=t) for t in ORDER]
    handles.append(Line2D([0], [0], marker='o', color='#8C9299', alpha=LIBRARY_ALPHA,
                          linestyle='None', ms=4, label='Individual libraries'))
    legend_ax.legend(handles=handles, loc='center', frameon=False, fontsize=9.5,
                     handlelength=2.2, labelspacing=1.0)
    fig.supylabel('RNA: mean log10(1 + pseudobulk CPM)', x=.012, y=.55, fontsize=10)
    fig.supxlabel('Embryonic age', x=.53, y=.035, fontsize=10)
    fig.canvas.draw(); renderer = fig.canvas.get_renderer()
    for ax in axes:
        boxes = [t.get_window_extent(renderer) for t in ax.get_xticklabels() if t.get_visible()]
        assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[i + 1:])
    return save_figure(fig, 'VGN-lines')


In [ ]:
#5. =========== Save and preview this panel
figure_qa = render_lines(data)
display(Image(filename=str(PNG_FILE), width=900))
